In [1]:
!pip install psycopg2-binary sqlalchemy pandas
!pip install google-generativeai
!pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.9/609.9 kB 15.4 MB/s eta 0:00:00
Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 63.4 MB/s et

In [ ]:
import pandas as pd
import psycopg2
import re
import glob
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
import google.generativeai as genai
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()

genai.configure(api_key=os.getenv("GEMINI_KEY"))

model = genai.GenerativeModel('gemini-2.0-flash')


In [ ]:
!pip install -q google-generativeai python-dotenv pandas openpyxl


In [ ]:
"""
Pipeline 1 - Single run with filters and columns (ONE-STAGE PROMPT).
"""

import os, time, random, json, ast
import re
import pandas as pd
from dotenv import load_dotenv

import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, DeadlineExceeded, InternalServerError

# ============================
# CONFIG
# ============================
EXCEL_PATH_IN  = "runs/query-chosen-filters-columns.xlsx"
EXCEL_PATH_OUT = "runs/query-2.5-chosen-filters-columns_with_runs_pipeline1.xlsx"
SHEET_NAME     = 0
QUESTION_COLUMN = "question"   # auto-detected if missing
RUNS_PER_QUESTION = 3          # adjust as needed
DELAY_BETWEEN_CALLS_SEC = 0.0
MAX_RETRIES = 3
MODEL_NAME = "gemini-2.5-flash"

# Definition files (pointed to your uploaded assets)
FILTER_NAMES_DEF_PATH = "definitions_folder/definitions - aim2 - filter names.txt"  # filter CATEGORY NAMES only
COLUMN_DEFS_PATH      = "definitions_folder/definitions - aim2 - column.txt"        # column definitions
FILTER_DEFS_FULL_PATH = "definitions_folder/definitions - aim2 - filter.txt"        # full filter (categories + values)

# Columns that are always included by default (do NOT count toward the limit below)
REQUIRED_COLS = ["NCT", "PMID", "Authors", "Year"]
MAX_ADDITIONAL_COLS = 6  # number of columns the model may add on top of REQUIRED_COLS

# ============================
# SETUP
# ============================
load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
if not GEMINI_KEY:
    raise RuntimeError("GEMINI_KEY environment variable is not set")

genai.configure(api_key=GEMINI_KEY)

generation_config_json = {
    "response_mime_type": "application/json",
    "temperature": 0.0,
}

with open(FILTER_NAMES_DEF_PATH, "r", encoding="utf-8") as f:
    FILTER_NAMES_TEXT = f.read()
with open(COLUMN_DEFS_PATH, "r", encoding="utf-8") as f:
    COLUMN_DEFS_TEXT = f.read()
with open(FILTER_DEFS_FULL_PATH, "r", encoding="utf-8") as f:
    FILTER_DEFS_TEXT = f.read()

# ============================
# HELPERS
# ============================

def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```") and s.endswith("```"):
        s = s.strip("`")
        s = "\n".join(s.splitlines()[1:])
    return s.strip()


def extract_json(text: str):
    raw = strip_code_fences(text or "")
    try:
        return json.loads(raw)
    except Exception:
        pass
    try:
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = raw[start : end + 1]
            try:
                return json.loads(candidate)
            except Exception:
                try:
                    obj = ast.literal_eval(candidate)
                    if isinstance(obj, (dict, list)):
                        return obj
                except Exception:
                    pass
    except Exception:
        pass
    raise ValueError(f"Could not parse JSON from model text: {text[:200]}...")


def call_model_with_retries(prompt: str, seed: int | None = None) -> str:
    attempts = max(1, MAX_RETRIES)
    model = genai.GenerativeModel(MODEL_NAME)
    delay = 1.2
    for attempt in range(1, attempts + 1):
        try:
            resp = model.generate_content(
                prompt,
                generation_config=generation_config_json,
                request_options={"timeout": 90},
                safety_settings=None,
            )
            txt = getattr(resp, "text", None)
            return (txt if txt is not None else str(resp)).strip()
        except (ResourceExhausted, DeadlineExceeded, InternalServerError) as e:
            if attempt == attempts:
                return f"[ERROR after {attempt} attempts] {e}"
            time.sleep(delay * (2 ** (attempt - 1)) * (1 + random.uniform(0, 0.25)))
        except Exception as e:
            if attempt >= min(3, attempts):
                return f"[ERROR non-retryable? attempt {attempt}] {e}"
            time.sleep(delay * attempt)

# ============================
# SINGLE-STAGE PROMPT BUILDER
# ============================

def build_single_stage_prompt(question: str) -> str:
    """
    Build ONE prompt that includes:
      - Filter CATEGORY NAMES
      - Full FILTER definitions (categories + values, including 'All')
      - COLUMN definitions
    Ask Gemini to return the FINAL merged JSON in ONE response:

    {
      "selected_filter": {"Filter Category 1": "Chosen Value", ...},   # omit any 'All'
      "selected_column": { "Column 1":"NCT", "Column 2":"PMID", ... }  # enumerated object
    }
    """
    schema = (
        '{\n'
        '  "selected_filter": {\n'
        '    "Filter Category 1": "Chosen Value",\n'
        '    "Filter Category 2": "Chosen Value"\n'
        '  },\n'
        '  "selected_column": {\n'
        '    "Column 1": "Name",\n'
        '    "Column 2": "Name",\n'
        '    "Column 3": "Name",\n'
        '    "Column 4": "Name",\n'
        '    "Column 5": "Name",\n'
        '    "Column 6": "Name",\n'
        '    "Column 7": "Name",\n'
        '    "Column 8": "Name",\n'
        '    "Column 9": "Name",\n'
        '    "Column 10": "Name"\n'
        '  }\n'
        '}'
    )

    example = (
        '{'
        '"selected_filter":{"Cancer Type":"NSCLC","Trial Phase":"Phase 3","Type of Therapy":"Combination Therapy"},'
        '"selected_column":{"Column 1":"NCT","Column 2":"PMID","Column 3":"Authors","Column 4":"Year","Column 5":"Primary Endpoint","Column 6":"Sample Size","Column 7":"Name of ICI","Column 8":"Control regimen","Column 9":"Monotherapy/combination","Column 10":"Lines of treatment"}'
        '}'
    )

    # Clear, strict, one-shot instructions
    return (
        "You are a medical expert researching cancer trials. Build the ENTIRE selection in ONE step.\n"
        "Return ONLY a valid JSON object (no prose, no code fences) with this schema:\n"
        f"{schema}\n"
        "Strict rules:\n"
        f"- Columns: ALWAYS include these first (do NOT count toward the limit): {', '.join(REQUIRED_COLS)}.\n"
        f"- You may add at most {MAX_ADDITIONAL_COLS} additional columns beyond those required.\n"
        "- The enumerated object keys MUST be exactly 'Column 1', 'Column 2', ... in display order.\n"
        "- Use *exact* names from the available column list.\n"
        "- Filters: choose only categories/values justified by the question. If a category would be 'All', OMIT that category entirely.\n"
        "- Use *exact* category and value strings from the available filter definitions.\n"
        "- Use double quotes everywhere. No trailing commas. No explanations.\n\n"
        "Available filter CATEGORY NAMES (for reference):\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available FULL filter categories and values (use exact strings; omit categories that would be 'All'):\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        "Available columns and definitions (select by exact name):\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n\n"
        "Return JSON now. Example of valid formatting (not prescriptive):\n"
        f"{example}"
    )

# ============================
# NORMALIZATION & CONVERSION
# ============================

def _ordered_values_from_column_object(col_obj: dict) -> list[str]:
    """Accepts a mapping like {'Column 1': 'NCT', ...} and returns values ordered by numeric index."""
    if not isinstance(col_obj, dict):
        return []
    items = []
    for k, v in col_obj.items():
        m = re.search(r"Column\s*(\d+)", str(k))
        idx = int(m.group(1)) if m else 10**9
        items.append((idx, v))
    items.sort(key=lambda x: x[0])
    return [val for _, val in items if isinstance(val, str) and val.strip()]


def normalize_selected_column_object(selected_column) -> dict:
    """Ensure REQUIRED_COLS are present first, dedupe, and rebuild enumerated mapping with cap."""
    # Convert input (list or dict) → ordered list
    if isinstance(selected_column, dict):
        cols = _ordered_values_from_column_object(selected_column)
    elif isinstance(selected_column, list):
        cols = [c for c in selected_column if isinstance(c, str) and c.strip()]
    else:
        cols = []

    # Ensure required at the front, keep order & dedupe
    out_list, seen = [], set()
    for rc in REQUIRED_COLS:
        if rc not in seen:
            out_list.append(rc); seen.add(rc)
    for c in cols:
        if c not in seen:
            seen.add(c); out_list.append(c)

    # Keep only up to REQUIRED + MAX_ADDITIONAL_COLS
    cap = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    out_list = out_list[:cap]

    # Build enumerated object
    return {f"Column {i+1}": name for i, name in enumerate(out_list)}


def normalize_selected_filter(selected_filter: dict) -> dict:
    """Drop empty/None/'All' values and trim keys/values."""
    out = {}
    for k, v in (selected_filter or {}).items():
        if v is None:
            continue
        s = str(v).strip()
        if not s or s.lower() == "all":
            continue
        out[str(k).strip()] = s
    return out

# ============================
# CORE RUNNERS
# ============================

def run_once_for_question(question: str, seed: int | None = None) -> tuple[dict, float]:
    """Single run with timing measurement."""
    prompt = build_single_stage_prompt(question)
    start = time.time()
    text = call_model_with_retries(prompt, seed=seed)
    elapsed = time.time() - start

    try:
        obj = extract_json(text)
    except Exception:
        obj = {}

    selected_filter_raw  = obj.get("selected_filter", {}) if isinstance(obj, dict) else {}
    selected_column_raw  = obj.get("selected_column", {}) if isinstance(obj, dict) else {}

    selected_filter = normalize_selected_filter(selected_filter_raw)
    selected_column_obj = normalize_selected_column_object(selected_column_raw)

    merged = {
        "selected_filter": selected_filter,
        "selected_column": selected_column_obj,
    }
    return merged, elapsed


def run_n_times_for_question(question: str, n_runs: int):
    outputs, timings = [], []
    for _ in range(n_runs):
        seed = random.randint(1, 10_000_000)
        merged, elapsed = run_once_for_question(question, seed=seed)
        outputs.append(json.dumps(merged, ensure_ascii=False))
        timings.append(elapsed)
        time.sleep(DELAY_BETWEEN_CALLS_SEC)
    return outputs, timings

# ============================
# MAIN
# ============================

def main():
    df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)

    global QUESTION_COLUMN
    if QUESTION_COLUMN not in df.columns:
        candidates = [c for c in df.columns if str(c).strip().lower() in {"question", "query", "prompt"}]
        if candidates:
            QUESTION_COLUMN = candidates[0]
        else:
            raise ValueError(
                f"Couldn't find a question column named '{QUESTION_COLUMN}'. Available columns: {list(df.columns)}"
            )

    run_cols = [f"run_{i+1}" for i in range(RUNS_PER_QUESTION)]
    for c in run_cols:
        if c not in df.columns:
            df[c] = ""

    # Collect timings in parallel DataFrame
    timings_records = []

    for idx, row in df.iterrows():
        q = str(row[QUESTION_COLUMN]).strip()
        if not q or q.lower() == "nan":
            continue
        print(f"Processing row {idx}: {q[:80]}{'...' if len(q)>80 else ''}")
        merged_runs, timings = run_n_times_for_question(q, RUNS_PER_QUESTION)
        for i, merged_json in enumerate(merged_runs):
            df.at[idx, run_cols[i]] = merged_json
        timings_records.append({
            "row_index": idx,
            "question": q,
            **{f"run_{i+1}_time_sec": t for i, t in enumerate(timings)}
        })

    # Save both sheets
    timings_df = pd.DataFrame(timings_records)
    with pd.ExcelWriter(EXCEL_PATH_OUT, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="results")
        timings_df.to_excel(writer, index=False, sheet_name="timings")

    print(f"Saved results to: {EXCEL_PATH_OUT}")


if __name__ == "__main__":
    main()

In [ ]:
"""
Pipeline 2 - Filter names + column names on first stage before filter selection based on names
"""


import os, time, random, json, ast
import re
import pandas as pd
from dotenv import load_dotenv

import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, DeadlineExceeded, InternalServerError

# ============================
# CONFIG
# ============================
EXCEL_PATH_IN  = "runs/query-chosen-filters-columns.xlsx"
EXCEL_PATH_OUT = "runs/query-2.5-chosen-filters-columns_with_runs_pipeline2.xlsx"
SHEET_NAME     = 0
QUESTION_COLUMN = "question"  # auto-detected if missing
RUNS_PER_QUESTION = 3          # adjust as needed
DELAY_BETWEEN_CALLS_SEC = 0.0
MAX_RETRIES = 3
MODEL_NAME = "gemini-2.5-flash"

# Definition files
FILTER_NAMES_DEF_PATH = "definitions_folder/definitions - aim2 - filter names.txt"  # filter CATEGORY NAMES only
COLUMN_DEFS_PATH      = "definitions_folder/definitions - aim2 - column.txt"        # column definitions
FILTER_DEFS_FULL_PATH = "definitions_folder/definitions - aim2 - filter.txt"       # full filter (categories + values)

# Columns that are always included by default (do NOT count toward the limit below)
REQUIRED_COLS = ["NCT", "PMID", "Authors", "Year"]
MAX_ADDITIONAL_COLS = 6  # number of columns the model may add on top of REQUIRED_COLS


# ============================
# SETUP
# ============================
load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
if not GEMINI_KEY:
    raise RuntimeError("GEMINI_KEY environment variable is not set")

genai.configure(api_key=GEMINI_KEY)

generation_config_json = {
    "response_mime_type": "application/json",
    "temperature": 0.0,

}

with open(FILTER_NAMES_DEF_PATH, "r", encoding="utf-8") as f:
    FILTER_NAMES_TEXT = f.read()
with open(COLUMN_DEFS_PATH, "r", encoding="utf-8") as f:
    COLUMN_DEFS_TEXT = f.read()
with open(FILTER_DEFS_FULL_PATH, "r", encoding="utf-8") as f:
    FILTER_DEFS_TEXT = f.read()

# ============================
# HELPERS
# ============================

def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```") and s.endswith("```"):
        s = s.strip("`")
        s = "\n".join(s.splitlines()[1:])
    return s.strip()


def extract_json(text: str):
    raw = strip_code_fences(text or "")
    try:
        return json.loads(raw)
    except Exception:
        pass
    try:
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = raw[start : end + 1]
            try:
                return json.loads(candidate)
            except Exception:
                try:
                    obj = ast.literal_eval(candidate)
                    if isinstance(obj, (dict, list)):
                        return obj
                except Exception:
                    pass
    except Exception:
        pass
    raise ValueError(f"Could not parse JSON from model text: {text[:200]}...")


def call_model_with_retries(prompt: str, seed: int | None = None) -> str:
    attempts = max(1, MAX_RETRIES)
    model = genai.GenerativeModel(MODEL_NAME)
    delay = 1.2
    for attempt in range(1, attempts + 1):
        try:
            resp = model.generate_content(
                prompt,
                generation_config=generation_config_json,
                request_options={"timeout": 90},
                safety_settings=None,
            )
            txt = getattr(resp, "text", None)
            return (txt if txt is not None else str(resp)).strip()
        except (ResourceExhausted, DeadlineExceeded, InternalServerError) as e:
            if attempt == attempts:
                return f"[ERROR after {attempt} attempts] {e}"
            time.sleep(delay * (2 ** (attempt - 1)) * (1 + random.uniform(0, 0.25)))
        except Exception as e:
            if attempt >= min(3, attempts):
                return f"[ERROR non-retryable? attempt {attempt}] {e}"
            time.sleep(delay * attempt)


# ============================
# PROMPT BUILDERS (2-STAGE)
# ============================

def build_stage1_prompt(question: str) -> str:
    """
    Stage 1:
    - Select relevant FILTER CATEGORY NAMES (names only, no values yet)
    - Select up to MAX_ADDITIONAL_COLS additional columns, knowing that REQUIRED_COLS are always included
    - Return ONLY JSON with keys: selected_filter_names (list[str]) and selected_column (object with enumerated keys)
    """

    total_slots = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    schema_columns = ',\n'.join([f'    "Column {i}": "Name"' for i in range(1, total_slots + 1)])

    schema = (
        '{\n'
        '  "selected_filter_names": ["Filter Category 1", "Filter Category 2"],\n'
        '  "selected_column": {\n'
        f'{schema_columns}\n'
        '  }\n'
        '}'
    )

    # A small, concrete example (not prescriptive)
    example_obj = {
        "selected_filter_names": ["Cancer type", "Trial phase", "Class of ICI"],
        "selected_column": {
            "Column 1": "NCT",
            "Column 2": "PMID",
            "Column 3": "Authors",
            "Column 4": "Year",
            "Column 5": "Primary Endpoint(s)",
            "Column 6": "Sample Size"
        }
    }
    example = json.dumps(example_obj, ensure_ascii=False)
    return (
        "You are a medical expert researching cancer.\n"
        "You also have a set of available columns with definitions. You must choose only the following additional columns "
        f"for the question: {MAX_ADDITIONAL_COLS}.\n"
        "NCT, PMID, Authors, and Year are ALWAYS included by default unless explicitly excluded.\n\n"
        "Return ONLY a valid JSON object (no prose, no code fences) with this schema:\n"
        f"{schema}\n\n"
        "Rules:\n"
        '- Use double quotes.\n'
        "- Ensure NCT, PMID, Authors, and Year are present among the chosen columns, unless the question explicitly says otherwise.\n"
        f"- Choose at most {MAX_ADDITIONAL_COLS} additional columns beyond the four default columns.\n"
        "- Include only columns justified by the question.\n"
        "- Also select the relevant FILTER CATEGORY NAMES (do NOT assign values yet).\n"
        "- Use exact names from the lists provided.\n"
        "- Do not include any explanation.\n\n"
        "Available filter CATEGORY NAMES (choose names only; do NOT assign values):\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available columns and definitions (select by exact name):\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now.\n"
        f"Example of valid formatting (not prescriptive): {example}"
    )



def build_stage2_prompt(question: str, selected_filter_names: list[str], selected_column_obj: dict) -> str:
    """
    Stage 2:
    - Based on selected filter category NAMES, pick specific values using full filter definitions.
    - Keep defaults (NCT, PMID, Authors, Year) and at most MAX_ADDITIONAL_COLS additional columns as context.
    - Return ONLY JSON with key: selected_filter (mapping category -> chosen value).
    """

    schema = (
        '{\n'
        '  "selected_filter": {\n'
        '    "Filter Category 1": "Chosen Value",\n'
        '    "Filter Category 2": "Chosen Value"\n'
        '  }\n'
        '}'
    )
    chosen_cols_json = json.dumps(selected_column_obj, ensure_ascii=False)
    chosen_names_json = json.dumps(selected_filter_names, ensure_ascii=False)

    # A tiny example (not prescriptive)
    example = '{"selected_filter":{"Cancer type":"NSCLC","Trial phase":"Phase III"}}'

    return (
        "You are a medical expert researching cancer.\n"
        "You also have a set of available columns with definitions.\n"
        f"You are given the following columns (REQUIRED + up to {MAX_ADDITIONAL_COLS} additional):\n"
        f"{chosen_cols_json}\n\n"
        "Return ONLY a valid JSON object (no prose, no code fences) with this schema:\n"
        f"{schema}\n\n"
        "Rules:\n"
        '- Use double quotes.\n'
        "- Ensure NCT, PMID, Authors, and Year are present among the chosen columns, unless the question explicitly says otherwise.\n"
        f"- Choose only the {MAX_ADDITIONAL_COLS} additional columns beyond the four default columns.\n"
        "- Include only columns justified by the question.\n"
        "- Based on the filter category chosen and the filter definitions, choose the appropriate filter values.\n"
        "- Omit any filters whose correct value would be 'All'.\n"
        "- Include only filters that are justified by the question.\n"
        "- Use exact names/strings from the lists provided.\n"
        "- Do not include any explanation.\n\n"
        "Chosen filter CATEGORY NAMES (names only; restrict your value choices to these):\n"
        f"{chosen_names_json}\n\n"
        "Available filter CATEGORY NAMES (for reference):\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available filter categories and definitions (use exact category and value strings):\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        "Available columns and definitions (for context when choosing filters):\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now.\n"
        f"Example of valid formatting (not prescriptive): {example}"
    )



# ============================
# NORMALIZATION & CONVERSION
# ============================

def _ordered_values_from_column_object(col_obj: dict) -> list[str]:
    """Accepts a mapping like {"Column 1": "NCT", ...} and returns values ordered by numeric index."""
    if not isinstance(col_obj, dict):
        return []
    items = []
    for k, v in col_obj.items():
        m = re.search(r"Column\s*(\d+)", str(k))
        idx = int(m.group(1)) if m else 10**9
        items.append((idx, v))
    items.sort(key=lambda x: x[0])
    return [val for _, val in items if isinstance(val, str) and val.strip()]


def normalize_selected_column_object(selected_column) -> dict:
    """Ensure REQUIRED_COLS are present first, dedupe, and rebuild enumerated mapping."""
    # Convert input (list or dict) → ordered list
    if isinstance(selected_column, dict):
        cols = _ordered_values_from_column_object(selected_column)
    elif isinstance(selected_column, list):
        cols = [c for c in selected_column if isinstance(c, str) and c.strip()]
    else:
        cols = []

    # Ensure required at the front, keep order & dedupe
    out_list = []
    seen = set()
    for rc in REQUIRED_COLS:
        if rc not in seen:
            out_list.append(rc)
            seen.add(rc)
    for c in cols:
        if c not in seen:
            seen.add(c)
            out_list.append(c)

    # Keep only up to REQUIRED + MAX_ADDITIONAL_COLS
    cap = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    out_list = out_list[:cap]

    # Build enumerated object
    out_obj = {f"Column {i+1}": name for i, name in enumerate(out_list)}
    return out_obj


def normalize_filter_names(names: list[str]) -> list[str]:
    names = [n for n in (names or []) if isinstance(n, str) and n.strip()]
    seen = set()
    unique = []
    for n in names:
        if n not in seen:
            seen.add(n)
            unique.append(n)
    return unique[:10]




def normalize_selected_filter(selected_filter: dict) -> dict:
    # Drop empty/None/"All" values
    out = {}
    for k, v in (selected_filter or {}).items():
        if v is None:
            continue
        s = str(v).strip()
        if not s or s.lower() == "all":
            continue
        out[str(k).strip()] = s
    return out


# ============================
# CORE RUNNERS
# ============================

def canonicalize_names(names):
    out = []
    for n in names:
        nn = FILTER_NAME_CANON.get(n.strip(), n.strip())
        if nn not in out:
            out.append(nn)
    return out

def run_once_for_question(question: str, seed: int | None = None) -> tuple[dict, dict]:
    # Stage 1
    p1 = build_stage1_prompt(question)
    start1 = time.time()
    t1 = call_model_with_retries(p1, seed=seed)
    elapsed1 = time.time() - start1
    try:
        o1 = extract_json(t1)
    except Exception:
        o1 = {}

    selected_filter_names_raw = o1.get("selected_filter_names", [])
    selected_filter_names = normalize_filter_names(selected_filter_names_raw)
    selected_filter_names = canonicalize_names(selected_filter_names)  # <- ensure canonical labels

    selected_column_obj_raw = o1.get("selected_column", {})
    selected_column_obj = normalize_selected_column_object(selected_column_obj_raw)

    # Stage 2
    p2 = build_stage2_prompt(question, selected_filter_names, selected_column_obj)
    start2 = time.time()
    t2 = call_model_with_retries(p2, seed=seed)
    elapsed2 = time.time() - start2
    try:
        o2 = extract_json(t2)
    except Exception:
        o2 = {}

    selected_filter = normalize_selected_filter(o2.get("selected_filter", {}))

    merged = {
        "selected_filter": selected_filter,
        "selected_column": selected_column_obj,
    }
    timings = {
        "stage1_time_sec": elapsed1,
        "stage2_time_sec": elapsed2,
        "total_time_sec": elapsed1 + elapsed2,
    }
    return merged, timings


# ============================
# MAIN
# ============================
# --- Canonical names for FILTER CATEGORIES (keys) ---
FILTER_NAME_CANON = {
    # Core from GTs
    "Class of ICI": "Class of ICI",
    "Cancer type": "Cancer type",
    "Monotherapy/combination": "Monotherapy/combination",
    "Trial phase": "Trial phase",
    "Clinical setting in relation to surgery": "Clinical setting in relation to surgery",
    "Primary endpoint": "Primary endpoint",
    "Total sample size": "Total sample size",
    "Year": "Year",

    "ICI Class": "Class of ICI",
    "ICI class": "Class of ICI",
    "Checkpoint class": "Class of ICI",

    "Cancer Type": "Cancer type",
    "Disease": "Cancer type",
    "Indication": "Cancer type",
    "Cancer type – The specific disease/indication under study (e.g., metastatic castration-sensitive prostate cancer, NSCLC, melanoma), including stage/setting when relevant.": "Cancer type",

    "Type of Therapy": "Monotherapy/combination",
    "Monotherapy/Combination": "Monotherapy/combination",
    "Monotherapy vs Combination": "Monotherapy/combination",
    "Mono/Combo": "Monotherapy/combination",

    "Trial Phase": "Trial phase",
    "Clinical Trial Phase": "Trial phase",
    "Phase": "Trial phase",

    "Clinical setting": "Clinical setting in relation to surgery",
    "Perioperative setting": "Clinical setting in relation to surgery",
    "Setting in relation to surgery": "Clinical setting in relation to surgery",
}

# def main():
#     df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)
#
#     # auto-detect question column if needed
#     global QUESTION_COLUMN
#     if QUESTION_COLUMN not in df.columns:
#         candidates = [c for c in df.columns if str(c).strip().lower() in {"question", "query", "prompt"}]
#         if candidates:
#             QUESTION_COLUMN = candidates[0]
#         else:
#             raise ValueError(
#                 f"Couldn't find a question column named '{QUESTION_COLUMN}'. Available columns: {list(df.columns)}"
#             )
#
#     # Prepare output columns: run_1..run_N for merged JSON
#     run_cols = [f"run_{i+1}" for i in range(RUNS_PER_QUESTION)]
#     for c in run_cols:
#         if c not in df.columns:
#             df[c] = ""
#
#     # Iterate and fill
#     for idx, row in df.iterrows():
#         q = str(row[QUESTION_COLUMN]).strip()
#         if not q or q.lower() == "nan":
#             continue
#         print(f"Processing row {idx}: {q[:80]}{'...' if len(q)>80 else ''}")
#         merged_runs = run_n_times_for_question(q, RUNS_PER_QUESTION)
#         for i, merged_json in enumerate(merged_runs):
#             df.at[idx, run_cols[i]] = merged_json
#
#     # Save
#     df.to_excel(EXCEL_PATH_OUT, index=False)
#     print(f"Saved results to: {EXCEL_PATH_OUT}")
#
#
# if __name__ == "__main__":
#     main()

def main():
    df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)

    global QUESTION_COLUMN
    if QUESTION_COLUMN not in df.columns:
        candidates = [c for c in df.columns if str(c).strip().lower() in {"question","query","prompt"}]
        if candidates:
            QUESTION_COLUMN = candidates[0]
        else:
            raise ValueError(f"Couldn't find a question column. Available: {list(df.columns)}")

    run_cols = [f"run_{i+1}" for i in range(RUNS_PER_QUESTION)]
    for c in run_cols:
        if c not in df.columns:
            df[c] = ""

    timings_records = []

    for idx,row in df.iterrows():
        q = str(row[QUESTION_COLUMN]).strip()
        if not q or q.lower()=="nan": continue
        print(f"Processing row {idx}: {q[:80]}{'...' if len(q)>80 else ''}")
        merged_runs, timings_list = run_n_times_for_question(q, RUNS_PER_QUESTION)
        for i,merged_json in enumerate(merged_runs):
            df.at[idx, run_cols[i]] = merged_json
        timings_records.append({
            "row_index": idx,
            "question": q,
            **{f"run_{i+1}_stage1_sec": t["stage1_time_sec"] for i,t in enumerate(timings_list)},
            **{f"run_{i+1}_stage2_sec": t["stage2_time_sec"] for i,t in enumerate(timings_list)},
            **{f"run_{i+1}_total_sec":  t["total_time_sec"]  for i,t in enumerate(timings_list)},
        })

    timings_df = pd.DataFrame(timings_records)
    with pd.ExcelWriter(EXCEL_PATH_OUT, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="results")
        timings_df.to_excel(writer, index=False, sheet_name="timings")

    print(f"Saved results to: {EXCEL_PATH_OUT}")

if __name__ == "__main__":
    main()


In [ ]:
# """
# Pipeline 3 - Stricter single prompt build up
# """

# import os, time, random, json, ast, re, math, textwrap, traceback
# import pandas as pd
# from dotenv import load_dotenv
# from typing import Any, Dict, List, Tuple
# from tqdm.auto import tqdm, trange  # NEW: tqdm progress bars

# # ================
# # CONFIG
# # ================
# EXCEL_PATH_IN  = "runs/query-chosen-filters-columns.xlsx"
# EXCEL_PATH_OUT = "runs/query-2.5-chosen-filters-columns_with_runs_pipeline3.xlsx"
# SHEET_NAME     = 0  # or the sheet name
# QUESTION_COLUMN_CANDIDATES = ["question", "Query", "QUESTIONS", "queries", "Questions"]
# GROUND_TRUTH_COLUMN_CANDIDATES = ["Ground Truth JSON", "ground_truth_json", "ground truth", "gt_json"]

# RUNS_PER_QUESTION = 3
# DELAY_BETWEEN_CALLS_SEC = 0.0
# MAX_RETRIES = 3
# MODEL_NAME = "gemini-2.5-flash"

# # Definition files (pointed to your uploaded assets)
# FILTER_NAMES_DEF_PATH = "definitions_folder/definitions - aim2 - filter names.txt"  # filter CATEGORY NAMES only
# COLUMN_DEFS_PATH      = "definitions_folder/definitions - aim2 - column.txt"        # column definitions (names + explanations)
# FILTER_DEFS_FULL_PATH = "definitions_folder/definitions - aim2 - filter.txt"        # full filter (categories + values)

# # Columns always included in "selected_column" (do NOT count toward MAX_ADDITIONAL_COLS)
# REQUIRED_COLS = ["NCT", "PMID", "Authors", "Year"]
# MAX_ADDITIONAL_COLS = 6  # model may add at most this many on top of REQUIRED_COLS

# # ================
# # GEMINI SETUP
# # ================
# load_dotenv()
# GEMINI_KEY = os.getenv("GEMINI_KEY")
# if not GEMINI_KEY:
#     raise RuntimeError("GEMINI_KEY environment variable is not set")

# try:
#     import google.generativeai as genai
#     from google.api_core.exceptions import ResourceExhausted, DeadlineExceeded, InternalServerError
# except Exception:
#     raise RuntimeError("Please `pip install google-generativeai`")

# genai.configure(api_key=GEMINI_KEY)
# generation_config_json = {"response_mime_type": "application/json",
#                              "temperature": 0.0,
#  }

# # ================
# # LOAD DEFINITIONS
# # ================
# def _read_text(path: str) -> str:
#     with open(path, "r", encoding="utf-8") as f:
#         return f.read()

# FILTER_NAMES_TEXT = _read_text(FILTER_NAMES_DEF_PATH)
# COLUMN_DEFS_TEXT  = _read_text(COLUMN_DEFS_PATH)
# FILTER_DEFS_TEXT  = _read_text(FILTER_DEFS_FULL_PATH)

# # Try to extract canonical column names from COLUMN_DEFS_TEXT (left side of an "–" en dash or "-")
# def _candidate_column_names_from_defs(def_text: str) -> List[str]:
#     names = []
#     for line in def_text.splitlines():
#         line = line.strip()
#         if not line:
#             continue
#         # Grab left side before an en dash or hyphen (common format: "Name – Explanation")
#         m = re.split(r"\s+[–-]\s+", line, maxsplit=1)
#         if m:
#             left = m[0].strip()
#             # Weed out section headers
#             if 1 <= len(left) <= 80 and " " in left or left.isalpha():
#                 names.append(left)
#     # Ensure required cols are present
#     for rc in REQUIRED_COLS:
#         if rc not in names:
#             names.append(rc)
#     return sorted(set(names), key=lambda x: x.lower())

# CANONICAL_COLUMN_NAMES = _candidate_column_names_from_defs(COLUMN_DEFS_TEXT)

# # ================
# # ROBUST JSON PARSING
# # ================
# def strip_code_fences(s: str) -> str:
#     s = s.strip()
#     if s.startswith("```") and s.endswith("```"):
#         s = s.strip("`")
#         s = "\n".join(s.splitlines()[1:])
#     return s.strip()

# def extract_json(text: str):
#     raw = strip_code_fences(text or "")
#     # First try direct JSON
#     try:
#         return json.loads(raw)
#     except Exception:
#         pass
#     # Then try trimming to first {...}
#     try:
#         start = raw.find("{")
#         end = raw.rfind("}")
#         if start != -1 and end != -1 and end > start:
#             candidate = raw[start : end + 1]
#             try:
#                 return json.loads(candidate)
#             except Exception:
#                 try:
#                     obj = ast.literal_eval(candidate)
#                     if isinstance(obj, (dict, list)):
#                         return obj
#                 except Exception:
#                     pass
#     except Exception:
#         pass
#     raise ValueError(f"Could not parse JSON from model text: {text[:200]}...")

# # ================
# # MODEL CALL + PROMPT
# # ================
# def call_model_with_retries(prompt: str) -> str:
#     attempts = max(1, MAX_RETRIES)
#     model = genai.GenerativeModel(MODEL_NAME)
#     base_delay = 1.0
#     last_err = None
#     for attempt in range(1, attempts + 1):
#         try:
#             resp = model.generate_content(
#                 prompt,
#                 generation_config=generation_config_json,
#                 request_options={"timeout": 90},
#                 safety_settings=None,
#             )
#             txt = getattr(resp, "text", None)
#             if txt is None:
#                 txt = str(resp)
#             return txt.strip()
#         except (ResourceExhausted, DeadlineExceeded, InternalServerError) as e:
#             last_err = e
#             if attempt < attempts:
#                 time.sleep(base_delay * (2 ** (attempt - 1)) * (1 + random.uniform(0, 0.3)))
#             else:
#                 return f"[ERROR after {attempt} attempts] {e}"
#         except Exception as e:
#             last_err = e
#             if attempt < min(3, attempts):
#                 time.sleep(base_delay * attempt)
#             else:
#                 return f"[ERROR non-retryable? attempt {attempt}] {e}"
#     return f"[ERROR] {last_err}"

# def build_single_stage_prompt(question: str) -> str:
#     schema = (
#         '{\n'
#         '  "selected_filter": {\n'
#         '    "Filter Category 1": "Chosen Value",\n'
#         '    "Filter Category 2": "Chosen Value"\n'
#         '  },\n'
#         '  "selected_column": {\n'
#         '    "Column 1": "Name",\n'
#         '    "Column 2": "Name",\n'
#         '    "Column 3": "Name",\n'
#         '    "Column 4": "Name",\n'
#         '    "Column 5": "Name",\n'
#         '    "Column 6": "Name",\n'
#         '    "Column 7": "Name",\n'
#         '    "Column 8": "Name",\n'
#         '    "Column 9": "Name",\n'
#         '    "Column 10": "Name"\n'
#         '  }\n'
#         '}'
#     )
#     example = (
#         '{'
#         '"selected_filter":{"Cancer Type":"NSCLC","Trial Phase":"Phase 3","Monotherapy/combination":"Combination"},'
#         '"selected_column":{"Column 1":"NCT","Column 2":"PMID","Column 3":"Authors","Column 4":"Year","Column 5":"Primary endpoint","Column 6":"Total sample size","Column 7":"Name of ICI","Column 8":"Control regimen","Column 9":"Monotherapy/combination","Column 10":"Lines of treatment"}'
#         '}'
#     )
#     return textwrap.dedent(f"""
#         You are a medical expert researching cancer trials. Build the ENTIRE selection in ONE step.
#         Return ONLY a valid JSON object (no prose, no code fences) with this schema:
#         {schema}

#         Strict rules:
#         - Columns: ALWAYS include these first (do NOT count toward the limit): {', '.join(REQUIRED_COLS)}.
#         - You may add at most {MAX_ADDITIONAL_COLS} additional columns beyond those required.
#         - The enumerated object keys MUST be exactly "Column 1", "Column 2", ... in display order.
#         - Use EXACT names from the available column list.
#         - Filters: choose only categories/values justified by the question. If a category would be "All", OMIT that category entirely.
#         - Use EXACT category/value strings from the available filter definitions.
#         - Use double quotes everywhere. No trailing commas. No explanations.

#         Available filter CATEGORY NAMES (for reference):
#         {FILTER_NAMES_TEXT}

#         Available FULL filter categories and values (use exact strings; omit categories that would be "All"):
#         {FILTER_DEFS_TEXT}

#         Available columns and definitions (select by EXACT name):
#         {COLUMN_DEFS_TEXT}

#         Question: {question}

#         Return JSON now. Example of valid formatting (not prescriptive):
#         {example}
#     """).strip()

# # ================
# # NORMALIZATION HELPERS
# # ================
# def _ordered_values_from_column_object(col_obj: dict) -> List[str]:
#     """Accepts {'Column 1': 'NCT', ...} and returns values ordered by numeric index."""
#     if not isinstance(col_obj, dict):
#         return []
#     items = []
#     for k, v in col_obj.items():
#         m = re.search(r"Column\s*(\d+)", str(k))
#         idx = int(m.group(1)) if m else 10**9
#         items.append((idx, v))
#     items.sort(key=lambda x: x[0])
#     return [str(val).strip() for _, val in items if isinstance(val, str) and str(val).strip()]

# def normalize_selected_column_object(selected_column) -> dict:
#     """Ensure REQUIRED_COLS are present first, dedupe, and rebuild enumerated mapping with cap."""
#     if isinstance(selected_column, dict):
#         cols = _ordered_values_from_column_object(selected_column)
#     elif isinstance(selected_column, list):
#         cols = [c for c in selected_column if isinstance(c, str) and c.strip()]
#     else:
#         cols = []

#     out_list, seen = [], set()
#     for rc in REQUIRED_COLS:
#         if rc not in seen:
#             out_list.append(rc); seen.add(rc)
#     for c in cols:
#         if c not in seen:
#             seen.add(c); out_list.append(c)

#     cap = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
#     out_list = out_list[:cap]
#     return {f"Column {i+1}": name for i, name in enumerate(out_list)}

# def normalize_selected_filter(selected_filter: dict) -> dict:
#     """Drop empty/None/'All' values and trim keys/values. (No synonym mapping here.)"""
#     out = {}
#     for k, v in (selected_filter or {}).items():
#         if v is None:
#             continue
#         s = str(v).strip()
#         if not s or s.lower() == "all":
#             continue
#         out[str(k).strip()] = s
#     return out

# # ================
# # RELAXED CANONICALIZATION (handles common mismatches)
# # ================
# def _simplify(s: str) -> str:
#     s = (s or "").strip().lower()
#     s = s.replace("—", "-").replace("–", "-")
#     # Drop parentheticals often used for clarifications e.g., "(NSCLC)"
#     s = re.sub(r"\([^)]*\)", "", s)
#     # Remove long " - explanation" tails (keep left-most meaningful name)
#     s = re.sub(r"\s+-\s+.*$", "", s)
#     s = re.sub(r"\s+–\s+.*$", "", s)
#     # Normalize slashes & pluses spacing
#     s = re.sub(r"\s*/\s*", " / ", s)
#     s = re.sub(r"\s*\+\s*", " + ", s)
#     # Remove non-alphanum except + / spaces
#     s = re.sub(r"[^a-z0-9+/ ]+", " ", s)
#     s = re.sub(r"\s+", " ", s).strip()
#     return s

# # Key/category alias map → canonical key
# KEY_ALIASES = {
#     "cancer type": "cancer type",
#     "class of ici": "class of ici",
#     "monotherapy/combination": "monotherapy/combination",
#     "trial phase": "trial phase",
#     "clinical setting": "clinical setting in relation to surgery",
#     "clinical setting in relation to surgery": "clinical setting in relation to surgery",
#     "name of ici": "name of ici",
#     "ici name": "name of ici",
#     "type of combination": "type of combination",
#     "type of combination (treatment arm)": "type of combination",
#     "type of control": "type of control",
#     "control arm": "type of control",
#     "primary endpoint": "primary endpoint",
#     "total sample size": "total sample size",
#     "treatment regimen": "treatment regimen",
#     "control regimen": "control regimen",
#     "lines of treatment": "lines of treatment",
#     "study name": "study name",
#     "trial phase - the stage or step in the clinical trial process typically denoted by a roman numeral e g i ii iii iv which defines the scope objectives and duration of the trial": "trial phase",
# }

# # Value synonyms (right-hand side of filters)
# def _canon_value(val: str) -> str:
#     v = _simplify(val)
#     # PD axis
#     v = v.replace("pd 1", "pd-1").replace("pd1", "pd-1").replace("pd- 1", "pd-1")
#     v = v.replace("pd l1", "pd-l1").replace("pdl1", "pd-l1").replace("pd- l1", "pd-l1")
#     v = v.replace("ctla 4", "ctla-4").replace("ctla4", "ctla-4").replace("ctla- 4", "ctla-4")
#     # Cancer abbreviations
#     v = v.replace("non small cell lung cancer", "nsclc")
#     v = v.replace("head and neck", "hnscc")
#     v = v.replace("renal cell carcinoma", "rcc")
#     v = v.replace("hepatocellular carcinoma", "hcc")
#     v = v.replace("bladder", "urothelial") if "urothelial" in v else v
#     v = v.replace("gastric / gej", "gastric gej").replace("gej / esophageal / gej", "gastric gej")
#     # Setting
#     repl = {
#         "combination therapy": "combination",
#         "combination": "combination",
#         "monotherapy": "monotherapy",
#         "adjuvant": "adjuvant",
#         "neoadjuvant": "neoadjuvant",
#         "maintenance": "maintenance",
#         "perioperative": "perioperative",
#         "metastatic recurrent unresectable": "metastatic/recurrent",
#     }
#     for k, tgt in repl.items():
#         if v == k:
#             v = tgt
#     # Type of combination shorthand
#     v = v.replace("ici + chemo", "ici+chemo")
#     v = v.replace("ici + tki", "ici+tki")
#     v = v.replace("ici + radiation", "ici+radiation")
#     v = v.replace("ici + anti vegf", "ici+anti-vegf")
#     v = v.replace("ici + vaccine", "ici+vaccine")
#     v = v.replace("ici + ici", "ici+ici")
#     v = v.replace("ici + ici + chemo", "ici+ici+chemo")
#     v = v.replace("ici + chemo + anti vegf", "ici+chemo+anti-vegf")
#     # Type of control shorthand
#     v = v.replace("chemo / anti egfr", "chemo/anti-egfr")
#     v = v.replace("chemo / anti vegf", "chemo/anti-vegf")
#     v = v.replace("best supportive care bsc", "bsc")
#     return v

# def canon_key(k: str) -> str:
#     ks = _simplify(k)
#     return KEY_ALIASES.get(ks, ks)

# def canon_colname(c: str) -> str:
#     c0 = _simplify(c)
#     col_aliases = {
#         "primary endpoint": "primary endpoint",
#         "total sample size": "total sample size",
#         "treatment regimen": "treatment regimen",
#         "control regimen": "control regimen",
#         "name of ici": "name of ici",
#         "class of ici": "class of ici",
#         "monotherapy/combination": "monotherapy/combination",
#         "type of combination": "type of combination",
#         "type of control": "type of control",
#         "cancer type": "cancer type",
#         "trial phase": "trial phase",
#         "clinical setting": "clinical setting in relation to surgery",
#         "clinical setting in relation to surgery": "clinical setting in relation to surgery",
#         "lines of treatment": "lines of treatment",
#         "study name": "study name",
#         "type of study": "type of study",
#         "follow up duration for primary endpoints overall rx control": "follow up duration for primary endpoints (overall, rx, control)",
#     }
#     if c.strip() in REQUIRED_COLS:
#         return c.strip()
#     return col_aliases.get(c0, c.strip())

# def canonicalize_filter_dict(d: Dict[str, str]) -> Dict[str, str]:
#     out = {}
#     for k, v in (d or {}).items():
#         if v is None:
#             continue
#         ck = canon_key(k)
#         cv = _canon_value(v)
#         if ck and cv:
#             out[ck] = cv
#     return out

# def canonicalize_columns_list_from_obj(selected_column_obj: Dict[str, str]) -> List[str]:
#     raw = _ordered_values_from_column_object(selected_column_obj)
#     return [canon_colname(c) for c in raw]

# # ================
# # METRICS
# # ================
# def _set_from_cols_obj(col_obj: Dict[str, str], relaxed: bool) -> set:
#     vals = _ordered_values_from_column_object(col_obj)
#     if relaxed:
#         vals = [canon_colname(v) for v in vals]
#     return set(vals)

# def precision_recall_filters(gt: Dict[str, str], pred: Dict[str, str], relaxed: bool) -> Tuple[float, float]:
#     if relaxed:
#         gt_c = canonicalize_filter_dict(gt)
#         pr_c = canonicalize_filter_dict(pred)
#     else:
#         gt_c = {str(k).strip(): str(v).strip() for k, v in (gt or {}).items() if str(v).strip()}
#         pr_c = {str(k).strip(): str(v).strip() for k, v in (pred or {}).items() if str(v).strip()}
#     correct = 0
#     for k, v in pr_c.items():
#         if k in gt_c and gt_c[k] == v:
#             correct += 1
#     p_den = len(pr_c)
#     r_den = len(gt_c)
#     precision = correct / p_den if p_den else (1.0 if r_den == 0 else 0.0)
#     recall    = correct / r_den if r_den else (1.0 if p_den == 0 else 0.0)
#     return precision, recall

# def precision_recall_columns(gt_obj: Dict[str, str], pr_obj: Dict[str, str], relaxed: bool) -> Tuple[float, float]:
#     gt_set = _set_from_cols_obj(gt_obj, relaxed=relaxed)
#     pr_set = _set_from_cols_obj(pr_obj, relaxed=relaxed)
#     inter = len(gt_set & pr_set)
#     p_den = len(pr_set)
#     r_den = len(gt_set)
#     precision = inter / p_den if p_den else (1.0 if r_den == 0 else 0.0)
#     recall    = inter / r_den if r_den else (1.0 if p_den == 0 else 0.0)
#     return precision, recall

# # ================
# # I/O HELPERS
# # ================
# def detect_column(df: pd.DataFrame, candidates: List[str]) -> str:
#     cols = list(df.columns)
#     for cand in candidates:
#         for c in cols:
#             if str(c).strip().lower() == cand.strip().lower():
#                 return c
#     for c in cols:
#         if "question" in str(c).lower():
#             return c
#     for c in cols:
#         if "ground" in str(c).lower() and "json" in str(c).lower():
#             return c
#     raise KeyError(f"Could not detect a column among candidates {candidates}. Columns present: {cols}")

# # ================
# # MAIN
# # ================
# def main():
#     # Load input
#     try:
#         df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)
#     except Exception:
#         df = pd.read_excel(EXCEL_PATH_IN)  # try default

#     q_col = detect_column(df, QUESTION_COLUMN_CANDIDATES)
#     gt_col = detect_column(df, GROUND_TRUTH_COLUMN_CANDIDATES)

#     # Prepare result columns
#     run_cols = [f"run_{i+1}" for i in range(RUNS_PER_QUESTION)]
#     for rc in run_cols:
#         if rc not in df.columns:
#             df[rc] = ""

#     # Metrics columns (strict + relaxed)
#     metric_cols = [
#         "avg_precision_filters", "avg_recall_filters",
#         "avg_precision_columns", "avg_recall_columns",
#         "avg_precision_filters_relaxed", "avg_recall_filters_relaxed",
#         "avg_precision_columns_relaxed", "avg_recall_columns_relaxed",
#     ]
#     for mc in metric_cols:
#         if mc not in df.columns:
#             df[mc] = float("nan")

#     # Optional row id
#     if "row_id" not in df.columns:
#         df["row_id"] = range(len(df))

#     # === NEW: tqdm progress over questions ===
#     total_rows = len(df)
#     row_pbar = tqdm(total=total_rows, desc="Questions", unit="row")
#     try:
#         for idx, row in df.iterrows():
#             question = str(row[q_col]).strip() if not pd.isna(row[q_col]) else ""
#             gt_raw = row[gt_col]
#             # Parse ground truth JSON
#             try:
#                 gt_json = extract_json(gt_raw) if isinstance(gt_raw, str) else gt_raw
#             except Exception:
#                 gt_json = {}
#             gt_filters = normalize_selected_filter((gt_json or {}).get("selected_filter", {}))
#             gt_columns = normalize_selected_column_object((gt_json or {}).get("selected_column", {}))

#             # Bookkeeping for metrics across runs
#             pfs, rfs, pcs, rcs = [], [], [], []                # strict
#             pfs_rel, rfs_rel, pcs_rel, rcs_rel = [], [], [], [] # relaxed

#             # === NEW: tqdm for per-question runs ===
#             run_pbar = tqdm(total=RUNS_PER_QUESTION, desc=f"Runs q{idx+1}", unit="run", leave=False)
#             try:
#                 for r in range(RUNS_PER_QUESTION):
#                     colname = f"run_{r+1}"
#                     already = str(row[colname]) if colname in df.columns and pd.notna(row[colname]) else ""
#                     if already.strip().startswith("{") and already.strip().endswith("}"):
#                         try:
#                             pred_json = extract_json(already)
#                         except Exception:
#                             pred_json = {}
#                     else:
#                         # Build prompt + call model
#                         prompt = build_single_stage_prompt(question)
#                         resp = call_model_with_retries(prompt)
#                         time.sleep(DELAY_BETWEEN_CALLS_SEC)

#                         # Parse model output → JSON
#                         try:
#                             pred_json = extract_json(resp)
#                         except Exception:
#                             pred_json = {}
#                         # Normalize, then store JSON string compactly
#                         pred_filters = normalize_selected_filter((pred_json or {}).get("selected_filter", {}))
#                         pred_columns = normalize_selected_column_object((pred_json or {}).get("selected_column", {}))
#                         pred_json_norm = {"selected_filter": pred_filters, "selected_column": pred_columns}
#                         df.at[idx, colname] = json.dumps(pred_json_norm, ensure_ascii=False)

#                     # Compute metrics (strict + relaxed)
#                     pf, rf = precision_recall_filters(gt_filters, (pred_json or {}).get("selected_filter", {}), relaxed=False)
#                     pc, rc = precision_recall_columns(gt_columns, (pred_json or {}).get("selected_column", {}), relaxed=False)
#                     pfs.append(pf); rfs.append(rf); pcs.append(pc); rcs.append(rc)

#                     pf2, rf2 = precision_recall_filters(gt_filters, (pred_json or {}).get("selected_filter", {}), relaxed=True)
#                     pc2, rc2 = precision_recall_columns(gt_columns, (pred_json or {}).get("selected_column", {}), relaxed=True)
#                     pfs_rel.append(pf2); rfs_rel.append(rf2); pcs_rel.append(pc2); rcs_rel.append(rc2)

#                     run_pbar.update(1)
#             finally:
#                 run_pbar.close()

#             # Averages
#             df.at[idx, "avg_precision_filters"] = sum(pfs) / len(pfs) if pfs else float("nan")
#             df.at[idx, "avg_recall_filters"]    = sum(rfs) / len(rfs) if rfs else float("nan")
#             df.at[idx, "avg_precision_columns"] = sum(pcs) / len(pcs) if pcs else float("nan")
#             df.at[idx, "avg_recall_columns"]    = sum(rcs) / len(rcs) if rcs else float("nan")

#             df.at[idx, "avg_precision_filters_relaxed"] = sum(pfs_rel) / len(pfs_rel) if pfs_rel else float("nan")
#             df.at[idx, "avg_recall_filters_relaxed"]    = sum(rfs_rel) / len(rfs_rel) if rfs_rel else float("nan")
#             df.at[idx, "avg_precision_columns_relaxed"] = sum(pcs_rel) / len(pcs_rel) if pcs_rel else float("nan")
#             df.at[idx, "avg_recall_columns_relaxed"]    = sum(rcs_rel) / len(rcs_rel) if rcs_rel else float("nan")

#             row_pbar.update(1)
#     finally:
#         row_pbar.close()

#     # Save output
#     os.makedirs(os.path.dirname(EXCEL_PATH_OUT), exist_ok=True)
#     with pd.ExcelWriter(EXCEL_PATH_OUT, engine="openpyxl") as xw:
#         # Renamed to pipeline3_runs to match this script
#         df.to_excel(xw, index=False, sheet_name="pipeline3_runs")

#     tqdm.write(f"Done. Wrote: {EXCEL_PATH_OUT}")

# if __name__ == "__main__":
#     try:
#         main()
#     except Exception as e:
#         traceback.print_exc()
#         raise


In [ ]:
"""
Pipeline 3 — Three-stage selection:
  Stage 1: choose the appropriate FILTERS (categories + values)
  Stage 2: choose the appropriate COLUMNS
  Stage 3: re-evaluate filters + columns together; accept or revise

Notes:
- Uses the same definition files and environment variable layout you already have.
- Canonicalizes filter NAMES to your new interface terms (e.g., "ICI Class", "Clinical Setting", etc.).
- Canonicalizes common filter VALUE synonyms (e.g., PD1->PD-1, NSCLC->Non-Small Cell Lung Cancer (NSCLC), 2L->Second-line+).
- Enforces REQUIRED_COLS and cap on additional columns.
- Writes the results + timings to an Excel file.

Env:
  GEMINI_KEY must be set (e.g., via .env)

Files:
  FILTER_NAMES_DEF_PATH  (filter CATEGORY names only)
  COLUMN_DEFS_PATH       (column definitions)
  FILTER_DEFS_FULL_PATH  (full filter categories + allowed values)

Output:
  runs/query-chosen-filters-columns_with_runs_pipeline4.xlsx
"""

import os, time, random, json, ast, re
import pandas as pd
from dotenv import load_dotenv

import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, DeadlineExceeded, InternalServerError

# ============================
# CONFIG
# ============================
EXCEL_PATH_IN   = "runs/query-chosen-filters-columns.xlsx"
EXCEL_PATH_OUT  = "runs/query-2.5-chosen-filters-columns_with_runs_pipeline4.xlsx"
SHEET_NAME      = 0
QUESTION_COLUMN = "question"  # auto-detected if missing
RUNS_PER_QUESTION = 3
DELAY_BETWEEN_CALLS_SEC = 0.0
MAX_RETRIES = 3
MAX_QUESTIONS = 100
MODEL_NAME = "gemini-2.5-flash"

# Definition files (reuse your existing)
FILTER_NAMES_DEF_PATH = "definitions_folder/definitions - aim2 - filter names.txt"
COLUMN_DEFS_PATH      = "definitions_folder/definitions - aim2 - column.txt"
FILTER_DEFS_FULL_PATH = "definitions_folder/definitions - aim2 - filter.txt"

# Columns always included (do NOT count toward MAX_ADDITIONAL_COLS)
REQUIRED_COLS = ["NCT", "PMID", "Authors", "Year"]
MAX_ADDITIONAL_COLS = 6  # number of columns the model may add beyond REQUIRED_COLS

# ============================
# SETUP
# ============================
load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
if not GEMINI_KEY:
    raise RuntimeError("GEMINI_KEY environment variable is not set")

genai.configure(api_key=GEMINI_KEY)

GENERATION_CONFIG = {
    "response_mime_type": "application/json",
    "temperature": 0.0,
}

with open(FILTER_NAMES_DEF_PATH, "r", encoding="utf-8") as f:
    FILTER_NAMES_TEXT = f.read()
with open(COLUMN_DEFS_PATH, "r", encoding="utf-8") as f:
    COLUMN_DEFS_TEXT = f.read()
with open(FILTER_DEFS_FULL_PATH, "r", encoding="utf-8") as f:
    FILTER_DEFS_TEXT = f.read()

# ============================
# JSON HELPERS
# ============================

def strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```") and s.endswith("```"):
        s = s.strip("`")
        s = "\n".join(s.splitlines()[1:])
    return s.strip()

def extract_json(text: str):
    raw = strip_code_fences(text or "")
    # Try pure JSON
    try:
        return json.loads(raw)
    except Exception:
        pass
    # Fallback: grab first {...} block
    try:
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = raw[start : end + 1]
            try:
                return json.loads(candidate)
            except Exception:
                try:
                    obj = ast.literal_eval(candidate)
                    if isinstance(obj, (dict, list)):
                        return obj
                except Exception:
                    pass
    except Exception:
        pass
    raise ValueError(f"Could not parse JSON from model text: {text[:200]}...")

def call_model_with_retries(prompt: str, seed: int | None = None) -> str:
    attempts = max(1, MAX_RETRIES)
    model = genai.GenerativeModel(MODEL_NAME)
    delay = 1.0
    for attempt in range(1, attempts + 1):
        try:
            resp = model.generate_content(
                prompt,
                generation_config=GENERATION_CONFIG,
                request_options={"timeout": 90},
                safety_settings=None,
            )
            txt = getattr(resp, "text", None)
            return (txt if txt is not None else str(resp)).strip()
        except (ResourceExhausted, DeadlineExceeded, InternalServerError) as e:
            if attempt == attempts:
                return f"[ERROR after {attempt} attempts] {e}"
            time.sleep(delay * (2 ** (attempt - 1)) * (1 + random.uniform(0, 0.35)))
        except Exception as e:
            if attempt >= min(3, attempts):
                return f"[ERROR non-retryable? attempt {attempt}] {e}"
            time.sleep(delay * attempt)

# ============================
# CANONICALIZATION (filter NAMES & VALUES)
# ============================

# Canonical filter category names for your interface:
CANON_KEYS = {
    # New interface names (final targets)
    "ICI Class": "ICI Class",
    "ICI Name": "ICI Name",
    "Cancer Type": "Cancer Type",
    "Type of Therapy": "Type of Therapy",
    "Type of combination (Treatment Arm)": "Type of combination (Treatment Arm)",
    "Control Arm": "Control Arm",
    "Clinical Setting": "Clinical Setting",
    "Trial Phase": "Trial Phase",
    "Type of Study": "Type of Study",
    "Primary Endpoint": "Primary Endpoint",
    "Included in MA": "Included in MA",

    # Older/synonymous names → new canonical names
    "Class of ICI": "ICI Class",
    "Name of ICI": "ICI Name",
    "Cancer type": "Cancer Type",
    "Monotherapy/combination": "Type of Therapy",
    "Monotherapy/Combination": "Type of Therapy",
    "Monotherapy vs Combination": "Type of Therapy",
    "Mono/Combo": "Type of Therapy",
    "Type of combination": "Type of combination (Treatment Arm)",
    "Type of combination (Treatment Arm)": "Type of combination (Treatment Arm)",
    "Type of control": "Control Arm",
    "Control regimen": "Control Arm",
    "Trial phase": "Trial Phase",
    "Clinical Trial Phase": "Trial Phase",
    "Phase": "Trial Phase",
    "Clinical setting in relation to surgery": "Clinical Setting",
    "Clinical Setting (Perioperative)": "Clinical Setting",
    "Clinical setting": "Clinical Setting",
    "Perioperative setting": "Clinical Setting",
    "Setting in relation to surgery": "Clinical Setting",
    "Primary endpoint": "Primary Endpoint",
    "Primary Endpoint(s)": "Primary Endpoint",
    "Primary endpoints": "Primary Endpoint",
    "Included in Meta-analysis": "Included in MA",
    # This one will be mapped into Clinical Setting values
    "Lines of treatment": "Clinical Setting",
}

# Value canon helpers per-category
def _canon_simple(value: str) -> str:
    return re.sub(r"\s+", " ", str(value or "").strip())

ICI_CLASS_VALUE_MAP = {
    "pd1": "PD-1",
    "pd-1": "PD-1",
    "pd 1": "PD-1",
    "pd_l1": "PD-L1",
    "pdl1": "PD-L1",
    "pd-l1": "PD-L1",
    "pd l1": "PD-L1",
    "ctla4": "CTLA-4",
    "ctla-4": "CTLA-4",
    "ctla 4": "CTLA-4",
}

TYPE_OF_THERAPY_VALUE_MAP = {
    "monotherapy": "Monotherapy",
    "mono": "Monotherapy",
    "combination": "Combination",
    "combination therapy": "Combination",
}

PRIMARY_ENDPOINT_VALUE_MAP = {
    "os (overall survival)": "OS",
    "os": "OS",
    "pfs": "PFS",
    "orr": "ORR",
    "rfs": "RFS",
    "dfs": "RFS",  # normalize to RFS
    "efs": "EFS",
    "pcr": "Path CR",
    "path cr": "Path CR",
    "safety": "Safety",
}

TRIAL_PHASE_VALUE_MAP = {
    "iii": "Phase 3",
    "phase iii": "Phase 3",
    "3": "Phase 3",
    "ii": "Phase 2",
    "phase ii": "Phase 2",
    "2": "Phase 2",
}

TYPE_OF_STUDY_VALUE_MAP = {
    "original": "Original publication",
    "original publication": "Original publication",
    "follow-up": "Follow-up",
    "follow up": "Follow-up",
}

INCLUDED_MA_VALUE_MAP = {
    "yes": "Yes",
    "no": "No",
}

COMBO_TREATMENT_VALUE_MAP = {
    "ici + meki (mek inhibitor)": "ICI + MEKi",
    "ici + meki": "ICI + MEKi",
    "ici + radiation": "ICI + Radiation",
    "ici + radiotherapy": "ICI + Radiation",
    "ici + vaccine": "ICI + Vaccine",
    "ici + tki": "ICI + TKI",
    "ici + chemo": "ICI + Chemo",
    "ici + anti-vegf": "ICI + Anti-VEGF",
    "ici + brafi + meki": "ICI + BRAFi + MEKi",
    "ici + chemo + anti-vegf": "ICI + Chemo + Anti-VEGF",
    "ici + ici": "ICI + ICI",
    "ici + ici + chemo": "ICI + ICI + Chemo",
}

CONTROL_ARM_VALUE_MAP = {
    "chemo": "Chemo",
    "tki": "TKI",
    "interferon": "Interferon",
    "multikinase inhibitor": "Multikinase inhibitor",
    "mtor inhibitor": "mTOR inhibitor",
    "radiation": "Radiation",
    "vaccine": "Vaccine",
    "placebo": "Placebo",
    "best supportive care": "Best Supportive Care",
    "bsc": "Best Supportive Care",
    "chemo / anti-egfr": "Chemo / Anti-EGFR",
    "chemo / anti-vegf": "Chemo / Anti-VEGF",
    "chemo / tki": "Chemo / TKI",
    "brafi + meki": "BRAFi + MEKi",
}

# Cancer Type normalization for frequent ones
CANCER_TYPE_VALUE_MAP = {
    "nsclc": "Non-Small Cell Lung Cancer (NSCLC)",
    "non-small cell lung cancer": "Non-Small Cell Lung Cancer (NSCLC)",
    "non small cell lung cancer": "Non-Small Cell Lung Cancer (NSCLC)",
    "sclc": "Small Cell Lung Cancer (SCLC)",
    "small cell lung cancer": "Small Cell Lung Cancer (SCLC)",
    "hnscc": "Head and Neck (HNSCC)",
    "head and neck": "Head and Neck",
    "head and neck (hnscc)": "Head and Neck (HNSCC)",
    "hcc": "Hepatocellular carcinoma (HCC)",
    "hepatocellular carcinoma": "Hepatocellular carcinoma (HCC)",
    "rcc": "Renal Cell Carcinoma (RCC)",
    "renal cell carcinoma": "Renal Cell Carcinoma (RCC)",
    "melanoma": "Melanoma",
    "breast": "Breast",
    "colorectal": "Colorectal",
    "gastric/gej": "Gastric/GEJ",
    "esophageal/gej": "Esophageal/GEJ",
    "urothelial": "Urothelial and Bladder",
    "bladder": "Bladder",
    "renal cell": "Renal Cell Carcinoma (RCC)",
    "non-small cell lung cancer (nsclc)": "Non-Small Cell Lung Cancer (NSCLC)",
    "small cell lung cancer (sclc)": "Small Cell Lung Cancer (SCLC)",
}

# Clinical Setting unifier (lines → setting strings)
def _canon_clinical_setting(raw_value: str) -> str:
    v = _canon_simple(raw_value).lower()
    # direct matches
    if v in {"neoadjuvant", "adjuvant", "perioperative", "maintenance", "metastatic / recurrent (unresectable)"}:
        return v.title() if v != "metastatic / recurrent (unresectable)" else "Metastatic / Recurrent (Unresectable)"
    if v in {"first-line", "first line", "1l"}:
        return "First-line metastatic"
    if v in {"second-line", "second line", "2l", "2l+" , "second-line+"}:
        return "Second-line+"
    if v in {"third-line", "third line", "3l"}:
        return "Second-line+"  # fold 3L+ into "Second-line+"
    # some prompts used "second-line+" as text already
    if "second-line" in v or "2l" in v:
        return "Second-line+"
    if "first-line" in v or "1l" in v:
        return "First-line metastatic"
    # keep unknown as-is (capitalized)
    return raw_value

def canonicalize_key(k: str) -> str:
    k = str(k or "").strip()
    return CANON_KEYS.get(k, k)

def canonicalize_value(cat: str, value: str) -> str:
    cat_c = canonicalize_key(cat)
    v = _canon_simple(value)
    low = v.lower()

    if cat_c == "ICI Class":
        return ICI_CLASS_VALUE_MAP.get(low, v)
    if cat_c == "Type of Therapy":
        return TYPE_OF_THERAPY_VALUE_MAP.get(low, v)
    if cat_c == "Primary Endpoint":
        return PRIMARY_ENDPOINT_VALUE_MAP.get(low, v)
    if cat_c == "Trial Phase":
        return TRIAL_PHASE_VALUE_MAP.get(low, v)
    if cat_c == "Type of Study":
        return TYPE_OF_STUDY_VALUE_MAP.get(low, v)
    if cat_c == "Included in MA":
        return INCLUDED_MA_VALUE_MAP.get(low, v)
    if cat_c == "Type of combination (Treatment Arm)":
        return COMBO_TREATMENT_VALUE_MAP.get(low, v)
    if cat_c == "Control Arm":
        return CONTROL_ARM_VALUE_MAP.get(low, v)
    if cat_c == "Cancer Type":
        return CANCER_TYPE_VALUE_MAP.get(low, v)
    if cat_c == "Clinical Setting":
        # accept direct settings or convert from "Lines of treatment" styles
        return _canon_clinical_setting(v)
    # default: return trimmed
    return v

def normalize_selected_filter(sel: dict) -> dict:
    """
    - Canonicalize keys to new interface terms.
    - Drop empty/None/"All".
    - Canonicalize values per-category.
    - If we see an original "Lines of treatment" filter, fold into Clinical Setting values.
    """
    out = {}
    for k, v in (sel or {}).items():
        if v is None:
            continue
        vs = str(v).strip()
        if not vs or vs.lower() == "all":
            continue

        k_can = canonicalize_key(k)
        # special fold: if the original key was "Lines of treatment", translate value to Clinical Setting
        if k in ("Lines of treatment",) and k_can == "Clinical Setting":
            vs = _canon_clinical_setting(vs)

        out[k_can] = canonicalize_value(k_can, vs)
    return out

# ============================
# COLUMN NORMALIZATION
# ============================

def _ordered_values_from_column_object(col_obj: dict) -> list[str]:
    if not isinstance(col_obj, dict):
        return []
    items = []
    for k, v in col_obj.items():
        m = re.search(r"Column\s*(\d+)", str(k))
        idx = int(m.group(1)) if m else 10**9
        items.append((idx, v))
    items.sort(key=lambda x: x[0])
    return [val for _, val in items if isinstance(val, str) and val.strip()]

def normalize_selected_column(selected_column) -> dict:
    """
    Ensure REQUIRED_COLS first, dedupe, cap to REQUIRED + MAX_ADDITIONAL_COLS, and
    rebuild enumerated mapping as {"Column 1": "...", ...}
    """
    # Input can be dict or list
    if isinstance(selected_column, dict):
        cols = _ordered_values_from_column_object(selected_column)
    elif isinstance(selected_column, list):
        cols = [c for c in selected_column if isinstance(c, str) and c.strip()]
    else:
        cols = []

    out_list, seen = [], set()
    for rc in REQUIRED_COLS:
        if rc not in seen:
            out_list.append(rc); seen.add(rc)

    for c in cols:
        if c not in seen:
            out_list.append(c); seen.add(c)

    cap = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    out_list = out_list[:cap]
    return {f"Column {i+1}": name for i, name in enumerate(out_list)}

# ============================
# PROMPTS (3 stages)
# ============================

def build_stage1_prompt(question: str) -> str:
    """
    Stage 1: choose FILTERS (categories + values) ONLY.
    Return ONLY:
      {"selected_filter": {"<Canonical Filter Name>": "<Value>", ...}}
    - Use exact category/value strings from the provided definitions when possible.
    - Omit any filters whose correct value would be "All".
    - Prefer using these canonical category names (MUST map to them if synonyms appear):
        ICI Class, ICI Name, Cancer Type, Type of Therapy, Type of combination (Treatment Arm),
        Control Arm, Clinical Setting, Trial Phase, Type of Study, Primary Endpoint, Included in MA
    """
    schema = (
        '{\n'
        '  "selected_filter": {\n'
        '    "Canonical Filter Name 1": "Value",\n'
        '    "Canonical Filter Name 2": "Value"\n'
        '  }\n'
        '}'
    )
    example = '{"selected_filter":{"Cancer Type":"Non-Small Cell Lung Cancer (NSCLC)","Trial Phase":"Phase 3"}}'
    return (
        "You are a medical expert researching cancer.\n"
        "Select ONLY the filters (categories + values) relevant to the question.\n"
        "Return ONLY a valid JSON object with key `selected_filter` as shown below (no extra prose/code):\n"
        f"{schema}\n\n"
        "Rules:\n"
        "- Use double quotes everywhere.\n"
        "- Use exact category names from the canonical list below (if synonyms appear, map to these names):\n"
        '  ["ICI Class","ICI Name","Cancer Type","Type of Therapy","Type of combination (Treatment Arm)",'
        '"Control Arm","Clinical Setting","Trial Phase","Type of Study","Primary Endpoint","Included in MA"]\n'
        "- Use exact strings for values from the filter definitions when applicable.\n"
        "- Omit any filters whose correct value would be \"All\".\n"
        "- Include only filters directly justified by the question.\n\n"
        "Available filter CATEGORY NAMES (reference):\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available filter CATEGORIES + VALUES (use exact strings whenever possible):\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now.\n"
        f"Example (not prescriptive): {example}"
    )

def build_stage2_prompt(question: str, selected_filter: dict) -> str:
    """
    Stage 2: choose COLUMNS ONLY, with REQUIRED_COLS included and at most MAX_ADDITIONAL_COLS extras.
    Return ONLY:
      {"selected_column": {"Column 1": "...", "Column 2": "...", ...}}
    """
    total_slots = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    schema_columns = ',\n'.join([f'    "Column {i}": "Column Name"' for i in range(1, total_slots + 1)])
    schema = '{\n  "selected_column": {\n' + schema_columns + '\n  }\n}'

    return (
        "You are a medical expert researching cancer.\n"
        "Now select the OUTPUT COLUMNS for the table, knowing these are ALWAYS included: "
        f"{', '.join(REQUIRED_COLS)}. You may add up to {MAX_ADDITIONAL_COLS} more.\n"
        "Return ONLY a valid JSON object with key `selected_column` (no extra prose/code):\n"
        f"{schema}\n\n"
        "Rules:\n"
        "- Use double quotes everywhere.\n"
        f"- Ensure {', '.join(REQUIRED_COLS)} are included.\n"
        f"- Choose at most {MAX_ADDITIONAL_COLS} additional columns, justified by the question.\n"
        "- Use exact column names from the list provided below.\n\n"
        "Context — the currently chosen filters (use them to guide column selection):\n"
        f"{json.dumps(selected_filter, ensure_ascii=False)}\n\n"
        "Available columns and definitions (select by exact name):\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now."
    )

def build_stage3_prompt(question: str, selected_filter: dict, selected_column_obj: dict) -> str:
    """
    Stage 3: joint sanity check of filters + columns. Either ACCEPT or REVISE.
    Return ONLY one of:
      {"status":"accept","selected_filter":{...},"selected_column":{...}}
    or
      {"status":"revise","selected_filter":{...},"selected_column":{...},"notes":"short reason"}
    """
    return (
        "You are a medical expert researching cancer.\n"
        "Final check: evaluate the COMBINATION of filters + columns for relevance, correctness, and parsimony.\n"
        "If everything is consistent and minimal, return status \"accept\". Otherwise, return \"revise\" with improved choices.\n"
        "Return ONLY one valid JSON object (no extra prose/code) with the schema below.\n\n"
        "Schemas:\n"
        '{ "status":"accept","selected_filter":{...},"selected_column":{...} }\n'
        '{ "status":"revise","selected_filter":{...},"selected_column":{...},"notes":"<25 chars>" }\n\n'
        "Rules:\n"
        "- Filters must use canonical category names (map synonyms to: "
        '["ICI Class","ICI Name","Cancer Type","Type of Therapy","Type of combination (Treatment Arm)",'
        '"Control Arm","Clinical Setting","Trial Phase","Type of Study","Primary Endpoint","Included in MA"]).\n'
        f"- Columns must include {', '.join(REQUIRED_COLS)} and only up to {MAX_ADDITIONAL_COLS} extras.\n"
        "- Omit any filter whose correct value would be \"All\".\n"
        "- Prefer minimal set that answers the question; drop irrelevant columns.\n\n"
        "Current selection to evaluate:\n"
        f"Question: {question}\n"
        f"Filters: {json.dumps(selected_filter, ensure_ascii=False)}\n"
        f"Columns: {json.dumps(selected_column_obj, ensure_ascii=False)}\n"
        "Return JSON now."
    )

# ============================
# CORE RUNNERS (per question)
# ============================

def run_pipeline4_for_question(question: str, seed: int | None = None) -> tuple[dict, dict]:
    # ---- Stage 1: Filters
    p1 = build_stage1_prompt(question)
    t0 = time.time()
    raw1 = call_model_with_retries(p1, seed=seed)
    t1_elapsed = time.time() - t0
    try:
        o1 = extract_json(raw1)
    except Exception:
        o1 = {}

    selected_filter_raw = o1.get("selected_filter", {})
    selected_filter = normalize_selected_filter(selected_filter_raw)

    # ---- Stage 2: Columns
    p2 = build_stage2_prompt(question, selected_filter)
    t1b = time.time()
    raw2 = call_model_with_retries(p2, seed=seed)
    t2_elapsed = time.time() - t1b
    try:
        o2 = extract_json(raw2)
    except Exception:
        o2 = {}

    selected_column_obj = normalize_selected_column(o2.get("selected_column", {}))

    # ---- Stage 3: Joint re-evaluation
    p3 = build_stage3_prompt(question, selected_filter, selected_column_obj)
    t2b = time.time()
    raw3 = call_model_with_retries(p3, seed=seed)
    t3_elapsed = time.time() - t2b
    try:
        o3 = extract_json(raw3)
    except Exception:
        o3 = {}

    status = str(o3.get("status", "")).strip().lower()
    if status == "revise":
        # Accept revised objects if present; re-normalize
        sel_f = o3.get("selected_filter", selected_filter)
        sel_c = o3.get("selected_column", selected_column_obj)
        selected_filter = normalize_selected_filter(sel_f)
        selected_column_obj = normalize_selected_column(sel_c)
    elif status == "accept":
        # Keep as-is (but still pass through normalizers once more)
        selected_filter = normalize_selected_filter(o3.get("selected_filter", selected_filter))
        selected_column_obj = normalize_selected_column(o3.get("selected_column", selected_column_obj))
    else:
        # No valid decision returned -> keep previous normalized selections
        pass

    # Final safety: enforce constraints again
    selected_filter = normalize_selected_filter(selected_filter)
    selected_column_obj = normalize_selected_column(selected_column_obj)

    merged = {
        "selected_filter": selected_filter,
        "selected_column": selected_column_obj,
    }
    timings = {
        "stage1_time_sec": t1_elapsed,
        "stage2_time_sec": t2_elapsed,
        "stage3_time_sec": t3_elapsed,
        "total_time_sec": t1_elapsed + t2_elapsed + t3_elapsed,
    }
    return merged, timings

def run_n_times_for_question(question: str, n: int) -> tuple[list[str], list[dict]]:
    runs, timing_list = [], []
    for i in range(n):
        seed = random.randint(1, 10_000_000)
        merged, timing = run_pipeline4_for_question(question, seed=seed)
        runs.append(json.dumps(merged, ensure_ascii=False))
        timing_list.append(timing)
        if DELAY_BETWEEN_CALLS_SEC > 0:
            time.sleep(DELAY_BETWEEN_CALLS_SEC)
    return runs, timing_list

# ============================
# MAIN
# ============================

def main():
    df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)

    global QUESTION_COLUMN
    if QUESTION_COLUMN not in df.columns:
        candidates = [c for c in df.columns if str(c).strip().lower() in {"question","query","prompt"}]
        if candidates:
            QUESTION_COLUMN = candidates[0]
        else:
            raise ValueError(f"Couldn't find a question column. Available: {list(df.columns)}")

    run_cols = [f"run_{i+1}" for i in range(RUNS_PER_QUESTION)]
    for c in run_cols:
        if c not in df.columns:
            df[c] = ""

    timings_records = []

    for idx, row in df.head(MAX_QUESTIONS).iterrows():
        q = str(row[QUESTION_COLUMN]).strip()
        if not q or q.lower() == "nan":
            continue
        print(f"[Pipeline 4] Row {idx}: {q[:100]}{'...' if len(q) > 100 else ''}")
        merged_runs, timings_list = run_n_times_for_question(q, RUNS_PER_QUESTION)
        for i, merged_json in enumerate(merged_runs):
            df.at[idx, run_cols[i]] = merged_json
        timings_records.append({
            "row_index": idx,
            "question": q,
            **{f"run_{i+1}_stage1_sec": t["stage1_time_sec"] for i, t in enumerate(timings_list)},
            **{f"run_{i+1}_stage2_sec": t["stage2_time_sec"] for i, t in enumerate(timings_list)},
            **{f"run_{i+1}_stage3_sec": t["stage3_time_sec"] for i, t in enumerate(timings_list)},
            **{f"run_{i+1}_total_sec":  t["total_time_sec"]  for i, t in enumerate(timings_list)},
        })

    timings_df = pd.DataFrame(timings_records)
    with pd.ExcelWriter(EXCEL_PATH_OUT, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="results")
        timings_df.to_excel(writer, index=False, sheet_name="timings")

    print(f"Saved results to: {EXCEL_PATH_OUT}")

if __name__ == "__main__":
    main()


In [ ]:
# Pipeline 4 — Schema-Constrained Self-Consistency + Verifier Reranking
# --------------------------------------------------------------------
# High-level:
#  - Generate K candidates (mix of single-stage and two-stage prompts).
#  - Deterministically canonicalize & validate (closed set for filters/columns).
#  - Score/rerank by intent coverage, parsimony, and schema fidelity.
#  - Short "judge" pass on the top M candidates to correct/trim within the same schema.
#  - Accept/Revise self-check on the winner; re-validate; write results + timings + audit.
#
# Notes:
#  - Uses your same definition files and environment variable layout.
#  - Enforces canonical filter NAMES and common VALUE synonyms.
#  - Enforces REQUIRED_COLS and a cap on additional columns.
#  - Saves results + timings to an Excel file (two sheets).
#
# Env:
#   GEMINI_KEY must be set (e.g., via .env)
#
# Files (update paths as needed):
#   FILTER_NAMES_DEF_PATH   (filter CATEGORY names only; used in prompts)
#   COLUMN_DEFS_PATH        (column definitions; used in prompts)
#   FILTER_DEFS_FULL_PATH   (full filter categories + allowed values; used in prompts)
#
# Input:
#   EXCEL_PATH_IN  (expects a column named one of {"question","query","prompt"})
#
# Output:
#   EXCEL_PATH_OUT with sheets:
#     - results: final_json, audit_json (per question)
#     - timings: per-stage timings (per question)

import os, time, random, json, ast, re, math
import pandas as pd
from dotenv import load_dotenv

import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, DeadlineExceeded, InternalServerError

# ============================
# CONFIG
# ============================
EXCEL_PATH_IN   = "runs/query-chosen-filters-columns.xlsx"
EXCEL_PATH_OUT  = "runs/query-2.5-chosen-filters-columns_with_runs_pipeline4.xlsx"
SHEET_NAME      = 0
QUESTION_COLUMN = "question"  # auto-detected if missing
MAX_QUESTIONS   = 100

# Candidate generation
K_CANDIDATES             = 6   # total candidates per question (e.g., 3 single-stage + 3 two-stage)
N_SINGLE_STAGE_CANDIDATES = 3  # Style A
N_TWO_STAGE_CANDIDATES    = 3  # Style B
TOP_M_FOR_JUDGE           = 2  # rerank -> judge the top M
MODEL_NAME                = "gemini-2.5-flash"

# Required columns & cap (do NOT count these toward the additional cap)
REQUIRED_COLS        = ["NCT", "PMID", "Authors", "Year"]
MAX_ADDITIONAL_COLS  = 6

# Definition files (reuse your existing)
#  - If your files live elsewhere, update these paths.
FILTER_NAMES_DEF_PATH = "definitions_folder/definitions - aim2 - filter names.txt"
COLUMN_DEFS_PATH      = "definitions_folder/definitions - aim2 - column.txt"
FILTER_DEFS_FULL_PATH = "definitions_folder/definitions - aim2 - filter.txt"

# ============================
# SETUP
# ============================
load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
if not GEMINI_KEY:
    raise RuntimeError("GEMINI_KEY environment variable is not set")
genai.configure(api_key=GEMINI_KEY)

GENERATION_CONFIG = {
    "response_mime_type": "application/json",
    "temperature": 0.0,
}

with open(FILTER_NAMES_DEF_PATH, "r", encoding="utf-8") as f:
    FILTER_NAMES_TEXT = f.read()
with open(COLUMN_DEFS_PATH, "r", encoding="utf-8") as f:
    COLUMN_DEFS_TEXT = f.read()
with open(FILTER_DEFS_FULL_PATH, "r", encoding="utf-8") as f:
    FILTER_DEFS_TEXT = f.read()

# ============================
# CANONICAL SCHEMA (closed sets)
# ============================

# Canonical filter category names (interface)
CANON_FILTER_KEYS = [
    "ICI Class",
    "ICI Name",
    "Cancer Type",
    "Type of Therapy",
    "Type of combination (Treatment Arm)",
    "Control Arm",
    "Clinical Setting",
    "Trial Phase",
    "Type of Study",
    "Primary Endpoint",
    "Included in MA",
]

# Canonical column names (add/trim as needed to match your UI exactly)
CANON_COLUMN_NAMES = [
    "NCT","PMID","Authors","Year","Original/Follow Up","Study name","Trial phase",
    "Number of arms","Cancer type","Treatment regimen","Name of ICI","Class of ICI",
    "Monotherapy/combination","Type of combination","Control regimen","Type of control",
    "Total sample size","Lines of treatment","Clinical setting in relation to surgery",
    "Is PD-L1 positivity inclusion criteria","Is any other biomarker used for inclusion",
    "Primary endpoint","Secondary endpoint","Type of follow-up given",
    "Follow-up duration for primary endpoint(s) in months","Included in MA",
    "Primary multiple, composite, or co-primary endpoints?"  # keep as a single column label if you use it
]

# Synonyms for filter keys (→ canonical)
CANON_KEYS_MAP = {
    # filters
    "Class of ICI": "ICI Class",
    "Name of ICI": "ICI Name",
    "Cancer type": "Cancer Type",
    "Monotherapy/combination": "Type of Therapy",
    "Monotherapy/Combination": "Type of Therapy",
    "Mono/Combo": "Type of Therapy",
    "Trial phase": "Trial Phase",
    "Clinical setting in relation to surgery": "Clinical Setting",
    "Clinical setting": "Clinical Setting",
    "Perioperative setting": "Clinical Setting",
    "Setting in relation to surgery": "Clinical Setting",
    "Type of combination": "Type of combination (Treatment Arm)",
    "Type of control": "Control Arm",
    "Control regimen": "Control Arm",
    "Primary endpoint": "Primary Endpoint",
    "Primary Endpoint(s)": "Primary Endpoint",
    "Primary endpoints": "Primary Endpoint",
    "Included in Meta-analysis": "Included in MA",
}

# Canonicalized value sets (where enumerated)
ALLOWED_VALUES = {
    "ICI Class": {"PD-1", "PD-L1", "CTLA-4"},
    "Type of Therapy": {"Monotherapy", "Combination"},
    "Primary Endpoint": {"OS","PFS","ORR","RFS","EFS","Path CR","Safety"},
    "Trial Phase": {"Phase 2","Phase 3"},
    "Type of Study": {"Original publication","Follow-up"},
    "Included in MA": {"Yes","No"},
    # Some common arms (not exhaustive; expand as needed)
    "Type of combination (Treatment Arm)": {
        "ICI + Chemo","ICI + TKI","ICI + ICI","ICI + ICI + Chemo","ICI + Radiation",
        "ICI + Vaccine","ICI + MEKi","ICI + Anti-VEGF","ICI + BRAFi + MEKi","ICI + Chemo + Anti-VEGF"
    },
    "Control Arm": {
        "Placebo","Chemo","TKI","Interferon","Best Supportive Care","Radiation","Vaccine",
        "Multikinase inhibitor","mTOR inhibitor","Chemo / Anti-VEGF","Chemo / Anti-EGFR"
    },
    # Cancer Type is large; we keep this open and rely on synonym canonicalization below.
}

# Value synonym maps
ICI_CLASS_VALUE_MAP = {
    "pd1": "PD-1","pd-1": "PD-1","pd 1":"PD-1",
    "pdl1": "PD-L1","pd-l1":"PD-L1","pd l1":"PD-L1","pd_l1":"PD-L1",
    "ctla4":"CTLA-4","ctla-4":"CTLA-4","ctla 4":"CTLA-4",
}
TYPE_OF_THERAPY_VALUE_MAP = {
    "mono":"Monotherapy","monotherapy":"Monotherapy",
    "combination therapy":"Combination","combination":"Combination",
}
PRIMARY_ENDPOINT_VALUE_MAP = {
    "os (overall survival)":"OS","os":"OS","pfs":"PFS","orr":"ORR",
    "rfs":"RFS","dfs":"RFS","efs":"EFS","pcr":"Path CR","path cr":"Path CR",
    "safety":"Safety",
}
TRIAL_PHASE_VALUE_MAP = {
    "iii":"Phase 3","phase iii":"Phase 3","3":"Phase 3",
    "ii":"Phase 2","phase ii":"Phase 2","2":"Phase 2",
}
TYPE_OF_STUDY_VALUE_MAP = {
    "original":"Original publication","original publication":"Original publication",
    "follow up":"Follow-up","follow-up":"Follow-up",
}
INCLUDED_MA_VALUE_MAP = {"yes":"Yes","no":"No"}

COMBO_TREATMENT_VALUE_MAP = {
    "ici + meki (mek inhibitor)":"ICI + MEKi","ici + meki":"ICI + MEKi",
    "ici + radiation":"ICI + Radiation","ici + radiotherapy":"ICI + Radiation",
    "ici + vaccine":"ICI + Vaccine","ici + tki":"ICI + TKI","ici + chemo":"ICI + Chemo",
    "ici + anti-vegf":"ICI + Anti-VEGF","ici + brafi + meki":"ICI + BRAFi + MEKi",
    "ici + chemo + anti-vegf":"ICI + Chemo + Anti-VEGF","ici + ici":"ICI + ICI",
    "ici + ici + chemo":"ICI + ICI + Chemo"
}
CONTROL_ARM_VALUE_MAP = {
    "chemo":"Chemo","tki":"TKI","interferon":"Interferon",
    "multikinase inhibitor":"Multikinase inhibitor","mtor inhibitor":"mTOR inhibitor",
    "radiation":"Radiation","vaccine":"Vaccine","placebo":"Placebo","bsc":"Best Supportive Care",
    "best supportive care":"Best Supportive Care","chemo / anti-egfr":"Chemo / Anti-EGFR",
    "chemo / anti-vegf":"Chemo / Anti-VEGF",
}
# Cancer Type synonyms (examples)
CANCER_TYPE_VALUE_MAP = {
    "nsclc":"Non-Small Cell Lung Cancer (NSCLC)",
    "non small cell lung cancer":"Non-Small Cell Lung Cancer (NSCLC)",
    "non-small cell lung cancer":"Non-Small Cell Lung Cancer (NSCLC)",
    "sclc":"Small Cell Lung Cancer (SCLC)",
    "small cell lung cancer":"Small Cell Lung Cancer (SCLC)",
    "hnscc":"Head and Neck (HNSCC)","head and neck (hnscc)":"Head and Neck (HNSCC)",
    "head and neck":"Head and Neck",
    "hcc":"Hepatocellular carcinoma (HCC)","hepatocellular carcinoma":"Hepatocellular carcinoma (HCC)",
    "rcc":"Renal Cell Carcinoma (RCC)","renal cell carcinoma":"Renal Cell Carcinoma (RCC)",
    "melanoma":"Melanoma","breast":"Breast","colorectal":"Colorectal",
    "gastric/gej":"Gastric/GEJ","esophageal/gej":"Esophageal/GEJ","bladder":"Bladder","urothelial":"Bladder"
}

# Column synonym normalizer (value -> canonical column)
COLUMN_SYNONYMS = {
    "original/follow up":"Original/Follow Up","original/follow-up":"Original/Follow Up",
    "follow up duration for primary endpoints (overall, rx, control)":"Follow-up duration for primary endpoint(s) in months",
    "trial phase":"Trial phase","cancer type":"Cancer type","name of ici":"Name of ICI","class of ici":"Class of ICI",
    "monotherapy/combination":"Monotherapy/combination","type of combination":"Type of combination",
    "control regimen":"Control regimen","type of control":"Type of control","total sample size":"Total sample size",
    "lines of treatment":"Lines of treatment","clinical setting in relation to surgery":"Clinical setting in relation to surgery",
    "is pd-l1 positivity inclusion criteria":"Is PD-L1 positivity inclusion criteria",
    "is any other biomarker used for inclusion":"Is any other biomarker used for inclusion",
    "primary endpoint":"Primary endpoint","secondary endpoint":"Secondary endpoint",
    "type of follow-up given":"Type of follow-up given",
    "follow-up duration for primary endpoint(s) in months":"Follow-up duration for primary endpoint(s) in months",
    "included in ma":"Included in MA",
    "primary multiple, composite, or co-primary endpoints?":"Primary multiple, composite, or co-primary endpoints?",
}

# ============================
# UTILS
# ============================

def strip_code_fences(s: str) -> str:
    s = (s or "").strip()
    if s.startswith("```") and s.endswith("```"):
        # remove first line fence and last fence
        lines = s.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        s = "\n".join(lines)
    return s.strip()

def extract_json(text: str):
    raw = strip_code_fences(text)
    # Try direct JSON
    try:
        return json.loads(raw)
    except Exception:
        pass
    # Fallback: first {...} block
    try:
        start = raw.find("{"); end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = raw[start:end+1]
            try:
                return json.loads(candidate)
            except Exception:
                obj = ast.literal_eval(candidate)
                if isinstance(obj, (dict, list)):
                    return obj
    except Exception:
        pass
    raise ValueError(f"Could not parse JSON from model text: {text[:200]}...")

def call_model_with_retries(prompt: str, timeout=90, retries=3):
    model = genai.GenerativeModel(MODEL_NAME)
    backoff = 1.0
    for attempt in range(1, retries+1):
        try:
            resp = model.generate_content(
                prompt,
                generation_config=GENERATION_CONFIG,
                request_options={"timeout": timeout},
                safety_settings=None,
            )
            txt = getattr(resp, "text", None)
            return (txt if txt is not None else str(resp)).strip()
        except (ResourceExhausted, DeadlineExceeded, InternalServerError) as e:
            if attempt == retries:
                return f"[ERROR after {attempt} attempts] {e}"
            time.sleep(backoff * (2 ** (attempt - 1)) * (1 + random.uniform(0, 0.35)))
        except Exception as e:
            if attempt >= min(3, retries):
                return f"[ERROR non-retryable? attempt {attempt}] {e}"
            time.sleep(backoff * attempt)

def _canon_simple(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())

def _norm_key(key: str) -> str:
    k = _canon_simple(key)
    return CANON_KEYS_MAP.get(k, k)

def _canon_filter_value(cat: str, value: str) -> str:
    cat_c = _norm_key(cat)
    v = _canon_simple(value)
    low = v.lower()

    if cat_c == "ICI Class": return ICI_CLASS_VALUE_MAP.get(low, v)
    if cat_c == "Type of Therapy": return TYPE_OF_THERAPY_VALUE_MAP.get(low, v)
    if cat_c == "Primary Endpoint": return PRIMARY_ENDPOINT_VALUE_MAP.get(low, v)
    if cat_c == "Trial Phase": return TRIAL_PHASE_VALUE_MAP.get(low, v)
    if cat_c == "Type of Study": return TYPE_OF_STUDY_VALUE_MAP.get(low, v)
    if cat_c == "Included in MA": return INCLUDED_MA_VALUE_MAP.get(low, v)
    if cat_c == "Type of combination (Treatment Arm)": return COMBO_TREATMENT_VALUE_MAP.get(low, v)
    if cat_c == "Control Arm": return CONTROL_ARM_VALUE_MAP.get(low, v)
    if cat_c == "Cancer Type": return CANCER_TYPE_VALUE_MAP.get(low, v)
    if cat_c == "Clinical Setting":
        vv = low
        if vv in {"neoadjuvant","adjuvant","perioperative","maintenance"}:
            return v.title()
        # fold line-of-therapy to settings
        if vv in {"1l","first line","first-line"}: return "First-line metastatic"
        if any(x in vv for x in ["2l","3l","second line","third line","second-line","third-line","2l+","second-line+"]):
            return "Second-line+"
        if vv in {"metastatic / recurrent (unresectable)"}:
            return "Metastatic / Recurrent (Unresectable)"
        return v
    return v

def _ordered_values_from_column_object(col_obj: dict) -> list[str]:
    if not isinstance(col_obj, dict):
        return []
    items = []
    for k, v in col_obj.items():
        m = re.search(r"Column\s*(\d+)", str(k))
        idx = int(m.group(1)) if m else 10**9
        items.append((idx, v))
    items.sort(key=lambda x: x[0])
    return [val for _, val in items if isinstance(val, str) and val and str(val).strip()]

def _normalize_column_name(name: str) -> str:
    raw = _canon_simple(name)
    key = raw.lower()
    if key in COLUMN_SYNONYMS:
        return COLUMN_SYNONYMS[key]
    return raw

def normalize_selected_column(selected_column) -> dict:
    # Accept dict or list and rebuild enumerated object with REQUIRED first and a cap
    if isinstance(selected_column, dict):
        cols = _ordered_values_from_column_object(selected_column)
    elif isinstance(selected_column, list):
        cols = [c for c in selected_column if isinstance(c, str) and c.strip()]
    else:
        cols = []

    # normalize names, drop placeholders
    normed = []
    for c in cols:
        cc = _normalize_column_name(c)
        if cc.lower() == "all the remaining columns":  # reject placeholder
            continue
        normed.append(cc)

    out_list, seen = [], set()
    for rc in REQUIRED_COLS:
        if rc not in seen:
            out_list.append(rc); seen.add(rc)
    for c in normed:
        c2 = _normalize_column_name(c)
        if c2 in CANON_COLUMN_NAMES and c2 not in seen:
            out_list.append(c2); seen.add(c2)

    cap = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    out_list = out_list[:cap]
    return {f"Column {i+1}": name for i, name in enumerate(out_list)}

def normalize_selected_filter(sel: dict) -> dict:
    out = {}
    for k, v in (sel or {}).items():
        if v is None:
            continue
        vs = _canon_simple(v)
        if not vs or vs.lower() == "all":
            continue
        k_can = _norm_key(k)
        out[k_can] = _canon_filter_value(k_can, vs)
    return out

def validate_and_fix(candidate: dict):
    """Return (fixed_candidate, audit_dict)"""
    audit = {
        "dropped_non_canonical_filter_keys": [],
        "dropped_filter_values": [],
        "composite_ici_class_rejected": False,
        "moved_filters_that_are_columns": [],
        "column_synonyms_applied": [],
        "trimmed_columns_count": 0,
        "notes": [],
    }

    sf = dict(candidate.get("selected_filter", {}) or {})
    sc = candidate.get("selected_column", {}) or {}

    # Canonicalize
    sf = normalize_selected_filter(sf)
    sc = normalize_selected_column(sc)

    # Reject non-canonical filter keys
    sf_fixed = {}
    for k, v in sf.items():
        if k not in CANON_FILTER_KEYS:
            audit["dropped_non_canonical_filter_keys"].append(k)
            continue
        # reject composite ICI Class
        if k == "ICI Class" and isinstance(v, str) and re.search(r"\bor\b|,|both", v, flags=re.I):
            audit["composite_ici_class_rejected"] = True
            continue
        # where enumerated, ensure value is in allowed set
        if k in ALLOWED_VALUES:
            if v not in ALLOWED_VALUES[k]:
                audit["dropped_filter_values"].append({k: v})
                continue
        sf_fixed[k] = v

    # Hard rule: things that were mistakenly put as filters but are columns -> drop from filters
    column_like_filters = {
        "Year","Total sample size","Lines of treatment",
        "Is PD-L1 positivity inclusion criteria","Is any other biomarker used for inclusion",
        "Primary multiple, composite, or co-primary endpoints?"
    }
    for bad in list(sf_fixed.keys()):
        if bad in column_like_filters:
            audit["moved_filters_that_are_columns"].append({bad: sf_fixed[bad]})
            del sf_fixed[bad]

    # Ensure columns are all canonical & capped (already done in normalize)
    # Count trimming (if any)
    final_cols = list(sc.values())
    if len(final_cols) > len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS:
        audit["trimmed_columns_count"] = len(final_cols) - (len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS)

    fixed = {"selected_filter": sf_fixed, "selected_column": normalize_selected_column(sc)}
    return fixed, audit

# ============================
# PROMPTS
# ============================

def build_single_stage_prompt(question: str) -> str:
    total_slots = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    schema_columns = ',\n'.join([f'    "Column {i}": "Name"' for i in range(1, total_slots + 1)])
    schema = (
        '{\n'
        '  "selected_filter": {\n'
        '    "Filter Category 1": "Chosen Value",\n'
        '    "Filter Category 2": "Chosen Value"\n'
        '  },\n'
        '  "selected_column": {\n'
        f'{schema_columns}\n'
        '  }\n'
        '}'
    )
    return (
        "You are a medical expert researching cancer trials.\n"
        "Return ONLY a valid JSON object (no prose, no code fences) with this schema:\n"
        f"{schema}\n"
        "Strict rules:\n"
        f"- Columns: ALWAYS include these first (do NOT count toward the limit): {', '.join(REQUIRED_COLS)}.\n"
        f"- You may add at most {MAX_ADDITIONAL_COLS} additional columns beyond those required.\n"
        "- The enumerated object keys MUST be exactly 'Column 1', 'Column 2', ... in display order.\n"
        "- Use exact column names from the list below.\n"
        "- Filters: choose only categories/values justified by the question. If a category would be 'All', OMIT it.\n"
        "- Use exact category and value strings from the lists below.\n"
        "- Use double quotes everywhere. No trailing commas. No explanations.\n\n"
        "Available filter CATEGORY NAMES (reference):\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available FULL filter categories and values (use exact strings; omit categories that would be 'All'):\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        "Available columns and definitions (select by exact name):\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now."
    )

def build_two_stage_prompt_stage1(question: str) -> str:
    total_slots = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    schema_columns = ',\n'.join([f'    "Column {i}": "Name"' for i in range(1, total_slots + 1)])
    schema = (
        '{\n'
        '  "selected_filter": {\n'
        '    "Filter Category 1": "Chosen Value",\n'
        '    "Filter Category 2": "Chosen Value"\n'
        '  }\n'
        '}'
    )
    # We directly choose filter values in stage 1 (names+values), then choose columns in stage 2
    return (
        "You are a medical expert researching cancer.\n"
        "Stage 1: choose FILTERS (categories + values) ONLY.\n"
        "Return ONLY a valid JSON with key `selected_filter` (no prose):\n"
        f"{schema}\n\n"
        "Rules:\n"
        "- Use exact category names from this canonical list: "
        '["ICI Class","ICI Name","Cancer Type","Type of Therapy","Type of combination (Treatment Arm)",'
        '"Control Arm","Clinical Setting","Trial Phase","Type of Study","Primary Endpoint","Included in MA"]\n'
        "- Use exact strings for values from the filter definitions.\n"
        "- Omit any filters whose correct value would be \"All\".\n"
        "- Include only filters directly justified by the question.\n\n"
        "Available filter CATEGORY NAMES:\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available filter CATEGORIES + VALUES (use exact strings whenever possible):\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now."
    )

def build_two_stage_prompt_stage2(question: str, selected_filter: dict) -> str:
    total_slots = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    schema_columns = ',\n'.join([f'    "Column {i}": "Name"' for i in range(1, total_slots + 1)])
    schema = '{\n  "selected_column": {\n' + schema_columns + '\n  }\n}'
    return (
        "You are a medical expert researching cancer.\n"
        "Stage 2: choose COLUMNS ONLY (REQUIRED first; at most additional columns as allowed).\n"
        "Return ONLY a valid JSON with key `selected_column` (no prose):\n"
        f"{schema}\n\n"
        "Rules:\n"
        f"- ALWAYS include: {', '.join(REQUIRED_COLS)} (do not count toward the cap).\n"
        f"- Choose at most {MAX_ADDITIONAL_COLS} additional columns.\n"
        "- Use exact column names from the list below.\n\n"
        "Context — chosen filters:\n"
        f"{json.dumps(selected_filter, ensure_ascii=False)}\n\n"
        "Available columns and definitions (select by exact name):\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now."
    )

def build_judge_prompt(question: str, candidate_json: dict) -> str:
    schema = (
        '{ "selected_filter": { ... }, "selected_column": { "Column 1": "...", "Column 2": "...", ... } }'
    )
    return (
        "You are a meticulous validator.\n"
        "Task: Examine the candidate JSON (filters+columns) against the schema and the question. "
        "If it is minimal and correct, return it unchanged. Otherwise, return a corrected JSON that:\n"
        "- Uses only canonical filter category names and allowed values (or valid cancer types),\n"
        "- Omits categories whose correct value would be 'All',\n"
        "- Includes REQUIRED columns first (NCT, PMID, Authors, Year) and at most 6 additional columns,\n"
        "- Uses exact column names from the list.\n\n"
        "Return ONLY a valid JSON object (no prose) with this schema:\n"
        f"{schema}\n\n"
        "Available canonical filter CATEGORY NAMES:\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available filter CATEGORIES + VALUES:\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        "Available columns and definitions:\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        f"Candidate JSON:\n{json.dumps(candidate_json, ensure_ascii=False)}\n"
        "Return JSON now."
    )

def build_accept_revise_prompt(question: str, merged: dict) -> str:
    return (
        "Final check: evaluate the COMBINATION of filters + columns for relevance, correctness, and parsimony.\n"
        'Return ONLY one JSON: {"status":"accept","selected_filter":{...},"selected_column":{...}} '
        'or {"status":"revise","selected_filter":{...},"selected_column":{...},"notes":"<25 chars>"}\n\n'
        f"Question: {question}\n"
        f"Current selection: {json.dumps(merged, ensure_ascii=False)}\n"
        "Rules:\n"
        "- Filters must use canonical category names.\n"
        f"- Columns must include {', '.join(REQUIRED_COLS)} and only up to {MAX_ADDITIONAL_COLS} extras.\n"
        "- Omit any filter whose correct value would be 'All'.\n"
        "- Prefer minimal set that answers the question; drop irrelevant columns.\n"
        "Return JSON now."
    )

# ============================
# SCORING / RERANK
# ============================

def score_candidate(question: str, cand: dict) -> float:
    """Heuristic scoring: intent coverage (+), parsimony (-), schema fidelity (+), consistency (+)."""
    q = (question or "").lower()

    sf = cand.get("selected_filter", {}) or {}
    sc = cand.get("selected_column", {}) or {}

    # Intent cues (simple keyword checks)
    cues = 0
    if re.search(r"\bphase\s*3\b|phase iii|\biii\b", q): cues += 1
    if re.search(r"\bphase\s*2\b|phase ii|\bii\b", q): cues += 1
    if any(x in q for x in ["adjuvant","neoadjuvant","perioperative","maintenance","first-line","second-line","1l","2l","3l"]): cues += 1
    if any(x in q for x in ["pd-1","pd1","pdl1","pd-l1","ctla-4","ctla4"]): cues += 1
    if any(x in q for x in ["nsclc","non-small cell","sclc","melanoma","rcc","hnscc","esophageal","gastric","bladder","urothelial","breast","colorectal","hcc","pancreatic","prostate","mesothelioma"]): cues += 1

    # Coverage: presence of matching filters
    coverage = 0
    if "Trial Phase" in sf and ("phase 3" in sf["Trial Phase"].lower() or "phase 2" in sf["Trial Phase"].lower()):
        coverage += 1
    if "Clinical Setting" in sf and any(x in sf["Clinical Setting"].lower() for x in ["adjuvant","neoadjuvant","perioperative","maintenance","first-line","second-line"]):
        coverage += 1
    if "ICI Class" in sf: coverage += 1
    if "Cancer Type" in sf: coverage += 1

    # Parsimony: columns beyond required
    addl_cols = max(0, len(sc) - len(REQUIRED_COLS))
    parsimony_penalty = 0.15 * max(0, addl_cols - MAX_ADDITIONAL_COLS)  # should be 0 due to normalization
    parsimony_penalty += 0.05 * max(0, addl_cols - 4)  # soft penalty for picking too many, even if allowed

    # Schema fidelity bonus
    schema_bonus = 0
    if all(k in CANON_FILTER_KEYS for k in sf.keys()):
        schema_bonus += 0.5
    if all(v in CANON_COLUMN_NAMES or v in REQUIRED_COLS for v in sc.values()):
        schema_bonus += 0.5

    # Consistency
    consistency = 0
    if "Trial Phase" in sf and sf["Trial Phase"] in {"Phase 2","Phase 3"}:
        consistency += 0.25
    if "ICI Class" in sf and sf["ICI Class"] in {"PD-1","PD-L1","CTLA-4"}:
        consistency += 0.25

    # Simple linear mix
    score = 0.8 * cues + 1.2 * coverage + schema_bonus + consistency - parsimony_penalty
    return float(score)

# ============================
# CANDIDATE GENERATION
# ============================

def gen_candidates_for_question(question: str):
    candidates = []

    # Style A: single-stage (N_SINGLE_STAGE_CANDIDATES)
    for _ in range(N_SINGLE_STAGE_CANDIDATES):
        p = build_single_stage_prompt(question)
        raw = call_model_with_retries(p)
        try:
            obj = extract_json(raw)
        except Exception:
            obj = {}
        cand = {
            "selected_filter": (obj.get("selected_filter") or {}),
            "selected_column": (obj.get("selected_column") or {}),
        }
        fixed, audit = validate_and_fix(cand)
        candidates.append({"raw": cand, "fixed": fixed, "audit": audit, "gen_style": "single"})

    # Style B: two-stage (N_TWO_STAGE_CANDIDATES)
    for _ in range(N_TWO_STAGE_CANDIDATES):
        p1 = build_two_stage_prompt_stage1(question)
        raw1 = call_model_with_retries(p1)
        try:
            o1 = extract_json(raw1)
        except Exception:
            o1 = {}
        sel_filter = o1.get("selected_filter") or {}
        # Normalize after Stage 1
        sel_filter_norm = normalize_selected_filter(sel_filter)

        p2 = build_two_stage_prompt_stage2(question, sel_filter_norm)
        raw2 = call_model_with_retries(p2)
        try:
            o2 = extract_json(raw2)
        except Exception:
            o2 = {}
        sel_col = o2.get("selected_column") or {}

        cand = {"selected_filter": sel_filter_norm, "selected_column": sel_col}
        fixed, audit = validate_and_fix(cand)
        candidates.append({"raw": cand, "fixed": fixed, "audit": audit, "gen_style": "two-stage"})

    return candidates

# ============================
# JUDGE + ACCEPT/REVISE
# ============================

def judge_top_candidates(question: str, fixed_candidates: list, top_m: int = TOP_M_FOR_JUDGE):
    # Rank by heuristic score
    scored = []
    for c in fixed_candidates:
        s = score_candidate(question, c["fixed"])
        scored.append((s, c))
    scored.sort(key=lambda x: x[0], reverse=True)
    shortlisted = [c for _, c in scored[:top_m]]

    judged = []
    for c in shortlisted:
        p = build_judge_prompt(question, c["fixed"])
        raw = call_model_with_retries(p)
        try:
            out = extract_json(raw)
        except Exception:
            out = c["fixed"]
        fixed2, audit2 = validate_and_fix(out)
        judged.append({"fixed": fixed2, "audit": audit2, "source": c})

    # pick best judged by score again
    rescored = []
    for j in judged:
        s2 = score_candidate(question, j["fixed"])
        rescored.append((s2, j))
    rescored.sort(key=lambda x: x[0], reverse=True)
    winner = rescored[0][1] if rescored else shortlisted[0]
    return winner

def accept_revise_pass(question: str, winner_fixed: dict):
    p = build_accept_revise_prompt(question, winner_fixed)
    raw = call_model_with_retries(p)
    try:
        obj = extract_json(raw)
    except Exception:
        obj = {}

    status = str(obj.get("status","")).strip().lower()
    if status == "revise":
        sel_f = obj.get("selected_filter", winner_fixed.get("selected_filter", {}))
        sel_c = obj.get("selected_column", winner_fixed.get("selected_column", {}))
        merged = {"selected_filter": sel_f, "selected_column": sel_c}
    elif status == "accept":
        sel_f = obj.get("selected_filter", winner_fixed.get("selected_filter", {}))
        sel_c = obj.get("selected_column", winner_fixed.get("selected_column", {}))
        merged = {"selected_filter": sel_f, "selected_column": sel_c}
    else:
        merged = winner_fixed

    # Final deterministic validation
    return validate_and_fix(merged)

# ============================
# MAIN
# ============================

def main():
    t_all0 = time.time()
    df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)

    # Auto-detect question column
    global QUESTION_COLUMN
    if QUESTION_COLUMN not in df.columns:
        candidates = [c for c in df.columns if str(c).strip().lower() in {"question","query","prompt"}]
        if candidates:
            QUESTION_COLUMN = candidates[0]
        else:
            raise ValueError(f"Couldn't find a question column. Available: {list(df.columns)}")

    results = []
    timings = []

    for idx, row in df.head(MAX_QUESTIONS).iterrows():
        q = str(row[QUESTION_COLUMN]).strip()
        if not q or q.lower() == "nan":
            continue

        t0 = time.time()
        # 1) Generate candidates
        t_gen0 = time.time()
        candidates = gen_candidates_for_question(q)
        t_gen = time.time() - t_gen0

        # 2) Judge top candidates
        t_judge0 = time.time()
        winner = judge_top_candidates(q, candidates, top_m=TOP_M_FOR_JUDGE)
        t_judge = time.time() - t_judge0

        # 3) Accept/Revise on winner
        t_final0 = time.time()
        final_fixed, final_audit = accept_revise_pass(q, winner["fixed"])
        t_final = time.time() - t_final0

        total_t = time.time() - t0

        results.append({
            "row_index": idx,
            "question": q,
            "final_json": json.dumps(final_fixed, ensure_ascii=False),
            "audit_json": json.dumps(final_audit, ensure_ascii=False),
        })
        timings.append({
            "row_index": idx,
            "question": q,
            "t_generate_sec": t_gen,
            "t_judge_sec": t_judge,
            "t_accept_revise_sec": t_final,
            "t_total_sec": total_t,
        })
        print(f"[Pipeline 4] Row {idx} done. Total {total_t:.2f}s")

    results_df = pd.DataFrame(results)
    timings_df = pd.DataFrame(timings)

    with pd.ExcelWriter(EXCEL_PATH_OUT, engine="openpyxl") as writer:
        results_df.to_excel(writer, index=False, sheet_name="results")
        timings_df.to_excel(writer, index=False, sheet_name="timings")

    print(f"Saved results to: {EXCEL_PATH_OUT} (elapsed {time.time()-t_all0:.2f}s)")

if __name__ == "__main__":
    main()



In [22]:
# Pipeline 4.1 — Schema-Constrained Self-Consistency + Verifier Reranking. Fixed for Gemini 2.5
# --------------------------------------------------------------------
# High-level:
#  - Generate K candidates (mix of single-stage and two-stage prompts).
#  - Deterministically canonicalize & validate (closed set for filters/columns).
#  - Score/rerank by intent coverage, parsimony, and schema fidelity.
#  - Short "judge" pass on the top M candidates to correct/trim within the same schema.
#  - Accept/Revise self-check on the winner; re-validate; write results + timings + audit.
#
# Notes:
#  - Uses your same definition files and environment variable layout.
#  - Enforces canonical filter NAMES and common VALUE synonyms.
#  - Enforces REQUIRED_COLS and a cap on additional columns.
#  - Saves results + timings to an Excel file (two sheets).
#
# Env:
#   GEMINI_KEY must be set (e.g., via .env)
#
# Files (update paths as needed):
#   FILTER_NAMES_DEF_PATH   (filter CATEGORY names only; used in prompts)
#   COLUMN_DEFS_PATH        (column definitions; used in prompts)
#   FILTER_DEFS_FULL_PATH   (full filter categories + allowed values; used in prompts)
#
# Input:
#   EXCEL_PATH_IN  (expects a column named one of {"question","query","prompt"})
#
# Output:
#   EXCEL_PATH_OUT with sheets:
#     - results: final_json, audit_json (per question)
#     - timings: per-stage timings (per question)

import os, time, random, json, ast, re, math
import pandas as pd
from dotenv import load_dotenv

import google.generativeai as genai
from google.api_core.exceptions import ResourceExhausted, DeadlineExceeded, InternalServerError

import hashlib, sqlite3, threading, concurrent.futures

# --- Concurrency knobs ---
MAX_MODEL_WORKERS = int(os.getenv("GEMINI_MAX_WORKERS", "8"))   # safe default; tune to your quota
EXECUTOR = concurrent.futures.ThreadPoolExecutor(max_workers=MAX_MODEL_WORKERS)

# --- Persistent on-disk cache for model calls ---
CACHE_PATH = os.getenv("GEMINI_CACHE_PATH", ".gemini_cache.sqlite3")
_CACHE_LOCK = threading.Lock()

def _cache_init():
    conn = sqlite3.connect(CACHE_PATH, check_same_thread=False)
    conn.execute("""
      CREATE TABLE IF NOT EXISTS cache (
        phash TEXT PRIMARY KEY,
        model TEXT,
        prompt TEXT,
        response TEXT,
        ts REAL
      )
    """)
    conn.commit()
    return conn

_CACHE_CONN = _cache_init()

def _prompt_hash(model: str, prompt: str) -> str:
    return hashlib.sha256((model + "||" + prompt).encode("utf-8")).hexdigest()

def cache_get(model: str, prompt: str):
    ph = _prompt_hash(model, prompt)
    with _CACHE_LOCK:
        cur = _CACHE_CONN.execute("SELECT response FROM cache WHERE phash=?", (ph,))
        row = cur.fetchone()
    return row[0] if row else None

def cache_set(model: str, prompt: str, response: str):
    ph = _prompt_hash(model, prompt)
    ts = time.time()
    with _CACHE_LOCK:
        _CACHE_CONN.execute(
            "INSERT OR REPLACE INTO cache(phash, model, prompt, response, ts) VALUES (?,?,?,?,?)",
            (ph, model, prompt, response, ts)
        )
        _CACHE_CONN.commit()


# ============================
# CONFIG
# ============================
EXCEL_PATH_IN   = "runs/query-chosen-filters-columns.xlsx"
EXCEL_PATH_OUT  = "runs/query-chosen-filters-columns_with_runs_pipeline4_base.xlsx"
SHEET_NAME      = 0
QUESTION_COLUMN = "question"  # auto-detected if missing
MAX_QUESTIONS   = 100

# Candidate generation
K_CANDIDATES             = 6   # total candidates per question (e.g., 3 single-stage + 3 two-stage)
N_SINGLE_STAGE_CANDIDATES = 3  # Style A
N_TWO_STAGE_CANDIDATES    = 3  # Style B
TOP_M_FOR_JUDGE           = 2  # rerank -> judge the top M
MODEL_NAME                = "gemini-2.0-flash"

# Required columns & cap (do NOT count these toward the additional cap)
REQUIRED_COLS        = ["NCT", "PMID", "Authors", "Year"]
MAX_ADDITIONAL_COLS  = 6

# Definition files (reuse your existing)
#  - If your files live elsewhere, update these paths.
FILTER_NAMES_DEF_PATH = "definitions_folder/definitions - aim2 - filter names.txt"
COLUMN_DEFS_PATH      = "definitions_folder/definitions - aim2 - column.txt"
FILTER_DEFS_FULL_PATH = "definitions_folder/definitions - aim2 - filter.txt"

# ============================
# SETUP
# ============================
load_dotenv()
GEMINI_KEY = os.getenv("GEMINI_KEY")
if not GEMINI_KEY:
    raise RuntimeError("GEMINI_KEY environment variable is not set")
genai.configure(api_key=GEMINI_KEY)

GENERATION_CONFIG = {
    "response_mime_type": "application/json",
    "temperature": 0.0,
}

with open(FILTER_NAMES_DEF_PATH, "r", encoding="utf-8") as f:
    FILTER_NAMES_TEXT = f.read()
with open(COLUMN_DEFS_PATH, "r", encoding="utf-8") as f:
    COLUMN_DEFS_TEXT = f.read()
with open(FILTER_DEFS_FULL_PATH, "r", encoding="utf-8") as f:
    FILTER_DEFS_TEXT = f.read()

# ============================
# CANONICAL SCHEMA (closed sets)
# ============================

# Canonical filter category names (interface)
CANON_FILTER_KEYS = [
    "ICI Class",
    "ICI Name",
    "Cancer Type",
    "Type of Therapy",
    "Type of combination (Treatment Arm)",
    "Control Arm",
    "Clinical Setting",
    "Trial Phase",
    "Type of Study",
    "Primary Endpoint",
    "Included in MA",
]

# Canonical column names (add/trim as needed to match your UI exactly)
CANON_COLUMN_NAMES = [
    "NCT","PMID","Authors","Year","Original/Follow Up","Study name","Trial phase",
    "Number of arms","Cancer type","Treatment regimen","Name of ICI","Class of ICI",
    "Monotherapy/combination","Type of combination","Control regimen","Type of control",
    "Total sample size","Lines of treatment","Clinical setting in relation to surgery",
    "Is PD-L1 positivity inclusion criteria","Is any other biomarker used for inclusion",
    "Primary endpoint","Secondary endpoint","Type of follow-up given",
    "Follow-up duration for primary endpoint(s) in months","Included in MA",
    "Primary multiple, composite, or co-primary endpoints?"  # keep as a single column label if you use it
]

# Synonyms for filter keys (→ canonical)
CANON_KEYS_MAP = {
    # filters
    "Class of ICI": "ICI Class",
    "Name of ICI": "ICI Name",
    "Cancer type": "Cancer Type",
    "Monotherapy/combination": "Type of Therapy",
    "Monotherapy/Combination": "Type of Therapy",
    "Mono/Combo": "Type of Therapy",
    "Trial phase": "Trial Phase",
    "Clinical setting in relation to surgery": "Clinical Setting",
    "Clinical setting": "Clinical Setting",
    "Perioperative setting": "Clinical Setting",
    "Setting in relation to surgery": "Clinical Setting",
    "Type of combination": "Type of combination (Treatment Arm)",
    "Type of control": "Control Arm",
    "Control regimen": "Control Arm",
    "Primary endpoint": "Primary Endpoint",
    "Primary Endpoint(s)": "Primary Endpoint",
    "Primary endpoints": "Primary Endpoint",
    "Included in Meta-analysis": "Included in MA",
}

# Canonicalized value sets (where enumerated)
ALLOWED_VALUES = {
    "ICI Class": {"PD-1", "PD-L1", "CTLA-4"},
    "Type of Therapy": {"Monotherapy", "Combination"},
    "Primary Endpoint": {"OS","PFS","ORR","RFS","EFS","Path CR","Safety"},
    "Trial Phase": {"Phase 2","Phase 3"},
    "Type of Study": {"Original publication","Follow-up"},
    "Included in MA": {"Yes","No"},
    # Some common arms (not exhaustive; expand as needed)
    "Type of combination (Treatment Arm)": {
        "ICI + Chemo","ICI + TKI","ICI + ICI","ICI + ICI + Chemo","ICI + Radiation",
        "ICI + Vaccine","ICI + MEKi","ICI + Anti-VEGF","ICI + BRAFi + MEKi","ICI + Chemo + Anti-VEGF"
    },
    "Control Arm": {
        "Placebo","Chemo","TKI","Interferon","Best Supportive Care","Radiation","Vaccine",
        "Multikinase inhibitor","mTOR inhibitor","Chemo / Anti-VEGF","Chemo / Anti-EGFR"
    },
    # Cancer Type is large; we keep this open and rely on synonym canonicalization below.
}

# Value synonym maps
ICI_CLASS_VALUE_MAP = {
    "pd1": "PD-1","pd-1": "PD-1","pd 1":"PD-1",
    "pdl1": "PD-L1","pd-l1":"PD-L1","pd l1":"PD-L1","pd_l1":"PD-L1",
    "ctla4":"CTLA-4","ctla-4":"CTLA-4","ctla 4":"CTLA-4",
}
TYPE_OF_THERAPY_VALUE_MAP = {
    "mono":"Monotherapy","monotherapy":"Monotherapy",
    "combination therapy":"Combination","combination":"Combination",
}
PRIMARY_ENDPOINT_VALUE_MAP = {
    "os (overall survival)":"OS","os":"OS","pfs":"PFS","orr":"ORR",
    "rfs":"RFS","dfs":"RFS","efs":"EFS","pcr":"Path CR","path cr":"Path CR",
    "safety":"Safety",
}
TRIAL_PHASE_VALUE_MAP = {
    "iii":"Phase 3","phase iii":"Phase 3","3":"Phase 3",
    "ii":"Phase 2","phase ii":"Phase 2","2":"Phase 2",
}
TYPE_OF_STUDY_VALUE_MAP = {
    "original":"Original publication","original publication":"Original publication",
    "follow up":"Follow-up","follow-up":"Follow-up",
}
INCLUDED_MA_VALUE_MAP = {"yes":"Yes","no":"No"}

COMBO_TREATMENT_VALUE_MAP = {
    "ici + meki (mek inhibitor)":"ICI + MEKi","ici + meki":"ICI + MEKi",
    "ici + radiation":"ICI + Radiation","ici + radiotherapy":"ICI + Radiation",
    "ici + vaccine":"ICI + Vaccine","ici + tki":"ICI + TKI","ici + chemo":"ICI + Chemo",
    "ici + anti-vegf":"ICI + Anti-VEGF","ici + brafi + meki":"ICI + BRAFi + MEKi",
    "ici + chemo + anti-vegf":"ICI + Chemo + Anti-VEGF","ici + ici":"ICI + ICI",
    "ici + ici + chemo":"ICI + ICI + Chemo"
}
CONTROL_ARM_VALUE_MAP = {
    "chemo":"Chemo","tki":"TKI","interferon":"Interferon",
    "multikinase inhibitor":"Multikinase inhibitor","mtor inhibitor":"mTOR inhibitor",
    "radiation":"Radiation","vaccine":"Vaccine","placebo":"Placebo","bsc":"Best Supportive Care",
    "best supportive care":"Best Supportive Care","chemo / anti-egfr":"Chemo / Anti-EGFR",
    "chemo / anti-vegf":"Chemo / Anti-VEGF",
}
# Cancer Type synonyms (examples)
CANCER_TYPE_VALUE_MAP = {
    "nsclc":"Non-Small Cell Lung Cancer (NSCLC)",
    "non small cell lung cancer":"Non-Small Cell Lung Cancer (NSCLC)",
    "non-small cell lung cancer":"Non-Small Cell Lung Cancer (NSCLC)",
    "sclc":"Small Cell Lung Cancer (SCLC)",
    "small cell lung cancer":"Small Cell Lung Cancer (SCLC)",
    "hnscc":"Head and Neck (HNSCC)","head and neck (hnscc)":"Head and Neck (HNSCC)",
    "head and neck":"Head and Neck",
    "hcc":"Hepatocellular carcinoma (HCC)","hepatocellular carcinoma":"Hepatocellular carcinoma (HCC)",
    "rcc":"Renal Cell Carcinoma (RCC)","renal cell carcinoma":"Renal Cell Carcinoma (RCC)",
    "melanoma":"Melanoma","breast":"Breast","colorectal":"Colorectal",
    "gastric/gej":"Gastric/GEJ","esophageal/gej":"Esophageal/GEJ","bladder":"Bladder","urothelial":"Bladder"
}

# Column synonym normalizer (value -> canonical column)
COLUMN_SYNONYMS = {
    "original/follow up":"Original/Follow Up","original/follow-up":"Original/Follow Up",
    "follow up duration for primary endpoints (overall, rx, control)":"Follow-up duration for primary endpoint(s) in months",
    "trial phase":"Trial phase","cancer type":"Cancer type","name of ici":"Name of ICI","class of ici":"Class of ICI",
    "monotherapy/combination":"Monotherapy/combination","type of combination":"Type of combination",
    "control regimen":"Control regimen","type of control":"Type of control","total sample size":"Total sample size",
    "lines of treatment":"Lines of treatment","clinical setting in relation to surgery":"Clinical setting in relation to surgery",
    "is pd-l1 positivity inclusion criteria":"Is PD-L1 positivity inclusion criteria",
    "is any other biomarker used for inclusion":"Is any other biomarker used for inclusion",
    "primary endpoint":"Primary endpoint","secondary endpoint":"Secondary endpoint",
    "type of follow-up given":"Type of follow-up given",
    "follow-up duration for primary endpoint(s) in months":"Follow-up duration for primary endpoint(s) in months",
    "included in ma":"Included in MA",
    "primary multiple, composite, or co-primary endpoints?":"Primary multiple, composite, or co-primary endpoints?",
}

# ============================
# UTILS
# ============================

def _as_dict(obj, preferred_keys=("selected_filter","selected_column","status")) -> dict:
    """
    Coerce model output into a dict.
    - If dict: return as-is.
    - If list: return the first dict that contains any preferred key; else first dict; else merged dicts; else {}.
    - Otherwise: {}.
    """
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, list):
        for item in obj:
            if isinstance(item, dict) and any(k in item for k in preferred_keys):
                return item
        dicts = [d for d in obj if isinstance(d, dict)]
        if dicts:
            if len(dicts) == 1:
                return dicts[0]
            merged = {}
            for d in dicts:
                merged.update(d)
            return merged
        return {}
    return {}


def strip_code_fences(s: str) -> str:
    s = (s or "").strip()
    if s.startswith("```") and s.endswith("```"):
        # remove first line fence and last fence
        lines = s.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        s = "\n".join(lines)
    return s.strip()

def extract_json(text: str):
    raw = strip_code_fences(text)
    # Try direct JSON
    try:
        return json.loads(raw)
    except Exception:
        pass
    # Fallback: first {...} block
    try:
        start = raw.find("{"); end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = raw[start:end+1]
            try:
                return json.loads(candidate)
            except Exception:
                obj = ast.literal_eval(candidate)
                if isinstance(obj, (dict, list)):
                    return obj
    except Exception:
        pass
    raise ValueError(f"Could not parse JSON from model text: {text[:200]}...")

def call_model_with_retries(prompt: str, timeout=90, retries=3):
    # cache hit?
    cached = cache_get(MODEL_NAME, prompt)
    if cached is not None:
        return cached

    model = genai.GenerativeModel(MODEL_NAME)
    backoff = 1.0
    last_err = None
    for attempt in range(1, retries+1):
        try:
            resp = model.generate_content(
                prompt,
                generation_config=GENERATION_CONFIG,
                request_options={"timeout": timeout},
                safety_settings=None,
            )
            txt = getattr(resp, "text", None)
            out = (txt if txt is not None else str(resp)).strip()
            cache_set(MODEL_NAME, prompt, out)  # persist for restarts/repeats
            return out
        except (ResourceExhausted, DeadlineExceeded, InternalServerError) as e:
            last_err = e
            # soft backoff; let concurrency keep others moving
            time.sleep(backoff * (2 ** (attempt - 1)) * (1 + random.uniform(0, 0.35)))
        except Exception as e:
            last_err = e
            # try a couple of retries on unknowns; then bail with the error string (won't crash)
            if attempt < min(3, retries):
                time.sleep(backoff * attempt)
            else:
                break
    return f"[ERROR after {retries} attempts] {last_err}"


def _canon_simple(s) -> str:
    """
    Return a trimmed single string for any input.
    - Lists/Tuples/Sets -> first non-empty item (stringified)
    - Dicts -> try common keys ('value','name','label'), else first non-empty value
    - Anything else -> str(...) and trim
    """
    # Handle list-like
    if isinstance(s, (list, tuple, set)):
        for item in s:
            if item is not None and str(item).strip():
                return re.sub(r"\s+", " ", str(item).strip())
        return ""
    # Handle dict-like
    if isinstance(s, dict):
        for key in ("value", "name", "label"):
            if key in s and s[key] is not None and str(s[key]).strip():
                return re.sub(r"\s+", " ", str(s[key]).strip())
        # fallback: first non-empty value
        for v in s.values():
            if v is not None and str(v).strip():
                return re.sub(r"\s+", " ", str(v).strip())
        return ""
    # Default string path
    return re.sub(r"\s+", " ", (str(s or "")).strip())


def _listify_values(v) -> list:
    """Return a list of candidate raw values for a filter value field."""
    if isinstance(v, (list, tuple, set)):
        return list(v)
    if isinstance(v, dict):
        # prioritize common keys then everything
        candidates = []
        for k in ("value", "name", "label"):
            if k in v:
                candidates.append(v[k])
        # include all values as a last resort
        candidates.extend([x for x in v.values() if x not in candidates])
        return candidates
    return [v]


def normalize_selected_filter(sel: dict) -> dict:
    """
    - Accepts values that may be strings, lists, tuples, sets, or dicts.
    - Picks a single canonical value per category.
    - Drops empties and 'All'.
    """
    out = {}
    for k, v in (sel or {}).items():
        k_can = _norm_key(k)
        if not k_can:
            continue

        candidates = _listify_values(v)
        chosen = ""

        # 1) Try to pick the first candidate that canonicalizes into an allowed value (when enumerated)
        if k_can in ALLOWED_VALUES:
            allowed = ALLOWED_VALUES[k_can]
            for c in candidates:
                c1 = _canon_simple(c)
                if not c1 or c1.lower() == "all":
                    continue
                c2 = _canon_filter_value(k_can, c1)
                if c2 in allowed:
                    chosen = c2
                    break

        # 2) If none matched (or category is open like Cancer Type), pick first non-empty candidate after canonicalization
        if not chosen:
            for c in candidates:
                c1 = _canon_simple(c)
                if not c1 or c1.lower() == "all":
                    continue
                chosen = _canon_filter_value(k_can, c1)
                if chosen:
                    break

        if chosen:
            out[k_can] = chosen

    return out

def _norm_key(key: str) -> str:
    k = _canon_simple(key)
    return CANON_KEYS_MAP.get(k, k)

def _canon_filter_value(cat: str, value: str) -> str:
    cat_c = _norm_key(cat)
    v = _canon_simple(value)
    low = v.lower()

    if cat_c == "ICI Class": return ICI_CLASS_VALUE_MAP.get(low, v)
    if cat_c == "Type of Therapy": return TYPE_OF_THERAPY_VALUE_MAP.get(low, v)
    if cat_c == "Primary Endpoint": return PRIMARY_ENDPOINT_VALUE_MAP.get(low, v)
    if cat_c == "Trial Phase": return TRIAL_PHASE_VALUE_MAP.get(low, v)
    if cat_c == "Type of Study": return TYPE_OF_STUDY_VALUE_MAP.get(low, v)
    if cat_c == "Included in MA": return INCLUDED_MA_VALUE_MAP.get(low, v)
    if cat_c == "Type of combination (Treatment Arm)": return COMBO_TREATMENT_VALUE_MAP.get(low, v)
    if cat_c == "Control Arm": return CONTROL_ARM_VALUE_MAP.get(low, v)
    if cat_c == "Cancer Type": return CANCER_TYPE_VALUE_MAP.get(low, v)
    if cat_c == "Clinical Setting":
        vv = low
        if vv in {"neoadjuvant","adjuvant","perioperative","maintenance"}:
            return v.title()
        # fold line-of-therapy to settings
        if vv in {"1l","first line","first-line"}: return "First-line metastatic"
        if any(x in vv for x in ["2l","3l","second line","third line","second-line","third-line","2l+","second-line+"]):
            return "Second-line+"
        if vv in {"metastatic / recurrent (unresectable)"}:
            return "Metastatic / Recurrent (Unresectable)"
        return v
    return v

def _ordered_values_from_column_object(col_obj: dict) -> list[str]:
    if not isinstance(col_obj, dict):
        return []
    items = []
    for k, v in col_obj.items():
        m = re.search(r"Column\s*(\d+)", str(k))
        idx = int(m.group(1)) if m else 10**9
        items.append((idx, v))
    items.sort(key=lambda x: x[0])
    return [val for _, val in items if isinstance(val, str) and val and str(val).strip()]

def _normalize_column_name(name: str) -> str:
    raw = _canon_simple(name)
    key = raw.lower()
    if key in COLUMN_SYNONYMS:
        return COLUMN_SYNONYMS[key]
    return raw

def normalize_selected_column(selected_column) -> dict:
    # Accept dict or list and rebuild enumerated object with REQUIRED first and a cap
    if isinstance(selected_column, dict):
        cols = _ordered_values_from_column_object(selected_column)
    elif isinstance(selected_column, list):
        cols = [c for c in selected_column if isinstance(c, str) and c.strip()]
    else:
        cols = []

    # normalize names, drop placeholders
    normed = []
    for c in cols:
        cc = _normalize_column_name(c)
        if cc.lower() == "all the remaining columns":  # reject placeholder
            continue
        normed.append(cc)

    out_list, seen = [], set()
    for rc in REQUIRED_COLS:
        if rc not in seen:
            out_list.append(rc); seen.add(rc)
    for c in normed:
        c2 = _normalize_column_name(c)
        if c2 in CANON_COLUMN_NAMES and c2 not in seen:
            out_list.append(c2); seen.add(c2)

    cap = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    out_list = out_list[:cap]
    return {f"Column {i+1}": name for i, name in enumerate(out_list)}


def validate_and_fix(candidate: dict):
    """Return (fixed_candidate, audit_dict)"""
    audit = {
        "dropped_non_canonical_filter_keys": [],
        "dropped_filter_values": [],
        "composite_ici_class_rejected": False,
        "moved_filters_that_are_columns": [],
        "column_synonyms_applied": [],
        "trimmed_columns_count": 0,
        "notes": [],
    }

    sf = dict(candidate.get("selected_filter", {}) or {})
    sc = candidate.get("selected_column", {}) or {}

    # Canonicalize
    sf = normalize_selected_filter(sf)
    sc = normalize_selected_column(sc)

    # Reject non-canonical filter keys
    sf_fixed = {}
    for k, v in sf.items():
        if k not in CANON_FILTER_KEYS:
            audit["dropped_non_canonical_filter_keys"].append(k)
            continue
        # reject composite ICI Class
        if k == "ICI Class" and isinstance(v, str) and re.search(r"\bor\b|,|both", v, flags=re.I):
            audit["composite_ici_class_rejected"] = True
            continue
        # where enumerated, ensure value is in allowed set
        if k in ALLOWED_VALUES:
            if v not in ALLOWED_VALUES[k]:
                audit["dropped_filter_values"].append({k: v})
                continue
        sf_fixed[k] = v

    # Hard rule: things that were mistakenly put as filters but are columns -> drop from filters
    column_like_filters = {
        "Year","Total sample size","Lines of treatment",
        "Is PD-L1 positivity inclusion criteria","Is any other biomarker used for inclusion",
        "Primary multiple, composite, or co-primary endpoints?"
    }
    for bad in list(sf_fixed.keys()):
        if bad in column_like_filters:
            audit["moved_filters_that_are_columns"].append({bad: sf_fixed[bad]})
            del sf_fixed[bad]

    # Ensure columns are all canonical & capped (already done in normalize)
    # Count trimming (if any)
    final_cols = list(sc.values())
    if len(final_cols) > len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS:
        audit["trimmed_columns_count"] = len(final_cols) - (len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS)

    fixed = {"selected_filter": sf_fixed, "selected_column": normalize_selected_column(sc)}
    return fixed, audit

# ============================
# PROMPTS
# ============================

def build_single_stage_prompt(question: str) -> str:
    total_slots = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    schema_columns = ',\n'.join([f'    "Column {i}": "Name"' for i in range(1, total_slots + 1)])
    schema = (
        '{\n'
        '  "selected_filter": {\n'
        '    "Filter Category 1": "Chosen Value",\n'
        '    "Filter Category 2": "Chosen Value"\n'
        '  },\n'
        '  "selected_column": {\n'
        f'{schema_columns}\n'
        '  }\n'
        '}'
    )
    return (
        "You are a medical expert researching cancer trials.\n"
        "Return ONLY a valid JSON object (no prose, no code fences) with this schema:\n"
        f"{schema}\n"
        "Strict rules:\n"
        f"- Columns: ALWAYS include these first (do NOT count toward the limit): {', '.join(REQUIRED_COLS)}.\n"
        f"- You may add at most {MAX_ADDITIONAL_COLS} additional columns beyond those required.\n"
        "- The enumerated object keys MUST be exactly 'Column 1', 'Column 2', ... in display order.\n"
        "- Use exact column names from the list below.\n"
        "- Filters: choose only categories/values justified by the question. If a category would be 'All', OMIT it.\n"
        "- Use exact category and value strings from the lists below.\n"
        "- Use double quotes everywhere. No trailing commas. No explanations.\n\n"
        "Available filter CATEGORY NAMES (reference):\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available FULL filter categories and values (use exact strings; omit categories that would be 'All'):\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        "Available columns and definitions (select by exact name):\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now."
    )

def build_two_stage_prompt_stage1(question: str) -> str:
    total_slots = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    schema_columns = ',\n'.join([f'    "Column {i}": "Name"' for i in range(1, total_slots + 1)])
    schema = (
        '{\n'
        '  "selected_filter": {\n'
        '    "Filter Category 1": "Chosen Value",\n'
        '    "Filter Category 2": "Chosen Value"\n'
        '  }\n'
        '}'
    )
    # We directly choose filter values in stage 1 (names+values), then choose columns in stage 2
    return (
        "You are a medical expert researching cancer.\n"
        "Stage 1: choose FILTERS (categories + values) ONLY.\n"
        "Return ONLY a valid JSON with key `selected_filter` (no prose):\n"
        f"{schema}\n\n"
        "Rules:\n"
        "- Use exact category names from this canonical list: "
        '["ICI Class","ICI Name","Cancer Type","Type of Therapy","Type of combination (Treatment Arm)",'
        '"Control Arm","Clinical Setting","Trial Phase","Type of Study","Primary Endpoint","Included in MA"]\n'
        "- Use exact strings for values from the filter definitions.\n"
        "- Omit any filters whose correct value would be \"All\".\n"
        "- Include only filters directly justified by the question.\n\n"
        "Available filter CATEGORY NAMES:\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available filter CATEGORIES + VALUES (use exact strings whenever possible):\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now."
    )

def build_two_stage_prompt_stage2(question: str, selected_filter: dict) -> str:
    total_slots = len(REQUIRED_COLS) + MAX_ADDITIONAL_COLS
    schema_columns = ',\n'.join([f'    "Column {i}": "Name"' for i in range(1, total_slots + 1)])
    schema = '{\n  "selected_column": {\n' + schema_columns + '\n  }\n}'
    return (
        "You are a medical expert researching cancer.\n"
        "Stage 2: choose COLUMNS ONLY (REQUIRED first; at most additional columns as allowed).\n"
        "Return ONLY a valid JSON with key `selected_column` (no prose):\n"
        f"{schema}\n\n"
        "Rules:\n"
        f"- ALWAYS include: {', '.join(REQUIRED_COLS)} (do not count toward the cap).\n"
        f"- Choose at most {MAX_ADDITIONAL_COLS} additional columns.\n"
        "- Use exact column names from the list below.\n\n"
        "Context — chosen filters:\n"
        f"{json.dumps(selected_filter, ensure_ascii=False)}\n\n"
        "Available columns and definitions (select by exact name):\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        "Return JSON now."
    )

def build_judge_prompt(question: str, candidate_json: dict) -> str:
    schema = (
        '{ "selected_filter": { ... }, "selected_column": { "Column 1": "...", "Column 2": "...", ... } }'
    )
    return (
        "You are a meticulous validator.\n"
        "Task: Examine the candidate JSON (filters+columns) against the schema and the question. "
        "If it is minimal and correct, return it unchanged. Otherwise, return a corrected JSON that:\n"
        "- Uses only canonical filter category names and allowed values (or valid cancer types),\n"
        "- Omits categories whose correct value would be 'All',\n"
        "- Includes REQUIRED columns first (NCT, PMID, Authors, Year) and at most 6 additional columns,\n"
        "- Uses exact column names from the list.\n\n"
        "Return ONLY a valid JSON object (no prose) with this schema:\n"
        f"{schema}\n\n"
        "Available canonical filter CATEGORY NAMES:\n"
        f"{FILTER_NAMES_TEXT}\n\n"
        "Available filter CATEGORIES + VALUES:\n"
        f"{FILTER_DEFS_TEXT}\n\n"
        "Available columns and definitions:\n"
        f"{COLUMN_DEFS_TEXT}\n\n"
        f"Question: {question}\n"
        f"Candidate JSON:\n{json.dumps(candidate_json, ensure_ascii=False)}\n"
        "Return JSON now."
    )

def build_accept_revise_prompt(question: str, merged: dict) -> str:
    return (
        "Final check: evaluate the COMBINATION of filters + columns for relevance, correctness, and parsimony.\n"
        'Return ONLY one JSON: {"status":"accept","selected_filter":{...},"selected_column":{...}} '
        'or {"status":"revise","selected_filter":{...},"selected_column":{...},"notes":"<25 chars>"}\n\n'
        f"Question: {question}\n"
        f"Current selection: {json.dumps(merged, ensure_ascii=False)}\n"
        "Rules:\n"
        "- Filters must use canonical category names.\n"
        f"- Columns must include {', '.join(REQUIRED_COLS)} and only up to {MAX_ADDITIONAL_COLS} extras.\n"
        "- Omit any filter whose correct value would be 'All'.\n"
        "- Prefer minimal set that answers the question; drop irrelevant columns.\n"
        "Return JSON now."
    )

# ============================
# SCORING / RERANK
# ============================

def score_candidate(question: str, cand: dict) -> float:
    """Heuristic scoring: intent coverage (+), parsimony (-), schema fidelity (+), consistency (+)."""
    q = (question or "").lower()

    sf = cand.get("selected_filter", {}) or {}
    sc = cand.get("selected_column", {}) or {}

    # Intent cues (simple keyword checks)
    cues = 0
    if re.search(r"\bphase\s*3\b|phase iii|\biii\b", q): cues += 1
    if re.search(r"\bphase\s*2\b|phase ii|\bii\b", q): cues += 1
    if any(x in q for x in ["adjuvant","neoadjuvant","perioperative","maintenance","first-line","second-line","1l","2l","3l"]): cues += 1
    if any(x in q for x in ["pd-1","pd1","pdl1","pd-l1","ctla-4","ctla4"]): cues += 1
    if any(x in q for x in ["nsclc","non-small cell","sclc","melanoma","rcc","hnscc","esophageal","gastric","bladder","urothelial","breast","colorectal","hcc","pancreatic","prostate","mesothelioma"]): cues += 1

    # Coverage: presence of matching filters
    coverage = 0
    if "Trial Phase" in sf and ("phase 3" in sf["Trial Phase"].lower() or "phase 2" in sf["Trial Phase"].lower()):
        coverage += 1
    if "Clinical Setting" in sf and any(x in sf["Clinical Setting"].lower() for x in ["adjuvant","neoadjuvant","perioperative","maintenance","first-line","second-line"]):
        coverage += 1
    if "ICI Class" in sf: coverage += 1
    if "Cancer Type" in sf: coverage += 1

    # Parsimony: columns beyond required
    addl_cols = max(0, len(sc) - len(REQUIRED_COLS))
    parsimony_penalty = 0.15 * max(0, addl_cols - MAX_ADDITIONAL_COLS)  # should be 0 due to normalization
    parsimony_penalty += 0.05 * max(0, addl_cols - 4)  # soft penalty for picking too many, even if allowed

    # Schema fidelity bonus
    schema_bonus = 0
    if all(k in CANON_FILTER_KEYS for k in sf.keys()):
        schema_bonus += 0.5
    if all(v in CANON_COLUMN_NAMES or v in REQUIRED_COLS for v in sc.values()):
        schema_bonus += 0.5

    # Consistency
    consistency = 0
    if "Trial Phase" in sf and sf["Trial Phase"] in {"Phase 2","Phase 3"}:
        consistency += 0.25
    if "ICI Class" in sf and sf["ICI Class"] in {"PD-1","PD-L1","CTLA-4"}:
        consistency += 0.25

    # Simple linear mix
    score = 0.8 * cues + 1.2 * coverage + schema_bonus + consistency - parsimony_penalty
    return float(score)

# ============================
# CANDIDATE GENERATION
# ============================

def _extract_json_safe(text):
    try:
        return extract_json(text)
    except Exception:
        return {}

# def gen_candidates_for_question(question: str):
#     candidates = []

#     # Style A: single-stage (N_SINGLE_STAGE_CANDIDATES)
#     for _ in range(N_SINGLE_STAGE_CANDIDATES):
#         p = build_single_stage_prompt(question)
#         raw = call_model_with_retries(p)
#         try:
#             obj = extract_json(raw)
#         except Exception:
#             obj = {}
#         cand = {
#             "selected_filter": (obj.get("selected_filter") or {}),
#             "selected_column": (obj.get("selected_column") or {}),
#         }
#         fixed, audit = validate_and_fix(cand)
#         candidates.append({"raw": cand, "fixed": fixed, "audit": audit, "gen_style": "single"})

#     # Style B: two-stage (N_TWO_STAGE_CANDIDATES)
#     for _ in range(N_TWO_STAGE_CANDIDATES):
#         p1 = build_two_stage_prompt_stage1(question)
#         raw1 = call_model_with_retries(p1)
#         try:
#             o1 = extract_json(raw1)
#         except Exception:
#             o1 = {}
#         sel_filter = o1.get("selected_filter") or {}
#         # Normalize after Stage 1
#         sel_filter_norm = normalize_selected_filter(sel_filter)

#         p2 = build_two_stage_prompt_stage2(question, sel_filter_norm)
#         raw2 = call_model_with_retries(p2)
#         try:
#             o2 = extract_json(raw2)
#         except Exception:
#             o2 = {}
#         sel_col = o2.get("selected_column") or {}

#         cand = {"selected_filter": sel_filter_norm, "selected_column": sel_col}
#         fixed, audit = validate_and_fix(cand)
#         candidates.append({"raw": cand, "fixed": fixed, "audit": audit, "gen_style": "two-stage"})

#    return candidates

def gen_candidates_for_question(question: str):
    candidates = []

    # --- Style A (single-stage) in parallel ---
    single_prompts = [build_single_stage_prompt(question) for _ in range(N_SINGLE_STAGE_CANDIDATES)]
    single_futs = [EXECUTOR.submit(call_model_with_retries, p) for p in single_prompts]
    for fut in concurrent.futures.as_completed(single_futs):
        raw = fut.result()
        obj = _as_dict(_extract_json_safe(raw))
        cand = {
            "selected_filter": (obj.get("selected_filter") or {}),
            "selected_column": (obj.get("selected_column") or {}),
        }

        fixed, audit = validate_and_fix(cand)
        candidates.append({"raw": cand, "fixed": fixed, "audit": audit, "gen_style": "single"})

    # --- Style B (two-stage): Stage 1 calls in parallel ---
    stage1_prompts = [build_two_stage_prompt_stage1(question) for _ in range(N_TWO_STAGE_CANDIDATES)]
    stage1_futs = [EXECUTOR.submit(call_model_with_retries, p) for p in stage1_prompts]
    stage1_objs = [_as_dict(_extract_json_safe(f.result()), preferred_keys=("selected_filter",)) 
               for f in stage1_futs]

    # Normalize Stage 1 results, then Stage 2 calls in parallel
    stage2_prompts = []
    stage2_contexts = []
    for o1 in stage1_objs:
        sel_filter = o1.get("selected_filter") or {}
        sel_filter_norm = normalize_selected_filter(sel_filter)
        stage2_prompts.append(build_two_stage_prompt_stage2(question, sel_filter_norm))
        stage2_contexts.append(sel_filter_norm)

    stage2_futs = [EXECUTOR.submit(call_model_with_retries, p) for p in stage2_prompts]
    for sel_filter_norm, fut in zip(stage2_contexts, stage2_futs):
        raw2 = fut.result()
        o2 = _as_dict(_extract_json_safe(raw2), preferred_keys=("selected_column",))
        sel_col = o2.get("selected_column") or {}
        cand = {"selected_filter": sel_filter_norm, "selected_column": sel_col}
        fixed, audit = validate_and_fix(cand)
        candidates.append({"raw": cand, "fixed": fixed, "audit": audit, "gen_style": "two-stage"})

    return candidates


# ============================
# JUDGE + ACCEPT/REVISE
# ============================

def judge_top_candidates(question: str, fixed_candidates: list, top_m: int = TOP_M_FOR_JUDGE):
    # Guard: nothing to judge
    if not fixed_candidates:
        return {"fixed": {"selected_filter": {}, "selected_column": {}}, "audit": {}, "source": None}

    # Rank by heuristic score
    scored = [(score_candidate(question, c["fixed"]), c) for c in fixed_candidates]
    scored.sort(key=lambda x: x[0], reverse=True)
    shortlisted = [c for _, c in scored[:top_m]] or [scored[0][1]]

    # Judge calls in parallel
    futs, prompts = [], []
    for c in shortlisted:
        p = build_judge_prompt(question, c["fixed"])
        prompts.append(c)
        futs.append(EXECUTOR.submit(call_model_with_retries, p))

    judged = []
    for c, fut in zip(prompts, futs):
        raw = fut.result()
        # ⬇️ Coerce list/dict/junk into a dict with the keys we care about
        out = _as_dict(_extract_json_safe(raw), preferred_keys=("selected_filter","selected_column"))
        if not out:
            out = c["fixed"]  # fallback to the candidate if judge returned junk
        fixed2, audit2 = validate_and_fix(out)
        judged.append({"fixed": fixed2, "audit": audit2, "source": c})

    # Rescore and pick winner
    rescored = [(score_candidate(question, j["fixed"]), j) for j in judged] or [(0.0, {"fixed": shortlisted[0]["fixed"], "audit": {}, "source": shortlisted[0]})]
    rescored.sort(key=lambda x: x[0], reverse=True)
    return rescored[0][1]



def accept_revise_pass(question: str, winner_fixed: dict):
    # Build prompt and call model (cache+retries already inside call_model_with_retries)
    p = build_accept_revise_prompt(question, winner_fixed)
    raw = call_model_with_retries(p)

    # Parse robustly: handle dicts, lists, or junk
    obj = _as_dict(_extract_json_safe(raw), preferred_keys=("status","selected_filter","selected_column"))

    # Normalize fields
    status = str(obj.get("status") or "accept").strip().lower()
    if status not in ("accept", "revise"):
        status = "accept"

    sel_f = obj.get("selected_filter")
    sel_c = obj.get("selected_column")
    if not isinstance(sel_f, dict): sel_f = {}
    if not isinstance(sel_c, dict): sel_c = {}

    # Merge: model’s edits override only what it provided; keep the rest from winner
    merged = {
        "selected_filter": (sel_f or winner_fixed.get("selected_filter", {})),
        "selected_column": (sel_c or winner_fixed.get("selected_column", {})),
    }

    # Final deterministic validation
    fixed, audit = validate_and_fix(merged)

    # Record what happened for troubleshooting/restarts
    audit = audit or {}
    audit["accept_revise_status"] = status
    audit["accept_revise_raw"] = (raw[:2000] if isinstance(raw, str) else str(raw))

    return fixed, audit


# ============================
# MAIN
# ============================

def main():
    t_all0 = time.time()
    df = pd.read_excel(EXCEL_PATH_IN, sheet_name=SHEET_NAME)

    # Auto-detect question column
    global QUESTION_COLUMN
    if QUESTION_COLUMN not in df.columns:
        candidates = [c for c in df.columns if str(c).strip().lower() in {"question","query","prompt"}]
        if candidates:
            QUESTION_COLUMN = candidates[0]
        else:
            raise ValueError(f"Couldn't find a question column. Available: {list(df.columns)}")

    results = []
    timings = []

    for idx, row in df.head(MAX_QUESTIONS).iterrows():
        q = str(row[QUESTION_COLUMN]).strip()
        if not q or q.lower() == "nan":
            continue

        t0 = time.time()
        # 1) Generate candidates
        t_gen0 = time.time()
        candidates = gen_candidates_for_question(q)
        t_gen = time.time() - t_gen0

        # 2) Judge top candidates
        t_judge0 = time.time()
        winner = judge_top_candidates(q, candidates, top_m=TOP_M_FOR_JUDGE)
        t_judge = time.time() - t_judge0

        # 3) Accept/Revise on winner
        t_final0 = time.time()
        final_fixed, final_audit = accept_revise_pass(q, winner["fixed"])
        t_final = time.time() - t_final0

        total_t = time.time() - t0

        results.append({
            "row_index": idx,
            "question": q,
            "final_json": json.dumps(final_fixed, ensure_ascii=False),
            "audit_json": json.dumps(final_audit, ensure_ascii=False),
        })
        timings.append({
            "row_index": idx,
            "question": q,
            "t_generate_sec": t_gen,
            "t_judge_sec": t_judge,
            "t_accept_revise_sec": t_final,
            "t_total_sec": total_t,
        })
        print(f"[Pipeline 4] Row {idx} done. Total {total_t:.2f}s")

    results_df = pd.DataFrame(results)
    timings_df = pd.DataFrame(timings)

    with pd.ExcelWriter(EXCEL_PATH_OUT, engine="openpyxl") as writer:
        results_df.to_excel(writer, index=False, sheet_name="results")
        timings_df.to_excel(writer, index=False, sheet_name="timings")

    print(f"Saved results to: {EXCEL_PATH_OUT} (elapsed {time.time()-t_all0:.2f}s)")

if __name__ == "__main__":
    main()



[Pipeline 4] Row 0 done. Total 5.33s
[Pipeline 4] Row 1 done. Total 5.75s
[Pipeline 4] Row 2 done. Total 5.66s
[Pipeline 4] Row 3 done. Total 6.15s
[Pipeline 4] Row 4 done. Total 5.74s
[Pipeline 4] Row 5 done. Total 5.58s
[Pipeline 4] Row 6 done. Total 4.54s
[Pipeline 4] Row 7 done. Total 4.47s
[Pipeline 4] Row 8 done. Total 4.58s
[Pipeline 4] Row 9 done. Total 6.24s
[Pipeline 4] Row 10 done. Total 5.60s
[Pipeline 4] Row 11 done. Total 5.21s
[Pipeline 4] Row 12 done. Total 4.52s
[Pipeline 4] Row 13 done. Total 4.85s
[Pipeline 4] Row 14 done. Total 5.20s
[Pipeline 4] Row 15 done. Total 5.10s
[Pipeline 4] Row 16 done. Total 5.11s
[Pipeline 4] Row 17 done. Total 5.99s
[Pipeline 4] Row 18 done. Total 5.32s
[Pipeline 4] Row 19 done. Total 5.10s
[Pipeline 4] Row 20 done. Total 6.04s
[Pipeline 4] Row 21 done. Total 5.85s
[Pipeline 4] Row 22 done. Total 5.10s
[Pipeline 4] Row 23 done. Total 4.89s
[Pipeline 4] Row 24 done. Total 5.51s
[Pipeline 4] Row 25 done. Total 5.17s
[Pipeline 4] Row 26 do

In [27]:
"""
Run this cell to produce a table to compare ground truth results with the 5 runs from Gemini. ANY PIPELINE in `runs/`
"""

import pandas as pd, re, json, os, glob
from datetime import datetime

# Find ALL matching pipeline files in runs/
candidates = sorted(glob.glob('runs/query-chosen-filters-columns_with_runs_pipeline*.xlsx'), key=os.path.getmtime, reverse=True)
if not candidates:
    raise FileNotFoundError("No files matching 'runs/*_with_runs_pipeline*.xlsx' were found.")

# Helper: extract only the JSON object from a messy string (code fences, explanations, etc.)
def extract_json(text):
    if not isinstance(text, str):
        return text
    # Strip code fence markers if present
    cleaned = text.replace("```json", "").replace("```", "").strip()
    # Try to isolate the JSON block
    m = re.search(r'\{[\s\S]*\}', cleaned)
    cleaned = m.group(0).strip() if m else cleaned.strip()
    # Normalize smart quotes
    cleaned = cleaned.replace("“", '"').replace("”", '"').replace("’", "'")
    # Try to pretty-format valid JSON; if it fails, just return the cleaned block
    try:
        return json.dumps(json.loads(cleaned), indent=2, ensure_ascii=False)
    except Exception:
        return cleaned

processed = []
for file_path in candidates:
    # Read the sheet named 'results' if it exists, otherwise fall back to the first sensible sheet
    try:
        data = pd.read_excel(file_path, sheet_name='results')
        used_sheet = 'results'
    except ValueError:
        with pd.ExcelFile(file_path) as xls:
            sheet_names = xls.sheet_names
            preferred = next((s for s in sheet_names if re.search(r'(result|run)', s, re.I)), sheet_names[0])
            data = pd.read_excel(xls, sheet_name=preferred)
            used_sheet = preferred

    # Columns to process (keep as-is, but only those present)
    json_columns = ['Ground Truth JSON', 'run_1', 'run_2', 'run_3', 'run_4', 'run_5']
    json_columns = [c for c in json_columns if c in data.columns]

    # Extract/clean JSON for each run column
    for col in json_columns:
        data[col] = data[col].apply(extract_json)

    # Save cleaned version to corresponding pipeline number
    m = re.search(r'pipeline(\d+)', file_path, re.IGNORECASE)
    pipeline_num = m.group(1) if m else 'X'
    output_path = f'runs/gt-vs-runs_pipeline{pipeline_num}.xlsx'
    data.to_excel(output_path, index=False)

    print(f"Source file: {file_path}")
    print(f"Sheet used: {used_sheet}")
    print(f"Saved cleaned file as {output_path}\n")
    processed.append(output_path)

# Optional summary
print("Done. Files written:")
for p in processed:
    print(" -", p)


Source file: runs\query-chosen-filters-columns_with_runs_pipeline4_base.xlsx
Sheet used: results
Saved cleaned file as runs/gt-vs-runs_pipeline4.xlsx

Done. Files written:
 - runs/gt-vs-runs_pipeline4.xlsx


In [ ]:
# """
# Run this cell to produce a table to compare ground truth results with the 5 runs from Gemini. FOR PIPELINE 1
# """


# import pandas as pd, re, json
# from datetime import datetime

# # Load the Excel file
# file_path = 'runs/query-chosen-filters-columns_with_runs_pipeline1.xlsx'  # adjust if needed
# data = pd.read_excel(file_path, sheet_name='results')

# # Helper: extract only the JSON object from a messy string (code fences, explanations, etc.)
# def extract_json(text):
#     if not isinstance(text, str):
#         return text
#     # Strip code fence markers if present
#     cleaned = text.replace("```json", "").replace("```", "").strip()
#     # Try to isolate the JSON block
#     m = re.search(r'\{[\s\S]*\}', cleaned)
#     cleaned = m.group(0).strip() if m else cleaned.strip()
#     # Normalize smart quotes
#     cleaned = cleaned.replace("“", '"').replace("”", '"').replace("’", "'")
#     # Try to pretty-format valid JSON; if it fails, just return the cleaned block
#     try:
#         return json.dumps(json.loads(cleaned), indent=2, ensure_ascii=False)
#     except Exception:
#         return cleaned

# # Columns to process
# json_columns = ['Ground Truth JSON', 'run_1', 'run_2', 'run_3', 'run_4', 'run_5']
# json_columns = [c for c in json_columns if c in data.columns]  # handle missing safely

# # Extract/clean JSON for each run column
# for col in json_columns:
#     data[col] = data[col].apply(extract_json)

# # Save cleaned version
# output_path = 'gt-vs-runs_pipeline1.xlsx'
# data.to_excel(output_path, index=False)

# print(f"Saved cleaned file as {output_path}")


In [ ]:
# """
# Run this cell to produce a table to compare ground truth results with the 5 runs from Gemini. FOR PIPELINE 2
# """


# import pandas as pd, re, json
# from datetime import datetime

# # Load the Excel file
# file_path = 'runs/query-chosen-filters-columns_with_runs_pipeline2.xlsx'  # adjust if needed
# data = pd.read_excel(file_path, sheet_name='results')

# # Helper: extract only the JSON object from a messy string (code fences, explanations, etc.)
# def extract_json(text):
#     if not isinstance(text, str):
#         return text
#     # Strip code fence markers if present
#     cleaned = text.replace("```json", "").replace("```", "").strip()
#     # Try to isolate the JSON block
#     m = re.search(r'\{[\s\S]*\}', cleaned)
#     cleaned = m.group(0).strip() if m else cleaned.strip()
#     # Normalize smart quotes
#     cleaned = cleaned.replace("“", '"').replace("”", '"').replace("’", "'")
#     # Try to pretty-format valid JSON; if it fails, just return the cleaned block
#     try:
#         return json.dumps(json.loads(cleaned), indent=2, ensure_ascii=False)
#     except Exception:
#         return cleaned

# # Columns to process
# json_columns = ['Ground Truth JSON', 'run_1', 'run_2', 'run_3', 'run_4', 'run_5']
# json_columns = [c for c in json_columns if c in data.columns]  # handle missing safely

# # Extract/clean JSON for each run column
# for col in json_columns:
#     data[col] = data[col].apply(extract_json)

# # Save cleaned version
# output_path = 'gt-vs-runs_pipeline2.xlsx'
# data.to_excel(output_path, index=False)

# print(f"Saved cleaned file as {output_path}")


In [ ]:
!pip install xlsxwriter


In [28]:
"""
This is the cell to run when we wish to get precision and recall values
for filter and column selection (STRICT ONLY, no canonicalization/relaxed scoring).
PROCESS **EVERY** 'gt-vs-runs_pipeline*.xlsx' file in the current folder.
"""

import pandas as pd, json, re, os, glob
from datetime import datetime

# --- Helpers: detect columns/sheets and parse JSON ---

def detect_gt_col(df):
    """Find the ground-truth column (prefer exact 'Ground Truth JSON')."""
    for c in df.columns:
        if str(c).strip().lower() == "ground truth json":
            return c
    for c in df.columns:
        lc = str(c).lower()
        if "ground" in lc and "json" in lc:
            return c
    raise KeyError("Could not find a 'Ground Truth JSON' column (or similar) in the file.")

def extract_json_block(text):
    if not isinstance(text, str):
        return ""
    t = text.replace("```json","").replace("```","").strip()
    m = re.search(r'\{[\s\S]*\}', t)
    return m.group(0).strip() if m else t

def parse_obj(text):
    if not isinstance(text, str) or not text.strip():
        return {}
    blob = extract_json_block(text)
    try:
        return json.loads(blob)
    except Exception:
        blob2 = re.sub(r',\s*([}\]])', r'\1', blob)  # tolerate trailing commas
        try:
            return json.loads(blob2)
        except Exception:
            return {}

def parse_spec_strict(text):
    """Return (filters_set, columns_set) with STRICT matching (trim only)."""
    obj = parse_obj(text)
    filt = obj.get('selected_filter', {}) or {}
    cols = obj.get('selected_column', {}) or {}

    filt_set = set()
    for k, v in filt.items():
        k1 = str(k).strip()
        v1 = str(v).strip()
        if k1 and v1 and v1.lower() != "all":
            filt_set.add((k1, v1))

    col_set = set()
    for v in cols.values():
        lab = str(v).strip()
        if lab:
            col_set.add(lab)

    return filt_set, col_set

def prf_strict(gt_set, pred_set):
    tp = len(gt_set & pred_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)
    prec = tp / (tp + fp) if (tp + fp) else (1.0 if not gt_set and not pred_set else 0.0)
    rec  = tp / (tp + fn) if (tp + fn) else 1.0
    return tp, fp, fn, prec, rec

# --- Find every cleaned 'gt-vs-runs' file and score each one ---
os.makedirs("results", exist_ok=True)

files = sorted(glob.glob('gt-vs-runs_pipeline*_gemini25.xlsx'))
if not files:
    raise FileNotFoundError("No files matching 'gt-vs-runs_pipeline*.xlsx' were found in the current directory.")

for src in files:
    # Read first sheet (cleaners wrote a single sheet)
    df = pd.read_excel(src, sheet_name=0)

    # **NEW**: Drop any pre-existing metrics (including relaxed) from upstream files
    drop_cols = [c for c in df.columns
                 if re.search(r'(^row_id$)|(^avg_)|(^precision_)|(^recall_)|relaxed', str(c), re.I)]
    df = df.drop(columns=drop_cols, errors='ignore')

    # Identify run columns present (run_1..run_N)
    run_cols = [c for c in df.columns if re.fullmatch(r'run_\d+', str(c))]
    run_cols = sorted(run_cols, key=lambda x: int(x.split('_')[1]))  # numeric order
    if not run_cols:
        print(f"[WARN] No 'run_*' columns found in {src}; skipping.")
        continue

    # Identify GT column
    GT_COL = detect_gt_col(df)

    # --- Per-row, per-run metrics (filters & columns STRICT only) ---
    rows = []
    for idx, row in df.iterrows():
        gt_filters, gt_cols = parse_spec_strict(row[GT_COL])

        for r in run_cols:
            run_filters, run_cols_set = parse_spec_strict(row[r])

            _, _, _, pF, rF = prf_strict(gt_filters, run_filters)
            _, _, _, pC, rC = prf_strict(gt_cols, run_cols_set)

            rows.append({
                'row_id': idx,
                'run': r,
                'precision_filters': pF, 'recall_filters': rF,
                'precision_columns': pC, 'recall_columns': rC,
            })

    per_run = pd.DataFrame(rows)

    # --- Averages across the runs, per query ---
    avg_per_query = per_run.groupby('row_id').agg(
        avg_precision_filters=('precision_filters', 'mean'),
        avg_recall_filters=('recall_filters', 'mean'),
        avg_precision_columns=('precision_columns', 'mean'),
        avg_recall_columns=('recall_columns', 'mean'),
    ).reset_index()

    # --- Merge back; keep only the new strict averages ---
    df_with_avgs = df.merge(avg_per_query, left_index=True, right_on='row_id', suffixes=('', '_old'))
    # If anything still snuck in from older runs, drop it
    df_with_avgs = df_with_avgs[[c for c in df_with_avgs.columns if not re.search(r'_old$|relaxed', c, re.I)]]

    # pipeline number from filename
    m = re.search(r'pipeline(\d+)', src, re.IGNORECASE)
    pipeline_num = m.group(1) if m else 'X'

    out_path = f'results/gt-vs-runs-with-filter-column-averages_pipeline{pipeline_num}_gemini25.xlsx'
    with pd.ExcelWriter(out_path, engine='xlsxwriter') as xw:
        df_with_avgs.to_excel(xw, sheet_name='queries_with_averages', index=False)
        per_run.to_excel(xw, sheet_name='per_row_per_run', index=False)

    print(f"Source: {src}")
    print("Saved:", out_path)


Source: gt-vs-runs_pipeline1_gemini25.xlsx
Saved: results/gt-vs-runs-with-filter-column-averages_pipeline1_gemini25.xlsx
Source: gt-vs-runs_pipeline2_gemini25.xlsx
Saved: results/gt-vs-runs-with-filter-column-averages_pipeline2_gemini25.xlsx
Source: gt-vs-runs_pipeline3_gemini25.xlsx
Saved: results/gt-vs-runs-with-filter-column-averages_pipeline3_gemini25.xlsx
[WARN] No 'run_*' columns found in gt-vs-runs_pipeline4_gemini25.xlsx; skipping.


In [31]:
import os, json, re
import pandas as pd

GT_PATH = "runs/query-chosen-filters-columns.xlsx"
PIPE4_PATH = "runs/gt-vs-runs_pipeline4.xlsx"
OUT_DIR = "results"
OUT_PATH = os.path.join(OUT_DIR, "gt-vs-runs-with-filter-column-averages_pipeline4_base.xlsx")

os.makedirs(OUT_DIR, exist_ok=True)

def extract_json_block(text: str) -> str:
    if not isinstance(text, str):
        return ""
    t = text.replace("```json", "").replace("```", "").strip()
    m = re.search(r"\{[\s\S]*\}", t)
    return m.group(0).strip() if m else t

def parse_obj(text: str):
    if not isinstance(text, str) or not text.strip():
        return {}
    blob = extract_json_block(text)
    try:
        return json.loads(blob)
    except Exception:
        blob2 = re.sub(r',\s*([}\]])', r'\1', blob)  # tolerate trailing commas
        try:
            return json.loads(blob2)
        except Exception:
            return {}

def parse_spec_strict(text: str):
    """Return (filters_set, columns_set) with STRICT matching (trim only)."""
    obj = parse_obj(text)
    filt = obj.get('selected_filter', {}) or {}
    cols = obj.get('selected_column', {}) or {}

    filt_set = set()
    for k, v in filt.items():
        k1 = str(k).strip()
        v1 = str(v).strip()
        if k1 and v1 and v1.lower() != "all":
            filt_set.add((k1, v1))

    col_set = set()
    if isinstance(cols, dict):
        for v in cols.values():
            lab = str(v).strip()
            if lab:
                col_set.add(lab)

    return filt_set, col_set

def prf_strict(gt_set, pred_set):
    tp = len(gt_set & pred_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)
    prec = tp / (tp + fp) if (tp + fp) else (1.0 if not gt_set and not pred_set else 0.0)
    rec  = tp / (tp + fn) if (tp + fn) else 1.0
    return tp, fp, fn, prec, rec

# Load GT & Pipeline 4
gt_df = pd.read_excel(GT_PATH, sheet_name=0)
p4_df = pd.read_excel(PIPE4_PATH, sheet_name="Sheet1")

# Detect columns
gt_query_col = None
for c in gt_df.columns:
    if str(c).strip().lower() in {"query","question","prompt"}:
        gt_query_col = c; break
if gt_query_col is None:
    raise KeyError(f"Could not find a question column in GT. Available: {list(gt_df.columns)}")

gt_json_col = None
for c in gt_df.columns:
    if str(c).strip().lower() == "ground truth json":
        gt_json_col = c; break
if gt_json_col is None:
    for c in gt_df.columns:
        if "ground" in str(c).lower() and "json" in str(c).lower():
            gt_json_col = c; break
if gt_json_col is None:
    raise KeyError(f"Could not find a 'Ground Truth JSON' column in GT. Available: {list(gt_df.columns)}")

# Align by row_index when available, otherwise by position
if "row_index" in p4_df.columns:
    p4_df["row_index"] = p4_df["row_index"].astype(int)
    gt_df = gt_df.reset_index().rename(columns={"index":"row_index"})
    merged = pd.merge(p4_df, gt_df[["row_index", gt_query_col, gt_json_col]], on="row_index", how="left")
else:
    gt_df = gt_df.reset_index().rename(columns={"index":"row_index"})
    p4_df = p4_df.reset_index().rename(columns={"index":"row_index"})
    merged = pd.merge(p4_df, gt_df[["row_index", gt_query_col, gt_json_col]], on="row_index", how="left")

# Compute per-row metrics
rows = []
for _, r in merged.iterrows():
    gt_filters, gt_cols = parse_spec_strict(r[gt_json_col])
    run_filters, run_cols = parse_spec_strict(r.get("final_json", ""))

    _, _, _, pF, rF = prf_strict(gt_filters, run_filters)
    _, _, _, pC, rC = prf_strict(gt_cols, run_cols)

    rows.append({
        "row_index": int(r["row_index"]),
        "question_gt": r[gt_query_col],
        "question_run": r.get("question",""),
        "precision_filters": pF, "recall_filters": rF,
        "precision_columns": pC, "recall_columns": rC,
        "gt_filter_count": len(gt_filters), "pred_filter_count": len(run_filters),
        "gt_column_count": len(gt_cols), "pred_column_count": len(run_cols),
        "final_json": r.get("final_json",""),
        "audit_json": r.get("audit_json",""),
    })

per_row = pd.DataFrame(rows).sort_values("row_index").reset_index(drop=True)

summary = pd.DataFrame([{
    "avg_precision_filters": per_row["precision_filters"].mean(),
    "avg_recall_filters": per_row["recall_filters"].mean(),
    "avg_precision_columns": per_row["precision_columns"].mean(),
    "avg_recall_columns": per_row["recall_columns"].mean(),
    "n_evaluated": len(per_row),
}])

with pd.ExcelWriter(OUT_PATH, engine="xlsxwriter") as xw:
    per_row.to_excel(xw, index=False, sheet_name="per_row_strict")
    summary.to_excel(xw, index=False, sheet_name="summary_strict")

print(f"Saved: {OUT_PATH}")

Saved: results\gt-vs-runs-with-filter-column-averages_pipeline4_base.xlsx


In [ ]:
# """
# This is the cell to run when we wish to get precision and recall values
# for filter and column selection (with canonicalization + relaxed scoring).
# SCORE FOR PIPELINE 1 SPECIFICALLY.
# """

# import pandas as pd, json, re, difflib
# from datetime import datetime
# from pathlib import Path

# # --- Load cleaned JSONs ---
# df = pd.read_excel('gt-vs-runs_pipeline1.xlsx')  # must contain: Ground Truth JSON, run_1..run_5
# RUN_COLS = [c for c in ['run_1','run_2','run_3','run_4','run_5'] if c in df.columns]
# GT_COL = 'Ground Truth JSON'


# COL_CONCISE_PATH = Path("definitions_folder/definitions - aim2 - column concise.txt")
# CANON_COLS = []
# if COL_CONCISE_PATH.exists():
#     for line in COL_CONCISE_PATH.read_text(encoding="utf-8").splitlines():
#         # split on common dash separators used in the file
#         label = re.split(r"\s+[—–-]\s+", line.strip(), maxsplit=1)[0]
#         label = label.strip()
#         if label and not label.startswith("#"):
#             CANON_COLS.append(label)
# CANON_COLS_NORM = {label.lower(): label for label in CANON_COLS}


# CATEGORY_ALIASES = {
#     'ici class': 'ici class',
#     'class of ici': 'ici class',
#     'cancer type': 'cancer type',
#     'monotherapy/combination': 'type of therapy',
#     'type of therapy': 'type of therapy',
# }

# # Value aliases keyed by (canonical_category, normalized_value)
# VALUE_ALIASES = {
#     ('ici class','pd1'): 'pd1',
#     ('ici class','pd-1'): 'pd1',
#     ('ici class','pd 1'): 'pd1',
#     ('ici class','pd1 inhibitor'): 'pd1',
#     ('ici class','pd-l1'): 'pd-l1',
#     ('ici class','pd l1'): 'pd-l1',
#     ('ici class','pd-l1 inhibitor'): 'pd-l1',
#     ('ici class','ctla-4'): 'ctla-4',
#     ('ici class','ctla 4'): 'ctla-4',

#     ('cancer type','nsclc'): 'non-small cell lung cancer',
#     ('cancer type','non small cell lung cancer'): 'non-small cell lung cancer',
#     ('cancer type','non-small cell lung cancer'): 'non-small cell lung cancer',
#     # add more tumor-type abbrev mappings here if needed

#     ('type of therapy','combination'): 'combination therapy',
#     ('type of therapy','combo'): 'combination therapy',
#     ('type of therapy','combination therapy'): 'combination therapy',
#     ('type of therapy','monotherapy'): 'monotherapy',
# }

# # --- Helpers ---
# def norm_text(s):
#     if s is None: return ""
#     s = str(s)
#     s = s.replace("“","\"").replace("”","\"").replace("’","'")
#     s = re.sub(r'\s+', ' ', s).strip().lower()
#     return s

# def strip_parens(s):
#     return re.sub(r'\s*\([^)]*\)\s*', ' ', s).strip()

# def extract_json_block(text):
#     if not isinstance(text, str): return ""
#     t = text.replace("```json","").replace("```","").strip()
#     m = re.search(r'\{[\s\S]*\}', t)
#     return m.group(0).strip() if m else t

# def parse_obj(text):
#     if not isinstance(text, str) or not text.strip():
#         return {}
#     blob = extract_json_block(text)
#     try:
#         return json.loads(blob)
#     except Exception:
#         blob2 = re.sub(r',\s*([}\]])', r'\1', blob)
#         try:
#             return json.loads(blob2)
#         except Exception:
#             return {}

# def canonicalize_filter_pair(k, v):
#     k0 = norm_text(k)
#     k0 = CATEGORY_ALIASES.get(k0, k0)          # category alias
#     v0 = norm_text(strip_parens(v))
#     v0 = VALUE_ALIASES.get((k0, v0), v0)       # value alias within category
#     # final tidy (common punctuation/dash variants)
#     v0 = v0.replace('–','-').replace('—','-')
#     return k0, v0

# def snap_to_canonical_col(label):
#     if not isinstance(label, str): return ""
#     raw = label.strip()
#     # keep only the part before dash if it looks like "Label — Definition"
#     left = re.split(r"\s+[—–-]\s+", raw, maxsplit=1)[0].strip()
#     candidate = norm_text(left)

#     if candidate in CANON_COLS_NORM:
#         return CANON_COLS_NORM[candidate]

#     if CANON_COLS:
#         best = max(CANON_COLS, key=lambda c: difflib.SequenceMatcher(None, candidate, c.lower()).ratio())
#         score = difflib.SequenceMatcher(None, candidate, best.lower()).ratio()
#         if score >= 0.92:    # high-confidence snap
#             return best
#     return left  # fallback to the cleaned left part

# def parse_spec(text):
#     obj = parse_obj(text)
#     filt = obj.get('selected_filter', {}) or {}
#     cols = obj.get('selected_column', {}) or {}

#     filt_set = set()
#     for k, v in filt.items():
#         k1, v1 = canonicalize_filter_pair(k, v)
#         if k1 and v1 and v1 != "all":
#             filt_set.add((k1, v1))

#     col_set = set()
#     for v in cols.values():
#         lab = snap_to_canonical_col(v)
#         lab = norm_text(lab)
#         if lab:
#             col_set.add(lab)

#     return filt_set, col_set

# def prf_strict(gt_set, pred_set):
#     tp = len(gt_set & pred_set)
#     fp = len(pred_set - gt_set)
#     fn = len(gt_set - pred_set)
#     prec = tp / (tp + fp) if (tp + fp) else (1.0 if not gt_set and not pred_set else 0.0)
#     rec  = tp / (tp + fn) if (tp + fn) else 1.0
#     return tp, fp, fn, prec, rec

# def similar(a, b):
#     ta = set(re.findall(r'\w+', a.lower()))
#     tb = set(re.findall(r'\w+', b.lower()))
#     if not ta and not tb: return 1.0
#     return len(ta & tb) / len(ta | tb)

# def prf_relaxed(gt_set, pred_set, partial_credit=0.5, thresh_full=0.9, thresh_partial=0.75):

#     # Build maps for pairwise best matches
#     gt_unused = set(gt_set)
#     pred_unused = set(pred_set)
#     exact_tp = len(gt_set & pred_set)

#     # remove exacts first
#     gt_unused -= (gt_set & pred_set)
#     pred_unused -= (gt_set & pred_set)

#     partial_tp = 0.0
#     # greedy match remaining by similarity
#     for p in list(pred_unused):
#         best = None; best_sim = 0.0
#         for g in gt_unused:
#             # Compare key and value separately for filters; single string for columns
#             if isinstance(p, tuple):
#                 sim = 0.5*similar(p[0], g[0]) + 0.5*similar(p[1], g[1])
#             else:
#                 sim = similar(p, g)
#             if sim > best_sim:
#                 best_sim, best = sim, g
#         if best is not None:
#             if best_sim >= thresh_full:
#                 exact_tp += 1
#                 gt_unused.remove(best)
#             elif best_sim >= thresh_partial:
#                 partial_tp += partial_credit
#                 gt_unused.remove(best)

#     tp = exact_tp + partial_tp
#     fp = max(0.0, len(pred_set) - (exact_tp + partial_tp))
#     fn = max(0.0, len(gt_set) - (exact_tp + partial_tp))
#     prec = tp / (tp + fp) if (tp + fp) else (1.0 if not gt_set and not pred_set else 0.0)
#     rec  = tp / (tp + fn) if (tp + fn) else 1.0
#     return tp, fp, fn, prec, rec

# # --- Per-row, per-run metrics (filters & columns separately) ---
# rows = []
# for idx, row in df.iterrows():
#     gt_filters, gt_cols = parse_spec(row[GT_COL])
#     for r in RUN_COLS:
#         run_filters, run_cols = parse_spec(row[r])

#         # strict
#         _, _, _, pF, rF = prf_strict(gt_filters, run_filters)
#         _, _, _, pC, rC = prf_strict(gt_cols, run_cols)

#         # relaxed (diagnostic)
#         _, _, _, pF_rel, rF_rel = prf_relaxed(gt_filters, run_filters)
#         _, _, _, pC_rel, rC_rel = prf_relaxed(gt_cols, run_cols)

#         rows.append({
#             'row_id': idx,
#             'run': r,
#             'precision_filters': pF, 'recall_filters': rF,
#             'precision_columns': pC, 'recall_columns': rC,
#             'precision_filters_relaxed': pF_rel, 'recall_filters_relaxed': rF_rel,
#             'precision_columns_relaxed': pC_rel, 'recall_columns_relaxed': rC_rel,
#         })

# per_run = pd.DataFrame(rows)

# # --- Averages across the runs, per query ---
# avg_per_query = per_run.groupby('row_id').agg(
#     avg_precision_filters=('precision_filters', 'mean'),
#     avg_recall_filters=('recall_filters', 'mean'),
#     avg_precision_columns=('precision_columns', 'mean'),
#     avg_recall_columns=('recall_columns', 'mean'),
#     avg_precision_filters_relaxed=('precision_filters_relaxed', 'mean'),
#     avg_recall_filters_relaxed=('recall_filters_relaxed', 'mean'),
#     avg_precision_columns_relaxed=('precision_columns_relaxed', 'mean'),
#     avg_recall_columns_relaxed=('recall_columns_relaxed', 'mean'),
# ).reset_index()

# # --- Merge back to original dataframe so each query has its averages ---
# df_with_avgs = df.merge(avg_per_query, left_index=True, right_on='row_id')

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# # --- Save both detailed and summary outputs ---
# out_path = f'results/gt-vs-runs-with-filter-column-averages_pipeline1.xlsx'
# with pd.ExcelWriter(out_path, engine='xlsxwriter') as xw:
#     df_with_avgs.to_excel(xw, sheet_name='queries_with_averages', index=False)
#     per_run.to_excel(xw, sheet_name='per_row_per_run', index=False)

# print("Saved:", out_path)


In [ ]:
# """
# This is the cell to run when we wish to get precision and recall values
# for filter and column selection (with canonicalization + relaxed scoring).
# SCORE FOR PIPELINE 2 SPECIFICALLY.
# """

# import pandas as pd, json, re, difflib
# from datetime import datetime
# from pathlib import Path

# # --- Load cleaned JSONs ---
# df = pd.read_excel('gt-vs-runs_pipeline2.xlsx')  # must contain: Ground Truth JSON, run_1..run_5
# RUN_COLS = [c for c in ['run_1','run_2','run_3','run_4','run_5'] if c in df.columns]
# GT_COL = 'Ground Truth JSON'


# COL_CONCISE_PATH = Path("definitions_folder/definitions - aim2 - column concise.txt")
# CANON_COLS = []
# if COL_CONCISE_PATH.exists():
#     for line in COL_CONCISE_PATH.read_text(encoding="utf-8").splitlines():
#         # split on common dash separators used in the file
#         label = re.split(r"\s+[—–-]\s+", line.strip(), maxsplit=1)[0]
#         label = label.strip()
#         if label and not label.startswith("#"):
#             CANON_COLS.append(label)
# CANON_COLS_NORM = {label.lower(): label for label in CANON_COLS}


# CATEGORY_ALIASES = {
#     'ici class': 'ici class',
#     'class of ici': 'ici class',
#     'cancer type': 'cancer type',
#     'monotherapy/combination': 'type of therapy',
#     'type of therapy': 'type of therapy',
# }

# # Value aliases keyed by (canonical_category, normalized_value)
# VALUE_ALIASES = {
#     ('ici class','pd1'): 'pd1',
#     ('ici class','pd-1'): 'pd1',
#     ('ici class','pd 1'): 'pd1',
#     ('ici class','pd1 inhibitor'): 'pd1',
#     ('ici class','pd-l1'): 'pd-l1',
#     ('ici class','pd l1'): 'pd-l1',
#     ('ici class','pd-l1 inhibitor'): 'pd-l1',
#     ('ici class','ctla-4'): 'ctla-4',
#     ('ici class','ctla 4'): 'ctla-4',

#     ('cancer type','nsclc'): 'non-small cell lung cancer',
#     ('cancer type','non small cell lung cancer'): 'non-small cell lung cancer',
#     ('cancer type','non-small cell lung cancer'): 'non-small cell lung cancer',
#     # add more tumor-type abbrev mappings here if needed

#     ('type of therapy','combination'): 'combination therapy',
#     ('type of therapy','combo'): 'combination therapy',
#     ('type of therapy','combination therapy'): 'combination therapy',
#     ('type of therapy','monotherapy'): 'monotherapy',
# }

# # --- Helpers ---
# def norm_text(s):
#     if s is None: return ""
#     s = str(s)
#     s = s.replace("“","\"").replace("”","\"").replace("’","'")
#     s = re.sub(r'\s+', ' ', s).strip().lower()
#     return s

# def strip_parens(s):
#     return re.sub(r'\s*\([^)]*\)\s*', ' ', s).strip()

# def extract_json_block(text):
#     if not isinstance(text, str): return ""
#     t = text.replace("```json","").replace("```","").strip()
#     m = re.search(r'\{[\s\S]*\}', t)
#     return m.group(0).strip() if m else t

# def parse_obj(text):
#     if not isinstance(text, str) or not text.strip():
#         return {}
#     blob = extract_json_block(text)
#     try:
#         return json.loads(blob)
#     except Exception:
#         blob2 = re.sub(r',\s*([}\]])', r'\1', blob)
#         try:
#             return json.loads(blob2)
#         except Exception:
#             return {}

# def canonicalize_filter_pair(k, v):
#     k0 = norm_text(k)
#     k0 = CATEGORY_ALIASES.get(k0, k0)          # category alias
#     v0 = norm_text(strip_parens(v))
#     v0 = VALUE_ALIASES.get((k0, v0), v0)       # value alias within category
#     # final tidy (common punctuation/dash variants)
#     v0 = v0.replace('–','-').replace('—','-')
#     return k0, v0

# def snap_to_canonical_col(label):
#     if not isinstance(label, str): return ""
#     raw = label.strip()
#     # keep only the part before dash if it looks like "Label — Definition"
#     left = re.split(r"\s+[—–-]\s+", raw, maxsplit=1)[0].strip()
#     candidate = norm_text(left)

#     if candidate in CANON_COLS_NORM:
#         return CANON_COLS_NORM[candidate]

#     if CANON_COLS:
#         best = max(CANON_COLS, key=lambda c: difflib.SequenceMatcher(None, candidate, c.lower()).ratio())
#         score = difflib.SequenceMatcher(None, candidate, best.lower()).ratio()
#         if score >= 0.92:    # high-confidence snap
#             return best
#     return left  # fallback to the cleaned left part

# def parse_spec(text):
#     obj = parse_obj(text)
#     filt = obj.get('selected_filter', {}) or {}
#     cols = obj.get('selected_column', {}) or {}

#     filt_set = set()
#     for k, v in filt.items():
#         k1, v1 = canonicalize_filter_pair(k, v)
#         if k1 and v1 and v1 != "all":
#             filt_set.add((k1, v1))

#     col_set = set()
#     for v in cols.values():
#         lab = snap_to_canonical_col(v)
#         lab = norm_text(lab)
#         if lab:
#             col_set.add(lab)

#     return filt_set, col_set

# def prf_strict(gt_set, pred_set):
#     tp = len(gt_set & pred_set)
#     fp = len(pred_set - gt_set)
#     fn = len(gt_set - pred_set)
#     prec = tp / (tp + fp) if (tp + fp) else (1.0 if not gt_set and not pred_set else 0.0)
#     rec  = tp / (tp + fn) if (tp + fn) else 1.0
#     return tp, fp, fn, prec, rec

# def similar(a, b):
#     ta = set(re.findall(r'\w+', a.lower()))
#     tb = set(re.findall(r'\w+', b.lower()))
#     if not ta and not tb: return 1.0
#     return len(ta & tb) / len(ta | tb)

# def prf_relaxed(gt_set, pred_set, partial_credit=0.5, thresh_full=0.9, thresh_partial=0.75):

#     # Build maps for pairwise best matches
#     gt_unused = set(gt_set)
#     pred_unused = set(pred_set)
#     exact_tp = len(gt_set & pred_set)

#     # remove exacts first
#     gt_unused -= (gt_set & pred_set)
#     pred_unused -= (gt_set & pred_set)

#     partial_tp = 0.0
#     # greedy match remaining by similarity
#     for p in list(pred_unused):
#         best = None; best_sim = 0.0
#         for g in gt_unused:
#             # Compare key and value separately for filters; single string for columns
#             if isinstance(p, tuple):
#                 sim = 0.5*similar(p[0], g[0]) + 0.5*similar(p[1], g[1])
#             else:
#                 sim = similar(p, g)
#             if sim > best_sim:
#                 best_sim, best = sim, g
#         if best is not None:
#             if best_sim >= thresh_full:
#                 exact_tp += 1
#                 gt_unused.remove(best)
#             elif best_sim >= thresh_partial:
#                 partial_tp += partial_credit
#                 gt_unused.remove(best)

#     tp = exact_tp + partial_tp
#     fp = max(0.0, len(pred_set) - (exact_tp + partial_tp))
#     fn = max(0.0, len(gt_set) - (exact_tp + partial_tp))
#     prec = tp / (tp + fp) if (tp + fp) else (1.0 if not gt_set and not pred_set else 0.0)
#     rec  = tp / (tp + fn) if (tp + fn) else 1.0
#     return tp, fp, fn, prec, rec

# # --- Per-row, per-run metrics (filters & columns separately) ---
# rows = []
# for idx, row in df.iterrows():
#     gt_filters, gt_cols = parse_spec(row[GT_COL])
#     for r in RUN_COLS:
#         run_filters, run_cols = parse_spec(row[r])

#         # strict
#         _, _, _, pF, rF = prf_strict(gt_filters, run_filters)
#         _, _, _, pC, rC = prf_strict(gt_cols, run_cols)

#         # relaxed (diagnostic)
#         _, _, _, pF_rel, rF_rel = prf_relaxed(gt_filters, run_filters)
#         _, _, _, pC_rel, rC_rel = prf_relaxed(gt_cols, run_cols)

#         rows.append({
#             'row_id': idx,
#             'run': r,
#             'precision_filters': pF, 'recall_filters': rF,
#             'precision_columns': pC, 'recall_columns': rC,
#             'precision_filters_relaxed': pF_rel, 'recall_filters_relaxed': rF_rel,
#             'precision_columns_relaxed': pC_rel, 'recall_columns_relaxed': rC_rel,
#         })

# per_run = pd.DataFrame(rows)

# # --- Averages across the runs, per query ---
# avg_per_query = per_run.groupby('row_id').agg(
#     avg_precision_filters=('precision_filters', 'mean'),
#     avg_recall_filters=('recall_filters', 'mean'),
#     avg_precision_columns=('precision_columns', 'mean'),
#     avg_recall_columns=('recall_columns', 'mean'),
#     avg_precision_filters_relaxed=('precision_filters_relaxed', 'mean'),
#     avg_recall_filters_relaxed=('recall_filters_relaxed', 'mean'),
#     avg_precision_columns_relaxed=('precision_columns_relaxed', 'mean'),
#     avg_recall_columns_relaxed=('recall_columns_relaxed', 'mean'),
# ).reset_index()

# # --- Merge back to original dataframe so each query has its averages ---
# df_with_avgs = df.merge(avg_per_query, left_index=True, right_on='row_id')

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# # --- Save both detailed and summary outputs ---
# out_path = f'results/gt-vs-runs-with-filter-column-averages_pipeline2.xlsx'
# with pd.ExcelWriter(out_path, engine='xlsxwriter') as xw:
#     df_with_avgs.to_excel(xw, sheet_name='queries_with_averages', index=False)
#     per_run.to_excel(xw, sheet_name='per_row_per_run', index=False)

# print("Saved:", out_path)


In [ ]:
import pandas as pd, re, glob, os
from pathlib import Path
from itertools import combinations
from xlsxwriter.utility import xl_col_to_name  # safer than chr(65+col)

METRICS = [
    "avg_precision_filters",
    "avg_recall_filters",
    "avg_precision_columns",
    "avg_recall_columns",
]

def load_queries_with_averages(path: str) -> pd.DataFrame:
    book = pd.read_excel(path, sheet_name=None)
    df = book["queries_with_averages"].copy()
    if "row_id" not in df.columns:
        df = df.reset_index().rename(columns={"index": "row_id"})
    if "Query" not in df.columns:
        df["Query"] = df["row_id"].astype(str)
    return df

def merge_pipelines(p1: pd.DataFrame, p2: pd.DataFrame) -> pd.DataFrame:
    merged = p1[["row_id","Query"]+METRICS].merge(
        p2[["row_id"]+METRICS],
        on="row_id",
        suffixes=("_p1","_p2")
    )
    return merged

def compute_summary(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for m in METRICS:
        m1, m2 = m+"_p1", m+"_p2"
        if m1 not in df or m2 not in df:
            continue
        s1, s2 = df[m1], df[m2]
        rows.append({
            "metric": m,
            "p1_avg": s1.mean(),
            "p2_avg": s2.mean(),
            "p1_wins": int((s1 > s2).sum()),
            "p2_wins": int((s2 > s1).sum()),
            "ties":     int((s1 == s2).sum()),
        })
    return pd.DataFrame(rows)

# explicit green/yellow/red counts per metric for each pipeline column
def compute_color_counts(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for m in METRICS:
        m1, m2 = m+"_p1", m+"_p2"
        if m1 not in df or m2 not in df:
            continue
        s1, s2 = df[m1], df[m2]

        p1_green = int((s1 > s2).sum())
        p1_yellow = int((s1 == s2).sum())
        p1_red = int((s1 < s2).sum())

        p2_green = int((s2 > s1).sum())
        p2_yellow = p1_yellow
        p2_red = p1_green  # when p1 is green, p2 is red, and vice versa

        rows.append({
            "metric": m,
            "p1_green": p1_green,
            "p1_yellow": p1_yellow,
            "p1_red": p1_red,
            "p2_green": p2_green,
            "p2_yellow": p2_yellow,
            "p2_red": p2_red,
        })
    return pd.DataFrame(rows)

def write_with_formatting(df: pd.DataFrame, summary: pd.DataFrame, out_path: str):
    color_counts = compute_color_counts(df)

    with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
        df.to_excel(writer, sheet_name="query_by_query", index=False)
        summary.to_excel(writer, sheet_name="summary", index=False)
        color_counts.to_excel(writer, sheet_name="color_counts", index=False)

        wb = writer.book
        ws = writer.sheets["query_by_query"]

        fmt_p1 = wb.add_format({"bg_color":"#C6EFCE"})  # green
        fmt_p2 = wb.add_format({"bg_color":"#FFC7CE"})  # red
        fmt_tie = wb.add_format({"bg_color":"#FFEB9C"}) # yellow

        ws.freeze_panes(1, 0)
        n_rows = len(df) + 1  # +1 for header

        for col_i, col in enumerate(df.columns):
            if not col.endswith("_p1"):
                continue
            base = col[:-3]
            col_p2 = base + "_p2"
            if col_p2 not in df:
                continue

            c1 = xl_col_to_name(col_i)
            c2 = xl_col_to_name(df.columns.get_loc(col_p2))

            rng1 = f"{c1}2:{c1}{n_rows}"
            rng2 = f"{c2}2:{c2}{n_rows}"

            # P1 column rules
            ws.conditional_format(rng1, {"type":"formula", "criteria":f"={c1}2>{c2}2", "format":fmt_p1})
            ws.conditional_format(rng1, {"type":"formula", "criteria":f"={c1}2<{c2}2", "format":fmt_p2})
            ws.conditional_format(rng1, {"type":"formula", "criteria":f"={c1}2={c2}2", "format":fmt_tie})
            # P2 column rules
            ws.conditional_format(rng2, {"type":"formula", "criteria":f"={c2}2>{c1}2", "format":fmt_p1})
            ws.conditional_format(rng2, {"type":"formula", "criteria":f"={c2}2<{c1}2", "format":fmt_p2})
            ws.conditional_format(rng2, {"type":"formula", "criteria":f"={c2}2={c1}2", "format":fmt_tie})

# === Auto-discover and compare ALL pipeline-result pairs ===
# Looks for: results/gt-vs-runs-with-filter-column-averages_pipelineN.xlsx
files = sorted(
    glob.glob("results/gt-vs-runs-with-filter-column-averages_pipeline*.xlsx"),
    key=os.path.getmtime
)
if len(files) < 2:
    raise FileNotFoundError("Need at least two result files in 'results/' to compare.")

# Map {pipeline_num:int -> path}
pipelines = {}
for p in files:
    m = re.search(r'pipeline(\d+)', p, re.IGNORECASE)
    if not m:
        continue
    pipelines[int(m.group(1))] = p

if len(pipelines) < 2:
    raise RuntimeError("Could not parse at least two pipeline numbers from result files.")

for p1_num, p2_num in combinations(sorted(pipelines.keys()), 2):
    p1_path = pipelines[p1_num]
    p2_path = pipelines[p2_num]
    out_path = f"results/pipeline_comparison_p{p1_num}_vs_p{p2_num}.xlsx"

    df1 = load_queries_with_averages(p1_path)
    df2 = load_queries_with_averages(p2_path)

    merged = merge_pipelines(df1, df2)
    summary = compute_summary(merged)
    write_with_formatting(merged, summary, out_path)

    print(f"Compared pipeline {p1_num} vs {p2_num} -> {out_path}")


In [ ]:
# import pandas as pd
# from pathlib import Path
# from xlsxwriter.utility import xl_col_to_name  # safer than chr(65+col)

# METRICS = [
#     "avg_precision_filters",
#     "avg_recall_filters",
#     "avg_precision_columns",
#     "avg_recall_columns",
# ]

# def load_queries_with_averages(path: str) -> pd.DataFrame:
#     book = pd.read_excel(path, sheet_name=None)
#     df = book["queries_with_averages"].copy()
#     if "row_id" not in df.columns:
#         df = df.reset_index().rename(columns={"index": "row_id"})
#     if "Query" not in df.columns:
#         df["Query"] = df["row_id"].astype(str)
#     return df

# def merge_pipelines(p1: pd.DataFrame, p2: pd.DataFrame) -> pd.DataFrame:
#     merged = p1[["row_id","Query"]+METRICS].merge(
#         p2[["row_id"]+METRICS],
#         on="row_id",
#         suffixes=("_p1","_p2")
#     )
#     return merged

# def compute_summary(df: pd.DataFrame) -> pd.DataFrame:
#     rows = []
#     for m in METRICS:
#         m1, m2 = m+"_p1", m+"_p2"
#         if m1 not in df or m2 not in df:
#             continue
#         s1, s2 = df[m1], df[m2]
#         rows.append({
#             "metric": m,
#             "p1_avg": s1.mean(),
#             "p2_avg": s2.mean(),
#             "p1_wins": int((s1 > s2).sum()),
#             "p2_wins": int((s2 > s1).sum()),
#             "ties":     int((s1 == s2).sum()),
#         })
#     return pd.DataFrame(rows)

# # NEW: explicit green/yellow/red counts per column (mirrors your CF rules)
# def compute_color_counts(df: pd.DataFrame) -> pd.DataFrame:
#     """
#     For each metric, returns counts of green/yellow/red for p1 column and p2 column.
#     Green in a column == that pipeline's value > the other pipeline's value.
#     Yellow == tie. Red == that pipeline's value < the other's.
#     """
#     rows = []
#     for m in METRICS:
#         m1, m2 = m+"_p1", m+"_p2"
#         if m1 not in df or m2 not in df:
#             continue
#         s1, s2 = df[m1], df[m2]

#         p1_green = int((s1 > s2).sum())
#         p1_yellow = int((s1 == s2).sum())
#         p1_red = int((s1 < s2).sum())

#         p2_green = int((s2 > s1).sum())
#         p2_yellow = p1_yellow
#         p2_red = p1_green  # when p1 is green, p2 is red, and vice versa

#         rows.append({
#             "metric": m,
#             # counts for the p1 column colors
#             "p1_green": p1_green,
#             "p1_yellow": p1_yellow,
#             "p1_red": p1_red,
#             # counts for the p2 column colors
#             "p2_green": p2_green,
#             "p2_yellow": p2_yellow,
#             "p2_red": p2_red,
#         })
#     return pd.DataFrame(rows)

# def write_with_formatting(df: pd.DataFrame, summary: pd.DataFrame, out_path: str):
#     color_counts = compute_color_counts(df)

#     with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
#         df.to_excel(writer, sheet_name="query_by_query", index=False)
#         summary.to_excel(writer, sheet_name="summary", index=False)
#         color_counts.to_excel(writer, sheet_name="color_counts", index=False)

#         wb = writer.book
#         ws = writer.sheets["query_by_query"]

#         fmt_p1 = wb.add_format({"bg_color":"#C6EFCE"})  # green
#         fmt_p2 = wb.add_format({"bg_color":"#FFC7CE"})  # red
#         fmt_tie = wb.add_format({"bg_color":"#FFEB9C"}) # yellow

#         ws.freeze_panes(1, 0)
#         n_rows = len(df) + 1  # +1 for header

#         for col_i, col in enumerate(df.columns):
#             if not col.endswith("_p1"):
#                 continue
#             base = col[:-3]
#             col_p2 = base + "_p2"
#             if col_p2 not in df:
#                 continue

#             # safe column letters (A, B, ..., AA, AB, ...)
#             c1 = xl_col_to_name(col_i)        # p1 column letter(s)
#             c2 = xl_col_to_name(df.columns.get_loc(col_p2))

#             rng1 = f"{c1}2:{c1}{n_rows}"  # data rows only
#             rng2 = f"{c2}2:{c2}{n_rows}"

#             # format rules for P1 col (green if p1>p2, red if p1<p2, yellow if tie)
#             ws.conditional_format(rng1, {"type":"formula", "criteria":f"={c1}2>{c2}2", "format":fmt_p1})
#             ws.conditional_format(rng1, {"type":"formula", "criteria":f"={c1}2<{c2}2", "format":fmt_p2})
#             ws.conditional_format(rng1, {"type":"formula", "criteria":f"={c1}2={c2}2", "format":fmt_tie})

#             # format rules for P2 col (green if p2>p1, red if p2<p1, yellow if tie)
#             ws.conditional_format(rng2, {"type":"formula", "criteria":f"={c2}2>{c1}2", "format":fmt_p1})
#             ws.conditional_format(rng2, {"type":"formula", "criteria":f"={c2}2<{c1}2", "format":fmt_p2})
#             ws.conditional_format(rng2, {"type":"formula", "criteria":f"={c2}2={c1}2", "format":fmt_tie})

# # === Example usage inside notebook ===
# p1_path = "results/gt-vs-runs-with-filter-column-averages_pipeline1.xlsx"
# p2_path = "results/gt-vs-runs-with-filter-column-averages_pipeline2.xlsx"
# out_path = "results/pipeline_comparison.xlsx"

# df1 = load_queries_with_averages(p1_path)
# df2 = load_queries_with_averages(p2_path)

# merged = merge_pipelines(df1, df2)
# summary = compute_summary(merged)

# write_with_formatting(merged, summary, out_path)
# print("Comparison written to:", out_path)
